# NB16 · TREAD-TWO / GEO preparation and contract gate

**Run this notebook first. This is engineering preparation, not the 60-epoch fit.**
It tests the real geometry/texture/error model using existing training images,
then publishes direct CSV/native evidence and actually restores a published
checkpoint in a fresh worker. It never edits either workstation or earlier runs.

| Kaggle setting | Required value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **ON** |
| Input | Add the existing **shanmuk4622/tire-dataset-prepared-phase2** expanded dataset |
| Secret | Enable **HF_TOKEN** with write access to **Shanmuk4622/tyre-wear-study** |
| Space | Start with at least **4 GB free** in working output; retain stopped-session files |
| Action | **Run All**, in order; no helper pasting or old session variables |

GPU1 runs the immutable selected TREAD-ONE anchor one image at a time. GPU0 runs
the small new modules at batch8. All tests use training-role images only; no
measured fit or held-out checkpoint selection takes place. The finite checks
compare four uninterrupted updates, a local restored path and a true HF-restored
continuation. Disposable checks are never eligible model checkpoints.

The approved next notebooks are NB17 (one resumable fit) and NB18 (review).
They follow actual NB16 evidence and completion of the measured-run CSV/shard
integration; this preparation notebook does not claim those remaining gates pass.


In [1]:
# 1. Install only the tested private HF transport; preserve Kaggle Torch/CUDA.
import base64, hashlib, json, os, subprocess, sys
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
private = Path('/kaggle/working/geo_private_hf_0362')
loaded = sys.modules.get('huggingface_hub')
if loaded is not None and loaded.__version__ != '0.36.2':
    raise RuntimeError('Restart the Kaggle kernel and run NB16 first; a different HF SDK is already loaded.')
if not (private/'huggingface_hub-0.36.2.dist-info').is_dir():
    subprocess.check_call([sys.executable,'-m','pip','install','--disable-pip-version-check',
                           '--no-deps','--target',str(private),'huggingface_hub==0.36.2'])
sys.path.insert(0,str(private))
import huggingface_hub
assert huggingface_hub.__version__=='0.36.2'
PAYLOAD = {'anchor_geo.py': {'base64': 'IiIiSG9va3MgZXhwb3NlIGZyb3plbiBvcmlnaW5hbCB0ZW5zb3JzIHdpdGhvdXQgcmV3cml0aW5nIGl0cyBmb3J3YXJkIG9yIGNoZWNrcG9pbnQga2V5cy4iIiIKaW1wb3J0IHRvcmNoCmZyb20gY29tbW9uIGltcG9ydCByZWFkLCBIRVJFLCBzaGEKZnJvbSBkYXRhX2dlbyBpbXBvcnQgbm9ybWFsaXplCmZyb20gZnJvemVuX2FuY2hvciBpbXBvcnQgVHJlYWRPbmUKCgpjbGFzcyBBbmNob3I6CiAgICBkZWYgX19pbml0X18oc2VsZiwgd2VpZ2h0cywgZGV2aWNlKToKICAgICAgICBhc3NlcnQgc2hhKHdlaWdodHMpID09IHJlYWQoSEVSRSAvICdQTEFOLmpzb24nKVsnYW5jaG9yJ11bJ3dlaWdodHNfc2hhMjU2J10sICdXcm9uZyBhbmNob3Igd2VpZ2h0cycKICAgICAgICBzZWxmLmRldmljZSA9IHRvcmNoLmRldmljZShkZXZpY2UpCiAgICAgICAgc2VsZi5tb2RlbCA9IFRyZWFkT25lKCkudG8oZGV2aWNlKS5ldmFsKCkKICAgICAgICBzdGF0ZSA9IHRvcmNoLmxvYWQod2VpZ2h0cywgbWFwX2xvY2F0aW9uPSdjcHUnLCB3ZWlnaHRzX29ubHk9VHJ1ZSkKICAgICAgICBzZWxmLm1vZGVsLmxvYWRfc3RhdGVfZGljdChzdGF0ZSwgc3RyaWN0PVRydWUpCiAgICAgICAgc2VsZi5tb2RlbC5yZXF1aXJlc19ncmFkXyhGYWxzZSkKICAgICAgICBzZWxmLmNhcHR1cmVkID0ge30KICAgICAgICBzZWxmLm1vZGVsLmQxLnJlZ2lzdGVyX2ZvcndhcmRfaG9vayhsYW1iZGEgbSwgYSwgeTogc2VsZi5jYXB0dXJlZC51cGRhdGUoZmVhdHVyZXM9eSkpCiAgICAgICAgc2VsZi5tb2RlbC5wb2ludHMucmVnaXN0ZXJfZm9yd2FyZF9wcmVfaG9vayhsYW1iZGEgbSwgYTogc2VsZi5jYXB0dXJlZC51cGRhdGUocG9pbnRfZmVhdHVyZXM9YVswXSkpCiAgICAgICAgc2VsZi5tb2RlbC5wb2ludHMucmVnaXN0ZXJfZm9yd2FyZF9ob29rKGxhbWJkYSBtLCBhLCB5OiBzZWxmLmNhcHR1cmVkLnVwZGF0ZShyYXc9eSkpCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIF9fY2FsbF9fKHNlbGYsIHJnYiwgZ3VpZGVzLCBvdXRwdXRfZGV2aWNlKToKICAgICAgICByZXN1bHRzID0gW10KICAgICAgICBmb3IgaW1hZ2UsIGd1aWRlIGluIHppcChyZ2IsIGd1aWRlcyk6CiAgICAgICAgICAgIHNlbGYuY2FwdHVyZWQuY2xlYXIoKQogICAgICAgICAgICBvdXQgPSBzZWxmLm1vZGVsKG5vcm1hbGl6ZShpbWFnZVtOb25lXS50byhzZWxmLmRldmljZSkpLCBndWlkZVtOb25lXS50byhzZWxmLmRldmljZSkpCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHsqKntrOiB2LmRldGFjaCgpLnRvKG91dHB1dF9kZXZpY2UpIGZvciBrLCB2IGluIHNlbGYuY2FwdHVyZWQuaXRlbXMoKX0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAnYW5jaG9yX2xvZ2l0cyc6IG91dFsnc2VnX2xvZ2l0cyddLmRldGFjaCgpLnRvKG91dHB1dF9kZXZpY2UpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgJ2FuY2hvcl9wb2ludHMnOiBvdXRbJ3BvaW50cyddLmRldGFjaCgpLnRvKG91dHB1dF9kZXZpY2UpfSkKICAgICAgICBzZWxmLmNhcHR1cmVkLmNsZWFyKCkKICAgICAgICByZXR1cm4ge25hbWU6IHRvcmNoLmNhdChbcltuYW1lXSBmb3IgciBpbiByZXN1bHRzXSkgZm9yIG5hbWUgaW4gcmVzdWx0c1swXX0K', 'sha256': 'a2dfaa7679b03786def3745b707a97f54039cc28f9ab25189bd3430b827d2d98'}, 'atlas_geo.py': {'base64': 'IiIiRGV0YWNoZWQgcHJlZGljdGVkLXN1cHBvcnQgYXRsYXMgYW5kIHN1ZmZpY2llbnQgcm93IHBhcmFtZXRlcnMgZm9yIGV4YWN0IHJlcGxheS4iIiIKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2gubm4gaW1wb3J0IGZ1bmN0aW9uYWwgYXMgRgpmcm9tIGdlb21ldHJ5IGltcG9ydCBwYXRjaGVzLCBiYW5kX3dlaWdodHMKCgpkZWYgZ3JpZF9mcm9tX3Jvd3MocGFyYW1ldGVycywgdmFsaWQpOgogICAgIyBwYXJhbWV0ZXJzOiBbTiw1MTIsM10gPSBzb3VyY2UteSwgbGVmdCwgcmlnaHQgaW4gaW5wdXQgcGl4ZWxzLgogICAgeXksIGxlZnQsIHJpZ2h0ID0gcGFyYW1ldGVycy51bmJpbmQoLTEpCiAgICB1ID0gdG9yY2gubGluc3BhY2UoMCwgMSwgMjU2LCBkZXZpY2U9cGFyYW1ldGVycy5kZXZpY2UpCiAgICB4eCA9IGxlZnRbLi4uLCBOb25lXSArIHUgKiAocmlnaHQgLSBsZWZ0KVsuLi4sIE5vbmVdCiAgICBncmlkID0gdG9yY2guc3RhY2soKDIgKiB4eCAvIDM4MyAtIDEsICgyICogeXkgLyA1MTEgLSAxKVsuLi4sIE5vbmVdLmV4cGFuZCgtMSwgLTEsIDI1NikpLCAtMSkKICAgIHJldHVybiBncmlkLm1hc2tlZF9maWxsKH52YWxpZFsuLi4sIE5vbmUsIE5vbmVdLCAwKQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGNhbm9uaWNhbGl6ZShyZ2IsIG1hc2tzLCBpZ25vcmU9Tm9uZSk6CiAgICByZ2IsIG1hc2tzID0gcmdiLmRldGFjaCgpLCBtYXNrcy5kZXRhY2goKQogICAgYmluYXJ5ID0gKG1hc2tzID49IC41KS5yZXBlYXRfaW50ZXJsZWF2ZSg0LCAtMikucmVwZWF0X2ludGVybGVhdmUoNCwgLTEpCiAgICBzdXBwb3J0ID0gYmluYXJ5WzosIDBdICYgYmluYXJ5WzosIDFdCiAgICB4cyA9IHRvcmNoLmFyYW5nZSgzODQsIGRldmljZT1yZ2IuZGV2aWNlKVtOb25lLCBOb25lXQogICAgbGVmdCA9IHRvcmNoLndoZXJlKHN1cHBvcnQsIHhzLCAzODQpLmFtaW4oLTEpCiAgICByaWdodCA9IHRvcmNoLndoZXJlKHN1cHBvcnQsIHhzLCAtMSkuYW1heCgtMSkKICAgIGdvb2QgPSByaWdodCAtIGxlZnQgKyAxID49IDE2CiAgICBwYXJhbWV0ZXJzLCBmbGFncyA9IFtdLCBbXQogICAgZm9yIGkgaW4gcmFuZ2UobGVuKHJnYikpOgogICAgICAgIHJvd3MgPSB0b3JjaC53aGVyZShnb29kW2ldKVswXQogICAgICAgIGlmIG5vdCBsZW4ocm93cyk6CiAgICAgICAgICAgIHBhcmFtZXRlcnMuYXBwZW5kKHJnYi5uZXdfemVyb3MoKDUxMiwgMykpKQogICAgICAgICAgICBmbGFncy5hcHBlbmQodG9yY2guemVyb3MoNTEyLCBkZXZpY2U9cmdiLmRldmljZSwgZHR5cGU9dG9yY2guYm9vbCkpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgeSA9IHRvcmNoLmxpbnNwYWNlKGludChyb3dzWzBdKSwgaW50KHJvd3NbLTFdKSwgNTEyLCBkZXZpY2U9cmdiLmRldmljZSkKICAgICAgICBpbmRpY2VzID0geS5yb3VuZCgpLmxvbmcoKQogICAgICAgIHBhcmFtZXRlcnMuYXBwZW5kKHRvcmNoLnN0YWNrKCh5LCBsZWZ0W2ksIGluZGljZXNdLmZsb2F0KCksIHJpZ2h0W2ksIGluZGljZXNdLmZsb2F0KCkpLCAtMSkpCiAgICAgICAgZmxhZ3MuYXBwZW5kKGdvb2RbaSwgaW5kaWNlc10pCiAgICBwYXJhbWV0ZXJzLCBmbGFncyA9IHRvcmNoLnN0YWNrKHBhcmFtZXRlcnMpLCB0b3JjaC5zdGFjayhmbGFncykKICAgIGdyaWQgPSBncmlkX2Zyb21fcm93cyhwYXJhbWV0ZXJzLCBmbGFncykKICAgIHZhbGlkID0gRi5ncmlkX3NhbXBsZShzdXBwb3J0WzosIE5vbmVdLmZsb2F0KCksIGdyaWQsIG1vZGU9J25lYXJlc3QnLCBhbGlnbl9jb3JuZXJzPVRydWUpCiAgICB2YWxpZCAqPSBmbGFnc1s6LCBOb25lLCA6LCBOb25lXQogICAgYXRsYXMgPSBGLmdyaWRfc2FtcGxlKHJnYiwgZ3JpZCwgbW9kZT0nYmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUpICogdmFsaWQKICAgIGlnbm9yZWQgPSB0b3JjaC56ZXJvc19saWtlKHZhbGlkKSBpZiBpZ25vcmUgaXMgTm9uZSBlbHNlIEYuZ3JpZF9zYW1wbGUoaWdub3JlLmRldGFjaCgpLCBncmlkLCBtb2RlPSduZWFyZXN0JywgYWxpZ25fY29ybmVycz1UcnVlKQogICAgbWFzcywgY292ZXJhZ2UgPSBiYW5kX3dlaWdodHModmFsaWQpCiAgICByZXR1cm4gZGljdChyZ2I9YXRsYXMuZGV0YWNoKCksIHZhbGlkPXZhbGlkLmRldGFjaCgpLCBpZ25vcmVkPWlnbm9yZWQuZGV0YWNoKCksCiAgICAgICAgICAgICAgICByb3dfcGFyYW1ldGVycz1wYXJhbWV0ZXJzLCByb3dfdmFsaWQ9ZmxhZ3MsIGdyaWQ9Z3JpZCwKICAgICAgICAgICAgICAgIG1hc3M9bWFzcywgY292ZXJhZ2U9Y292ZXJhZ2UpCg==', 'sha256': 'f5e5e69fd531c1d131f84d7b87b2b9858fc598bb22a6bdcc050885ff93392e31'}, 'BUDGET.json': {'base64': 'ew0KICAiYXNzdW1lZF9vdXRwdXRfYnl0ZXMiOiAyMDAwMDAwMDAwMCwNCiAgImZyZWVfcmVzZXJ2ZV9ieXRlcyI6IDIwMDAwMDAwMDAsDQogICJmdWxsX2NvcHlfc3RhZ2luZ19tYXlfZmFpbCI6IHRydWUsDQogICJsb2dpY2FsX2xpbWl0X2J5dGVzIjogMTUwMDAwMDAwMDAsDQogICJuYXRpdmVfZXhhY3Rfc2hhcGVfYnl0ZXMiOiB7DQogICAgImFsbF9nZW9tZXRyeV9sb2dpdHNfRlAzMiI6IDI5MDE5MzQwODAsDQogICAgImF0bGFzX3Jvd19wYXJhbWV0ZXJzX0ZQMzIiOiAxODEzNzA4ODAsDQogICAgIm1hc2tlZF9sb3NzX3dlaWdodHNfRlAzMiI6IDQ4MzY1NTY4MCwNCiAgICAibWFza2VkX3ByZWRpY3Rpb25zX2FuZF90YXJnZXRzX0ZQMzIiOiAzODY5MjQ1NDQwLA0KICAgICJwcmVfYW5kX3Bvc3RfY29udGV4dF90b2tlbnNfRlAzMiI6IDE0NTA5NjcwNDANCiAgfSwNCiAgInBlYWtfbGltaXRfYnl0ZXMiOiAxODAwMDAwMDAwMCwNCiAgInByb2plY3RlZF9sb2dpY2FsX2J5dGVzIjogMTQ2MzAwNzM2MDAuMCwNCiAgInByb2plY3RlZF9zYW1lX2ZpbGVzeXN0ZW1fcGVha19ieXRlcyI6IDE1MTY2OTQ0NTEyLjAsDQogICJwcm92aXNpb25hbF9hbGxvd2FuY2VfYnl0ZXMiOiB7DQogICAgImVuZHBvaW50X2RlbnNlX0NTViI6IDExNzY2OTg4ODAsDQogICAgImVwb2NoX2NoZWNrcG9pbnRzX2FuZF9yb2xsaW5nX3N0YXRlcyI6IDYwMDAwMDAwMCwNCiAgICAiZ2FsbGVyaWVzIjogMTUwMDAwMDAwLA0KICAgICJnbG9iYWxfYW5kX3Byb2plY3RlZF9DU1YiOiAzNzc4NTYwMDAsDQogICAgImludGVydmVudGlvbl9vdXRwdXRzIjogNDAwMDAwMDAwLA0KICAgICJvdGhlcl9DU1ZfYW5kX25hdGl2ZV9mbGFncyI6IDUwMDAwMDAwMCwNCiAgICAic291cmNlX2FuZF9TREsiOiAxMDAwMDAwMDANCiAgfSwNCiAgInByb3Zpc2lvbmFsX2RlbnNlX3Jvd19ieXRlcyI6IDQyLA0KICAicHJvdmlzaW9uYWxfdmVjdG9yX3Jvd19ieXRlcyI6IDQwLA0KICAicmVxdWlyZXNfbGl2ZV9maXh0dXJlIjogdHJ1ZSwNCiAgInNhbXBsZV91cHBlcl9ib3VuZCI6IDI5NTIwLA0KICAic3RhdHVzIjogImFyaXRobWV0aWNfZGVzaWduX3Byb2plY3Rpb25fbm90X2xpdmVfbWVhc3VyZW1lbnQiLA0KICAidGV4dHVyZV92aWV3X3VwcGVyX2JvdW5kIjogNTkwNDAsDQogICJ1bmFkanVzdGVkX2xvZ2ljYWxfYnl0ZXMiOiAxMjE5MTcyODAwMCwNCiAgInVuY2VydGFpbnR5X2ZhY3RvciI6IDEuMg0KfQ0K', 'sha256': '47fe81dc8e9cf56613e6f9bf3201821b2c412e7fb3c07ab951534a7912f53f1b'}, 'checks_geo.py': {'base64': 'IiIiSW5kZXBlbmRlbnQgbnVtZXJpY2FsIGFuZCB0cmFuc2FjdGlvbiBjaGVja3MgdXNlZCBieSBOQjE2IGFuZCBvZmZsaW5lIHZhbGlkYXRpb24uIiIiCmltcG9ydCBjb3B5CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2gubm4gaW1wb3J0IGZ1bmN0aW9uYWwgYXMgRgpmcm9tIGNvbW1vbiBpbXBvcnQgY3B1LCByZWFkLCBzZWVkX2FsbApmcm9tIG1vZGVsX2dlbyBpbXBvcnQgR2VvLCBvcmRlcmVkX3BvaW50cwpmcm9tIG9iamVjdGl2ZV9nZW8gaW1wb3J0IGNvYXJzZV90cnV0aCwgZnVsbF9nZW9tZXRyeSwgZ2VvbWV0cnlfbG9zcywgcXVhbGl0eV9sb3NzLCB0ZXh0dXJlX2xvc3MsIHZhcmlhbmNlX2NvdmFyaWFuY2UKZnJvbSBzdG9yZV9nZW8gaW1wb3J0IFN0b3JlCgoKZGVmIG51bWVyaWNfY2hlY2tzKG1vZGVsLCBiYXRjaCwgYmFzZSwgc2VlZHMpOgogICAgcmVwb3J0ID0ge30KICAgIG91dCA9IG1vZGVsKGJhc2UsIGJhdGNoLCBzZWVkcykKICAgIGEsIGIgPSBvdXRbJ3ZpZXdzJ10KICAgIF8sIHRlcm1zID0gdGV4dHVyZV9sb3NzKG91dFsndmlld3MnXSwgW3JbJ2NvbXBvbmVudCddIGZvciByIGluIGJhdGNoWydyb3dzJ11dLCAxKQogICAgYXNzZXJ0IHRlcm1zWyd0ZXh0dXJlX3N0ZXAnXQogICAgZWxpZ2libGUgPSBhWydlbGlnaWJsZSddICYgYlsnZWxpZ2libGUnXQogICAgeiA9IGFbJ3Byb2plY3RlZCddW2VsaWdpYmxlXQogICAgY2VudGVyZWQgPSB6LXoubWVhbigwKQogICAgIyBBbiBpbmRlcGVuZGVudCBsb29wIGNvdmFyaWFuY2UgYWdyZWVzIHdpdGggdGhlIG1hdHJpeCBpbXBsZW1lbnRhdGlvbi4KICAgIG1hbnVhbCA9IHN1bSh0b3JjaC5vdXRlcih2LCB2KSBmb3IgdiBpbiBjZW50ZXJlZCkvKGxlbih6KS0xKQogICAgdmFyaWFuY2UsIGNvdmFyaWFuY2UgPSB2YXJpYW5jZV9jb3ZhcmlhbmNlKHopCiAgICBhc3NlcnQgdG9yY2guYWxsY2xvc2UodmFyaWFuY2UsICgxLShtYW51YWwuZGlhZygpKzFlLTQpLnNxcnQoKSkuY2xhbXBfbWluKDApLm1lYW4oKSwgYXRvbD0xZS02KQogICAgb2ZmZGlhZ29uYWwgPSBzdW0obWFudWFsW2ksal0uc3F1YXJlKCkgZm9yIGkgaW4gcmFuZ2UoNjQpIGZvciBqIGluIHJhbmdlKDY0KSBpZiBpIT1qKS82NAogICAgYXNzZXJ0IHRvcmNoLmFsbGNsb3NlKGNvdmFyaWFuY2UsIG9mZmRpYWdvbmFsLCBhdG9sPTFlLTYsIHJ0b2w9MWUtNSkKICAgIG51bWVyYXRvciwgZGVub21pbmF0b3IgPSAwLiwgMC4KICAgIGZvciB2aWV3IGluIChhLCBiKToKICAgICAgICBwcmVkLCB0YXJnZXQsIHdlaWdodCA9IFtjcHUodmlld1trXSkubnVtcHkoKS5hc3R5cGUobnAuZmxvYXQ2NCkgZm9yIGsgaW4gKCdwcmVkaWN0aW9uJywgJ3RhcmdldCcsICd3ZWlnaHQnKV0KICAgICAgICBtYXNrID0gY3B1KHZpZXdbJ3NlbGVjdGVkJ10pLm51bXB5KClbLi4uLE5vbmUsTm9uZSxOb25lXSpjcHUoZWxpZ2libGUpLm51bXB5KClbOixOb25lLE5vbmUsTm9uZSxOb25lXQogICAgICAgIGRlbHRhID0gbnAuYWJzKHByZWQtdGFyZ2V0KQogICAgICAgIGVsZW1lbnQgPSBucC53aGVyZShkZWx0YTwuMSwgZGVsdGEqKjIvLjIsIGRlbHRhLS4wNSkKICAgICAgICBudW1lcmF0b3IgKz0gZmxvYXQoKGVsZW1lbnQqd2VpZ2h0Km1hc2spLnN1bSgpKQogICAgICAgIGRlbm9taW5hdG9yICs9IGZsb2F0KCh3ZWlnaHQqbWFzaykuc3VtKCkqNCkKICAgIGFzc2VydCBucC5pc2Nsb3NlKGZsb2F0KHRlcm1zWydtYXNrZWRfbnVtZXJhdG9yJ10pLCBudW1lcmF0b3IsIHJ0b2w9MWUtNSkKICAgIGFzc2VydCBkZW5vbWluYXRvciA9PSBmbG9hdCh0ZXJtc1snbWFza2VkX2Rlbm9taW5hdG9yJ10pCiAgICByZWNvbnN0cnVjdGVkID0gMjUqZmxvYXQodGVybXNbJ2ludmFyaWFuY2UnXSkrMjUqKGZsb2F0KHRlcm1zWyd2YXJpYW5jZV9hJ10pK2Zsb2F0KHRlcm1zWyd2YXJpYW5jZV9iJ10pKStmbG9hdCh0ZXJtc1snY292YXJpYW5jZV9hJ10pK2Zsb2F0KHRlcm1zWydjb3ZhcmlhbmNlX2InXSkrLjUqbnVtZXJhdG9yL2Rlbm9taW5hdG9yCiAgICBhc3NlcnQgbnAuaXNjbG9zZShmbG9hdCh0ZXJtc1sndGV4dHVyZSddKSwgcmVjb25zdHJ1Y3RlZCwgcnRvbD0xZS01KQogICAgcmVwb3J0WydWSUNfY292YXJpYW5jZV9hbmRfbWFza2VkX0h1YmVyX2luZGVwZW5kZW50X3JlZHVjdGlvbnMnXSA9IFRydWUKICAgICMgUmVhbCBoaWRkZW4gUkdCIGNoYW5nZXMgbGVhdmUgZXZlcnkgdmlzaWJsZSBMQyB0aWxlIGFuZCBtYXNrZWQtY29udGV4dAogICAgIyBvdXRwdXQgdW5jaGFuZ2VkLiBNZXRhZGF0YS9zdXBwb3J0IGFyZSBoZWxkIGZpeGVkIHRvIHRlc3QgdGhlIHRleHR1cmUgcGF0aC4KICAgIGFsdGVyZWRfYXRsYXMgPSBkaWN0KG91dFsnYXRsYXMnXSkKICAgIGFsdGVyZWRfYXRsYXNbJ3JnYiddID0gb3V0WydhdGxhcyddWydyZ2InXS5jbG9uZSgpCiAgICBmb3IgaSBpbiByYW5nZShsZW4oYmF0Y2hbJ3Jvd3MnXSkpOgogICAgICAgIGZvciBwYXRjaCBpbiB0b3JjaC53aGVyZShhWydzZWxlY3RlZCddW2ldKVswXS5jcHUoKS50b2xpc3QoKToKICAgICAgICAgICAgcm93LCBjb2wgPSBkaXZtb2QocGF0Y2gsIDQpCiAgICAgICAgICAgIGFsdGVyZWRfYXRsYXNbJ3JnYiddW2ksOixyb3cqNjQ6KHJvdysxKSo2NCxjb2wqNjQ6KGNvbCsxKSo2NF0gPSAuMzcKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGFsdGVyZWQgPSBtb2RlbC50ZXh0dXJlX3ZpZXcoYWx0ZXJlZF9hdGxhcywgW2ludChzKSoyIGZvciBzIGluIHNlZWRzXSkKICAgICAgICBhc3NlcnQgdG9yY2guZXF1YWwoYVsnbGMnXS5kZXRhY2goKVt+YVsnc2VsZWN0ZWQnXV0sIGFsdGVyZWRbJ2xjJ11bfmFbJ3NlbGVjdGVkJ11dKQogICAgICAgIGFzc2VydCB0b3JjaC5lcXVhbChhWydwcmVkaWN0aW9uJ10uZGV0YWNoKCksIGFsdGVyZWRbJ3ByZWRpY3Rpb24nXSkKICAgIHJlcG9ydFsnaGlkZGVuX1JHQl9kb2VzX25vdF9sZWFrX3RvX3Zpc2libGVfTENfb3JfbWFza2VkX3ByZWRpY3Rpb24nXSA9IFRydWUKICAgIF8sIHF1YWxpdHkgPSBxdWFsaXR5X2xvc3Mob3V0WydxdWFsaXR5J10sIG91dFsncmVmaW5lZCddLCBiYXRjaCkKICAgIGFzc2VydCB0b3JjaC5lcXVhbChxdWFsaXR5WydhdmFpbGFibGUnXVs6LDJdLCBiYXRjaFsnaHVtYW4nXSkKICAgIGFzc2VydCBub3QgcXVhbGl0eVsndGFyZ2V0cyddLnJlcXVpcmVzX2dyYWQKICAgIGVtcHR5ID0gZGljdChiYXRjaCkKICAgIGVtcHR5Wydwb2ludF93ZWlnaHQnXSA9IHRvcmNoLnplcm9zX2xpa2UoYmF0Y2hbJ3BvaW50X3dlaWdodCddKQogICAgZW1wdHlbJ21hc2tzJ10gPSBiYXRjaFsnbWFza3MnXS5jbG9uZSgpCiAgICBlbXB0eVsnbWFza3MnXVs6LDJdID0gMQogICAgbG9zcywgZyA9IGdlb21ldHJ5X2xvc3Mob3V0WydyZWZpbmVkJ10sIGJhc2UsIGVtcHR5KQogICAgYXNzZXJ0IGZsb2F0KGdbJ2JjZV9udW1lcmF0b3InXSk9PWZsb2F0KGdbJ2JjZV9kZW5vbWluYXRvciddKT09ZmxvYXQoZ1sncG9pbnRfbnVtZXJhdG9yJ10pPT1mbG9hdChnWydwb2ludF9kZW5vbWluYXRvciddKT09MAogICAgYXNzZXJ0IHRvcmNoLmlzZmluaXRlKGxvc3MpCiAgICBhc3NlcnQgKG91dFsncmVmaW5lZCddWydwb2ludHMnXS5yZXNoYXBlKC0xLDMsMilbLi4uLDFdID4gb3V0WydyZWZpbmVkJ11bJ3BvaW50cyddLnJlc2hhcGUoLTEsMywyKVsuLi4sMF0pLmFsbCgpCiAgICByZXBvcnRbJ2lnbm9yZV9taXNzaW5nX3RhcmdldHNfcG9pbnRfb3JkZXJfaHVtYW5fb25seV9xdWFsaXR5J10gPSBUcnVlCiAgICAjIFZlcmlmeSB0aGUgcHJvcG9zZWQgNHg0IHN1ZmZpY2llbnQgY291bnRzIHJlcHJvZHVjZSBvcmlnaW5hbC1waXhlbCBUUC9GUC9GTi4KICAgIHByZWQgPSBvdXRbJ3JlZmluZWQnXVsnbWFza3MnXS5kZXRhY2goKT49LjUKICAgIHRhcmdldCA9IGJhdGNoWydtYXNrcyddWzosOjJdCiAgICB2YWxpZCA9IDEtYmF0Y2hbJ21hc2tzJ11bOiwyOjNdCiAgICBuID0gbGVuKHByZWQpCiAgICBmb3JlZ3JvdW5kID0gKHRhcmdldCp2YWxpZCkucmVzaGFwZShuLDIsMTI4LDQsOTYsNCkuc3VtKCgzLDUpKQogICAgYXZhaWxhYmxlID0gdmFsaWQucmVzaGFwZShuLDEsMTI4LDQsOTYsNCkuc3VtKCgzLDUpKQogICAgbWV0cmljcyA9IGZ1bGxfZ2VvbWV0cnkob3V0WydyZWZpbmVkJ10sIGJhdGNoKQogICAgYXNzZXJ0IHRvcmNoLmVxdWFsKChwcmVkKmZvcmVncm91bmQpLnN1bSgoLTIsLTEpKS5sb25nKCksIG1ldHJpY3NbJ3RwJ10pCiAgICBhc3NlcnQgdG9yY2guZXF1YWwoKHByZWQqKGF2YWlsYWJsZS1mb3JlZ3JvdW5kKSkuc3VtKCgtMiwtMSkpLmxvbmcoKSwgbWV0cmljc1snZnAnXSkKICAgIGFzc2VydCB0b3JjaC5lcXVhbCgoKH5wcmVkKSpmb3JlZ3JvdW5kKS5zdW0oKC0yLC0xKSkubG9uZygpLCBtZXRyaWNzWydmbiddKQogICAgcmVwb3J0WydkZW5zZV9ibG9ja19jb3VudHNfcmVjb25zdHJ1Y3RfZnVsbF9ncmlkX2NvbmZ1c2lvbiddID0gVHJ1ZQogICAgIyBBIHNraXBwZWQgdGV4dHVyZSBncm91cCBtdXN0IHByZXNlcnZlIHBhcmFtZXRlcnMgQU5EIGV4aXN0aW5nIEFkYW0gbW9tZW50cy4KICAgIGZyb20gZW5naW5lZXJpbmdfZ2VvIGltcG9ydCBvcHRpbWl6ZXJfZm9yLCBlcXVhbAogICAgY2xvbmUgPSBHZW8oKS50byhiYXRjaFsncmdiJ10uZGV2aWNlKQogICAgY2xvbmUubG9hZF9zdGF0ZV9kaWN0KG1vZGVsLnN0YXRlX2RpY3QoKSkKICAgIG9wdCA9IG9wdGltaXplcl9mb3IoY2xvbmUpCiAgICBmb3IgcCBpbiBjbG9uZS5wYXJhbWV0ZXJzKCk6CiAgICAgICAgcC5ncmFkID0gdG9yY2gub25lc19saWtlKHApKi4wMDEKICAgIG9wdC5zdGVwKCkKICAgIG9sZF9wYXJhbXMgPSBbcC5kZXRhY2goKS5jbG9uZSgpIGZvciBwIGluIGNsb25lLmdyb3VwcygpWyd0ZXh0dXJlJ11dCiAgICBvbGRfbW9tZW50cyA9IFtjcHUob3B0LnN0YXRlW3BdKSBmb3IgcCBpbiBjbG9uZS5ncm91cHMoKVsndGV4dHVyZSddXQogICAgb3B0Lnplcm9fZ3JhZChzZXRfdG9fbm9uZT1UcnVlKQogICAgZm9yIG5hbWUgaW4gKCdnZW9tZXRyeScsJ3F1YWxpdHknKToKICAgICAgICBmb3IgcCBpbiBjbG9uZS5ncm91cHMoKVtuYW1lXToKICAgICAgICAgICAgcC5ncmFkID0gdG9yY2gub25lc19saWtlKHApKi4wMDEKICAgIG9wdC5zdGVwKCkKICAgIGFzc2VydCBhbGwodG9yY2guZXF1YWwoYSxwKSBmb3IgYSxwIGluIHppcChvbGRfcGFyYW1zLGNsb25lLmdyb3VwcygpWyd0ZXh0dXJlJ10pKQogICAgYXNzZXJ0IGFsbChlcXVhbChhLGNwdShvcHQuc3RhdGVbcF0pKSBmb3IgYSxwIGluIHppcChvbGRfbW9tZW50cyxjbG9uZS5ncm91cHMoKVsndGV4dHVyZSddKSkKICAgIGZha2Vfdmlld3MgPSBbeyoqdiwnZWxpZ2libGUnOnRvcmNoLnplcm9zX2xpa2UodlsnZWxpZ2libGUnXSl9IGZvciB2IGluIG91dFsndmlld3MnXV0KICAgIF8sIHNraXBwZWQgPSB0ZXh0dXJlX2xvc3MoZmFrZV92aWV3cywgW3JbJ2NvbXBvbmVudCddIGZvciByIGluIGJhdGNoWydyb3dzJ11dLCAxKQogICAgYXNzZXJ0IG5vdCBza2lwcGVkWyd0ZXh0dXJlX3N0ZXAnXQogICAgcmVwb3J0Wydza2lwcGVkX3RleHR1cmVfaGFzX25vX2RlY2F5X29yX21vbWVudF91cGRhdGUnXSA9IFRydWUKICAgIHJldHVybiByZXBvcnQKCgpkZWYgdHJhbnNhY3Rpb25fY2hlY2tzKGZvbGRlcik6CiAgICBmcm9tIGVuZ2luZWVyaW5nX2dlbyBpbXBvcnQgZXF1YWwKICAgIGZvbGRlciA9IFBhdGgoZm9sZGVyKQogICAgcmVzdWx0ID0ge30KICAgIGZvciBjcmFzaCBpbiAoJ2FmdGVyX2V2aWRlbmNlJywnYWZ0ZXJfc3RhdGUnLCdhZnRlcl9wb2ludGVyJyk6CiAgICAgICAgc2VlZF9hbGwoMTIzKQogICAgICAgIG1vZGVsID0gdG9yY2gubm4uTGluZWFyKDIsMikKICAgICAgICBvcHQgPSB0b3JjaC5vcHRpbS5BZGFtVyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPS4wMSkKICAgICAgICBzdG9yZSA9IFN0b3JlKGZvbGRlci9jcmFzaCwgJ2VuZ2luZWVyaW5nX2NyYXNoX2ZpeHR1cmUnKQogICAgICAgIHggPSB0b3JjaC50ZW5zb3IoW1sxLiwyLl1dKQogICAgICAgIG9wdC56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICBtb2RlbCh4KS5zcXVhcmUoKS5zdW0oKS5iYWNrd2FyZCgpCiAgICAgICAgb3B0LnN0ZXAoKQogICAgICAgIHN0b3JlLmNvbW1pdChtb2RlbCxvcHQsMSx7J2dlb21ldHJ5JzoxfSwgeydsb3NzJzpbeyd1cGRhdGUnOjEsJ3ZhbHVlJzpucC5mbG9hdDMyKC4xKX1dfSwgeydyYXcnOm5wLm9uZXMoKDIsMyksZHR5cGU9bnAuZmxvYXQzMil9KQogICAgICAgIG9sZCA9IGNwdShtb2RlbC5zdGF0ZV9kaWN0KCkpCiAgICAgICAgb2xkX29wdCA9IGNwdShvcHQuc3RhdGVfZGljdCgpKQogICAgICAgIG9wdC56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICBtb2RlbCh4KS5zcXVhcmUoKS5zdW0oKS5iYWNrd2FyZCgpCiAgICAgICAgb3B0LnN0ZXAoKQogICAgICAgIG5ldyA9IGNwdShtb2RlbC5zdGF0ZV9kaWN0KCkpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBzdG9yZS5jb21taXQobW9kZWwsb3B0LDIseydnZW9tZXRyeSc6Mn0sIHsnbG9zcyc6W3sndXBkYXRlJzoyLCd2YWx1ZSc6bnAuZmxvYXQzMiguMil9XX0sIHsncmF3JzpucC56ZXJvcygoMiwzKSxkdHlwZT1ucC5mbG9hdDMyKX0sIGNyYXNoPWNyYXNoKQogICAgICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcignQ3Jhc2ggd2FzIG5vdCBpbmplY3RlZCcpCiAgICAgICAgZXhjZXB0IEludGVycnVwdGVkRXJyb3I6CiAgICAgICAgICAgIHBhc3MKICAgICAgICBzdGF0ZSA9IHN0b3JlLnJlc3RvcmUobW9kZWwsb3B0KQogICAgICAgIGFzc2VydCBzdGF0ZVsndXBkYXRlJ109PSgyIGlmIGNyYXNoPT0nYWZ0ZXJfcG9pbnRlcicgZWxzZSAxKQogICAgICAgIGFzc2VydCBlcXVhbChjcHUobW9kZWwuc3RhdGVfZGljdCgpKSxuZXcgaWYgY3Jhc2g9PSdhZnRlcl9wb2ludGVyJyBlbHNlIG9sZCkKICAgICAgICBpZiBjcmFzaCE9J2FmdGVyX3BvaW50ZXInOgogICAgICAgICAgICBhc3NlcnQgZXF1YWwoY3B1KG9wdC5zdGF0ZV9kaWN0KCkpLG9sZF9vcHQpCiAgICAgICAgcmVzdWx0W2NyYXNoXSA9ICdjb21wbGV0ZV9wcmlvcl9vcl9uZXdfdW5pdF9yZXN0b3JlZCcKICAgIHZhbHVlcyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZygxOSkuc3RhbmRhcmRfbm9ybWFsKDEwMDAwKS5hc3R5cGUoJ2Zsb2F0MzInKQogICAgcmVjb3ZlcmVkID0gbnAuYXJyYXkoW1N0b3JlLmZvcm1hdCh4KSBmb3IgeCBpbiB2YWx1ZXNdLCBkdHlwZT0nZmxvYXQzMicpCiAgICBhc3NlcnQgbnAuYXJyYXlfZXF1YWwodmFsdWVzLnZpZXcoJ3VpbnQzMicpLHJlY292ZXJlZC52aWV3KCd1aW50MzInKSkKICAgIHJlc3VsdFsnRlAzMl9DU1Zfcm91bmR0cmlwXzEwMDAwJ10gPSAnZXhhY3RfYml0cycKICAgIHJldHVybiByZXN1bHQK', 'sha256': 'e9f7bb7fc8d2322a8a02cbe49c80f36a3e72bc52b94463d151d2945f1032cb53'}, 'common.py': {'base64': 'IiIiU3RyaWN0IGxvY2FsIHBlcnNpc3RlbmNlLCBkZXRlcm1pbmlzdGljIHN0YXRlIGFuZCBwb3J0YWJsZSBydW4gaWRlbnRpdGllcy4iIiIKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJhbmRvbQppbXBvcnQgc2h1dGlsCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpvcy5lbnZpcm9uLnNldGRlZmF1bHQoJ0NVQkxBU19XT1JLU1BBQ0VfQ09ORklHJywgJzo0MDk2OjgnKQppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpIRVJFID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudAoKCmRlZiBjYW5vbmljYWwodmFsdWUpOgogICAgcmV0dXJuIGpzb24uZHVtcHModmFsdWUsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgnLCcsICc6JyksIGFsbG93X25hbj1GYWxzZSkuZW5jb2RlKCkKCgpkZWYgZGlnZXN0KHZhbHVlKToKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihjYW5vbmljYWwodmFsdWUpKS5oZXhkaWdlc3QoKQoKCmRlZiByZWFkKHBhdGgpOgogICAgcmV0dXJuIGpzb24ubG9hZHMoUGF0aChwYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04JykpCgoKZGVmIHNoYShwYXRoKToKICAgIHJlc3VsdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggb3BlbihwYXRoLCAncmInKSBhcyBzdHJlYW06CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFtYmRhOiBzdHJlYW0ucmVhZCg0ICogMTAyNCoqMiksIGInJyk6CiAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gcmVzdWx0LmhleGRpZ2VzdCgpCgoKZGVmIHN5bmNfZGlyZWN0b3J5KHBhdGgpOgogICAgaWYgb3MubmFtZSAhPSAnbnQnOgogICAgICAgIGZkID0gb3Mub3BlbihQYXRoKHBhdGgpLCBvcy5PX0RJUkVDVE9SWSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIG9zLmZzeW5jKGZkKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIG9zLmNsb3NlKGZkKQoKCmRlZiBhdG9taWMocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHBhdGgud2l0aF9uYW1lKHBhdGgubmFtZSArICcudG1wJykKICAgIHdpdGggdG1wLm9wZW4oJ3diJykgYXMgc3RyZWFtOgogICAgICAgIHN0cmVhbS53cml0ZShjYW5vbmljYWwodmFsdWUpKQogICAgICAgIHN0cmVhbS5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMoc3RyZWFtLmZpbGVubygpKQogICAgb3MucmVwbGFjZSh0bXAsIHBhdGgpCiAgICBzeW5jX2RpcmVjdG9yeShwYXRoLnBhcmVudCkKCgpkZWYgbmF0aXZlKHBhdGgsIHZhbHVlKToKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0bXAgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAnLnRtcCcpCiAgICB3aXRoIHRtcC5vcGVuKCd3YicpIGFzIHN0cmVhbToKICAgICAgICBpZiBwYXRoLnN1ZmZpeCA9PSAnLm5weSc6CiAgICAgICAgICAgIG5wLnNhdmUoc3RyZWFtLCB2YWx1ZSwgYWxsb3dfcGlja2xlPUZhbHNlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGFzc2VydCBwYXRoLnN1ZmZpeCA9PSAnLnB0JwogICAgICAgICAgICB0b3JjaC5zYXZlKHZhbHVlLCBzdHJlYW0pCiAgICAgICAgc3RyZWFtLmZsdXNoKCkKICAgICAgICBvcy5mc3luYyhzdHJlYW0uZmlsZW5vKCkpCiAgICBvcy5yZXBsYWNlKHRtcCwgcGF0aCkKICAgIHN5bmNfZGlyZWN0b3J5KHBhdGgucGFyZW50KQoKCmRlZiBjc3Zfd3JpdGUocGF0aCwgcm93cywgZmllbGRzKToKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0bXAgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAnLnRtcCcpCiAgICB3aXRoIHRtcC5vcGVuKCd3JywgZW5jb2Rpbmc9J3V0Zi04JywgbmV3bGluZT0nJykgYXMgc3RyZWFtOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1maWVsZHMsIGV4dHJhc2FjdGlvbj0ncmFpc2UnLCBsaW5ldGVybWluYXRvcj0nXG4nKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgd3JpdGVyLndyaXRlcm93cyhyb3dzKQogICAgICAgIHN0cmVhbS5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMoc3RyZWFtLmZpbGVubygpKQogICAgb3MucmVwbGFjZSh0bXAsIHBhdGgpCgoKZGVmIGZpbGVfbWV0YShwYXRoKToKICAgIHJldHVybiBkaWN0KHNoYTI1Nj1zaGEocGF0aCksIGJ5dGVzPVBhdGgocGF0aCkuc3RhdCgpLnN0X3NpemUpCgoKZGVmIGluc2lkZShyb290LCByZWxhdGl2ZSk6CiAgICByb290ID0gUGF0aChyb290KS5yZXNvbHZlKCkKICAgIGFzc2VydCBub3QgUGF0aChyZWxhdGl2ZSkuaXNfYWJzb2x1dGUoKSBhbmQgJy4uJyBub3QgaW4gUGF0aChyZWxhdGl2ZSkucGFydHMKICAgIHBhdGggPSAocm9vdCAvIHJlbGF0aXZlKS5yZXNvbHZlKCkKICAgIGFzc2VydCBwYXRoLmlzX3JlbGF0aXZlX3RvKHJvb3QpIGFuZCBwYXRoICE9IHJvb3QKICAgIHJldHVybiBwYXRoCgoKZGVmIGNwdSh2YWx1ZSk6CiAgICBpZiB0b3JjaC5pc190ZW5zb3IodmFsdWUpOgogICAgICAgIHJldHVybiB2YWx1ZS5kZXRhY2goKS5jcHUoKS5jbG9uZSgpCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBkaWN0KToKICAgICAgICByZXR1cm4ge2tleTogY3B1KGl0ZW0pIGZvciBrZXksIGl0ZW0gaW4gdmFsdWUuaXRlbXMoKX0KICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIGxpc3QpOgogICAgICAgIHJldHVybiBbY3B1KGl0ZW0pIGZvciBpdGVtIGluIHZhbHVlXQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgdHVwbGUpOgogICAgICAgIHJldHVybiB0dXBsZShjcHUoaXRlbSkgZm9yIGl0ZW0gaW4gdmFsdWUpCiAgICByZXR1cm4gdmFsdWUKCgpkZWYgc2VlZF9hbGwoc2VlZCk6CiAgICByYW5kb20uc2VlZChzZWVkKQogICAgbnAucmFuZG9tLnNlZWQoc2VlZCkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHRvcmNoLmN1ZGEubWFudWFsX3NlZWRfYWxsKHNlZWQpCiAgICB0b3JjaC51c2VfZGV0ZXJtaW5pc3RpY19hbGdvcml0aG1zKFRydWUpCiAgICB0b3JjaC5iYWNrZW5kcy5jdWRubi5iZW5jaG1hcmsgPSBGYWxzZQogICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uZGV0ZXJtaW5pc3RpYyA9IFRydWUKICAgIHRvcmNoLmJhY2tlbmRzLmN1ZGEubWF0bXVsLmFsbG93X3RmMzIgPSBGYWxzZQogICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYWxsb3dfdGYzMiA9IEZhbHNlCgoKZGVmIHJuZ19zdGF0ZSgpOgogICAgcmV0dXJuIGRpY3QocHl0aG9uPXJhbmRvbS5nZXRzdGF0ZSgpLCBudW1weT1ucC5yYW5kb20uZ2V0X3N0YXRlKCksIGNwdT10b3JjaC5nZXRfcm5nX3N0YXRlKCksCiAgICAgICAgICAgICAgICBjdWRhPXRvcmNoLmN1ZGEuZ2V0X3JuZ19zdGF0ZV9hbGwoKSBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgW10pCgoKZGVmIHJlc3RvcmVfcm5nKHZhbHVlKToKICAgIHJhbmRvbS5zZXRzdGF0ZSh2YWx1ZVsncHl0aG9uJ10pCiAgICBucC5yYW5kb20uc2V0X3N0YXRlKHZhbHVlWydudW1weSddKQogICAgdG9yY2guc2V0X3JuZ19zdGF0ZSh2YWx1ZVsnY3B1J10pCiAgICBpZiB2YWx1ZVsnY3VkYSddOgogICAgICAgIGFzc2VydCBsZW4odmFsdWVbJ2N1ZGEnXSkgPT0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSwgJ0dQVSB0b3BvbG9neSBkaWZmZXJzIGZyb20gc2F2ZWQgc3RhdGUnCiAgICAgICAgdG9yY2guY3VkYS5zZXRfcm5nX3N0YXRlX2FsbCh2YWx1ZVsnY3VkYSddKQoKCmRlZiBlbnZpcm9ubWVudCgpOgogICAgaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YQogICAgcmV0dXJuIGRpY3QocHl0aG9uPXN5cy52ZXJzaW9uLCB0b3JjaD10b3JjaC5fX3ZlcnNpb25fXywgbnVtcHk9bnAuX192ZXJzaW9uX18sCiAgICAgICAgICAgICAgICBwaWxsb3c9bWV0YWRhdGEudmVyc2lvbignUGlsbG93JyksIGN1ZGE9dG9yY2gudmVyc2lvbi5jdWRhLAogICAgICAgICAgICAgICAgY3Vkbm49dG9yY2guYmFja2VuZHMuY3Vkbm4udmVyc2lvbigpLCBkZXZpY2VzPVt0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZShpKSBmb3IgaSBpbiByYW5nZSh0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpKV0pCgoKZGVmIGNhcGFjaXR5KHJvb3QsIHJlc2VydmU9MiAqIDEwMjQqKjMpOgogICAgcm9vdCA9IFBhdGgocm9vdCkKICAgIHNjb3BlID0gUGF0aCgnL2thZ2dsZS93b3JraW5nJykgaWYgUGF0aCgnL2thZ2dsZS93b3JraW5nJykuZXhpc3RzKCkgZWxzZSByb290CiAgICBsb2dpY2FsID0gc3VtKHBhdGguc3RhdCgpLnN0X3NpemUgZm9yIHBhdGggaW4gc2NvcGUucmdsb2IoJyonKSBpZiBwYXRoLmlzX2ZpbGUoKSkKICAgIGZyZWUgPSBzaHV0aWwuZGlza191c2FnZShyb290KS5mcmVlCiAgICByZXN1bHQgPSBkaWN0KHNjb3BlPXN0cihzY29wZSksIGxvZ2ljYWxfYnl0ZXM9bG9naWNhbCwgZnJlZV9ieXRlcz1mcmVlLCBidWRnZXRfYnl0ZXM9MTZfMDAwXzAwMF8wMDApCiAgICBhc3NlcnQgbG9naWNhbCA8IDE2XzAwMF8wMDBfMDAwIGFuZCBmcmVlID4gcmVzZXJ2ZSwgJ1N0b3JhZ2UgZ3VhcmQ6IHByZXNlcnZlIHN0YXRlLCBmbHVzaCBhbmQgaW5zcGVjdCBmcmVlIHNwYWNlJwogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBldmVudChwYXRoLCBraW5kLCAqKnZhbHVlcyk6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBwYXRoLm9wZW4oJ2FiJykgYXMgc3RyZWFtOgogICAgICAgIHN0cmVhbS53cml0ZShjYW5vbmljYWwoZGljdCh0aW1lX3V0Yz10aW1lLnN0cmZ0aW1lKCclWS0lbS0lZFQlSDolTTolU1onLCB0aW1lLmdtdGltZSgpKSwga2luZD1raW5kLCAqKnZhbHVlcykpICsgYidcbicpCiAgICAgICAgc3RyZWFtLmZsdXNoKCkKICAgICAgICBvcy5mc3luYyhzdHJlYW0uZmlsZW5vKCkpCg==', 'sha256': '922c82b82635313ce61ecc2992990f5bc0b7f3cb1d6c8fca00527de72a34a206'}, 'coordinator_geo.py': {'base64': '...', 'sha256': 'ed57590490cc25dc8490f1fc34cdb354d5e215db0841dc0b947adced63ed495f'}, 'data_geo.py': {'base64': 'IiIiRXhpc3RpbmcgZXhwYW5kZWQgS2FnZ2xlIGRhdGE7IGV4YWN0IG9sZCBvdmVybGF5LCBubyBmbGlwcyBvciBuZXcgcGhvdG9zLiIiIgppbXBvcnQgY3N2CmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gY29tbW9uIGltcG9ydCBIRVJFLCBkaWdlc3QsIHJlYWQsIHNoYQpmcm9tIGxlZ2FjeV9kYXRhIGltcG9ydCBwbGFuCmZyb20gcGhhc2UyX3RyYWluaW5nX2RhdGEgaW1wb3J0IHNhbXBsZQoKCmRlZiBsb2NhdGUodmFsdWU9JycpOgogICAgaWYgdmFsdWU6CiAgICAgICAgcm9vdHMgPSBbUGF0aCh2YWx1ZSldCiAgICBlbGlmIFBhdGgoJy9rYWdnbGUvaW5wdXQnKS5pc19kaXIoKToKICAgICAgICByb290cyA9IFtwLnBhcmVudCBmb3IgcCBpbiBQYXRoKCcva2FnZ2xlL2lucHV0Jykucmdsb2IoJ1ZFUlNJT04uanNvbicpCiAgICAgICAgICAgICAgICAgaWYgcmVhZChwKS5nZXQoJ2RhdGFzZXRfdGl0bGUnKSA9PSAnVGlyZSBEYXRhc2V0IFByZXBhcmVkIHBoYXNlMiddCiAgICBlbHNlOgogICAgICAgIHJvb3RzID0gW0hFUkUucGFyZW50c1szXSAvICdwaGFzZTIvZGF0YS9waGFzZTJfZGF0YXNldF92MSddCiAgICBpZiBsZW4ocm9vdHMpICE9IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQXR0YWNoIGV4YWN0bHkgb25lIGV4cGFuZGVkIHNoYW5tdWs0NjIyL3RpcmUtZGF0YXNldC1wcmVwYXJlZC1waGFzZTIgaW5wdXQuJykKICAgIHJvb3QgPSByb290c1swXS5yZXNvbHZlKCkKICAgIGFzc2VydCBzaGEocm9vdCAvICdTSEEyNTZTVU1TLnR4dCcpID09IHJlYWQoSEVSRSAvICdQTEFOLmpzb24nKVsnZGF0YSddWydjaGVja3N1bV9maWxlX3NoYTI1NiddCiAgICByZXR1cm4gcm9vdAoKCmRlZiBkYXRhc2V0KHJvb3QsIHZlcmlmeT1UcnVlKToKICAgIGRlc2lnbiA9IHJlYWQoSEVSRSAvICdQTEFOLmpzb24nKQogICAgb3ZlcmxheSA9IHBsYW4ocm9vdCkKICAgIGFzc2VydCBkaWdlc3Qob3ZlcmxheSkgPT0gZGVzaWduWydkYXRhJ11bJ292ZXJsYXlfc2hhMjU2J10sICdIaXN0b3JpY2FsIG92ZXJsYXkgZHJpZnQnCiAgICBpZiB2ZXJpZnk6CiAgICAgICAgZm9yIGxpbmUgaW4gKHJvb3QgLyAnU0hBMjU2U1VNUy50eHQnKS5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04Jykuc3BsaXRsaW5lcygpOgogICAgICAgICAgICBleHBlY3RlZCwgbmFtZSA9IGxpbmUuc3BsaXQoJyAgJywgMSkKICAgICAgICAgICAgcGF0aCA9IChyb290IC8gbmFtZSkucmVzb2x2ZSgpCiAgICAgICAgICAgIGFzc2VydCBwYXRoLmlzX3JlbGF0aXZlX3RvKHJvb3QpIGFuZCBzaGEocGF0aCkgPT0gZXhwZWN0ZWQsIG5hbWUKICAgIHdpdGggKEhFUkUgLyAnU09VUkNFX0NPTVBPTkVOVFMuY3N2Jykub3BlbihlbmNvZGluZz0ndXRmLTgnLCBuZXdsaW5lPScnKSBhcyBzdHJlYW06CiAgICAgICAgcmVnaXN0cnkgPSB7clsnaW1hZ2VfaWQnXTogciBmb3IgciBpbiBjc3YuRGljdFJlYWRlcihzdHJlYW0pfQogICAgZm9yIHJvdyBpbiBvdmVybGF5Wydyb3dzJ106CiAgICAgICAgcmVmZXJlbmNlID0gcmVnaXN0cnlbcm93WydpbWFnZV9pZCddXQogICAgICAgIGFzc2VydCByb3dbJ2ltYWdlX3NoYTI1NiddID09IHJlZmVyZW5jZVsnaW1hZ2Vfc2hhMjU2J10KICAgICAgICBhc3NlcnQgcm93Wydyb2xlJ10gPT0gcmVmZXJlbmNlWydoaXN0b3JpY2FsX3N0dWR5X3NwbGl0J10KICAgICAgICByb3dbJ2NvbXBvbmVudCddID0gcmVmZXJlbmNlWydsZWFrYWdlX2NvbXBvbmVudCddCiAgICB0cmFpbiA9IHNvcnRlZCgociBmb3IgciBpbiBvdmVybGF5Wydyb3dzJ10gaWYgclsncm9sZSddID09ICd0cmFpbicpLCBrZXk9bGFtYmRhIHI6IHJbJ2ltYWdlX2lkJ10pCiAgICBhc3NlcnQgQ291bnRlcihyWydwb2ludF9raW5kJ10gZm9yIHIgaW4gdHJhaW4pID09IHsnaHVtYW4nOiA3MCwgJ3BvbHlnb25fd2Vhayc6IDE1MiwgJ25vbmUnOiAxNjR9CiAgICBhc3NlcnQgc29ydGVkKENvdW50ZXIoclsnY29tcG9uZW50J10gZm9yIHIgaW4gdHJhaW4pLnZhbHVlcygpKSA9PSBbMiwgMywgNSwgNjAsIDY3LCAyNDldCiAgICByZXR1cm4gb3ZlcmxheSwgdHJhaW4KCgpkZWYgcGhvdG9tZXRyaWMocmdiLCBmYWN0b3JzKToKICAgICIiIkJyaWdodG5lc3MsIGNvbnRyYXN0IGFib3V0IGltYWdlIHNwYXRpYWwgbWVhbiBwZXIgY2hhbm5lbCwgZ2FtbWE7IGNsYW1wIGVhY2ggc3RhZ2UuIiIiCiAgICBiLCBjLCBnID0gZmFjdG9ycy51bmJpbmQoLTEpCiAgICBzaGFwZSA9ICgtMSwgMSwgMSwgMSkKICAgIHggPSAocmdiICogYi5yZXNoYXBlKHNoYXBlKSkuY2xhbXAoMCwgMSkKICAgIG1lYW4gPSB4Lm1lYW4oKC0yLCAtMSksIGtlZXBkaW09VHJ1ZSkKICAgIHggPSAoKHggLSBtZWFuKSAqIGMucmVzaGFwZShzaGFwZSkgKyBtZWFuKS5jbGFtcCgwLCAxKQogICAgcmV0dXJuIHguY2xhbXBfbWluKDFlLTgpLnBvdyhnLnJlc2hhcGUoc2hhcGUpKS5jbGFtcCgwLCAxKQoKCmRlZiBsb2FkX2JhdGNoKHJvb3QsIHJvd3MsIGRldmljZSwgc2VlZHM9Tm9uZSk6CiAgICBhcnJheXMgPSBbc2FtcGxlKHJvb3QsIHIsICg1MTIsIDM4NCksIHNlZWQ9Tm9uZSwgbW9kZWw9J3NlZ2Zvcm1lcicpIGZvciByIGluIHJvd3NdCiAgICByZ2IgPSB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKFthWzBdIGZvciBhIGluIGFycmF5c10pKS50byhkZXZpY2UpCiAgICBmYWN0b3JzID0gbnAub25lcygobGVuKHJvd3MpLCAzKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgIGlmIHNlZWRzIGlzIG5vdCBOb25lOgogICAgICAgIGZhY3RvcnMgPSBucC5zdGFjayhbbnAucmFuZG9tLmRlZmF1bHRfcm5nKHMpLnVuaWZvcm0oLjksIDEuMSwgMykgZm9yIHMgaW4gc2VlZHNdKS5hc3R5cGUoJ2Zsb2F0MzInKQogICAgICAgIHJnYiA9IHBob3RvbWV0cmljKHJnYiwgdG9yY2guZnJvbV9udW1weShmYWN0b3JzKS50byhkZXZpY2UpKQogICAgcmV0dXJuIHsncmdiJzogcmdiLCAnbWFza3MnOiB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKFthWzFdIGZvciBhIGluIGFycmF5c10pKS5mbG9hdCgpLnRvKGRldmljZSksCiAgICAgICAgICAgICdwb2ludHMnOiB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKFthWzJdIGZvciBhIGluIGFycmF5c10pKS50byhkZXZpY2UpLAogICAgICAgICAgICAncG9pbnRfd2VpZ2h0JzogcmdiLm5ld190ZW5zb3IoW3JbJ3BvaW50X3dlaWdodCddIGZvciByIGluIHJvd3NdKSwKICAgICAgICAgICAgJ2h1bWFuJzogdG9yY2gudGVuc29yKFtyWydwb2ludF9raW5kJ10gPT0gJ2h1bWFuJyBmb3IgciBpbiByb3dzXSwgZGV2aWNlPWRldmljZSksCiAgICAgICAgICAgICdndWlkZXMnOiByZ2IubmV3X3RlbnNvcihbclsnZ3VpZGVfZnJhY3Rpb25zJ10gZm9yIHIgaW4gcm93c10pLAogICAgICAgICAgICAnZmFjdG9ycyc6IGZhY3RvcnMsICdyb3dzJzogcm93c30KCgpkZWYgbm9ybWFsaXplKHJnYik6CiAgICByZXR1cm4gKHJnYiAtIHJnYi5uZXdfdGVuc29yKFsuNDg1LCAuNDU2LCAuNDA2XSlbTm9uZSwgOiwgTm9uZSwgTm9uZV0pIC8gcmdiLm5ld190ZW5zb3IoWy4yMjksIC4yMjQsIC4yMjVdKVtOb25lLCA6LCBOb25lLCBOb25lXQo=', 'sha256': 'a5e6aabc95ac3ca87cbfadd7170112879c768dffc8ac7ac6918b9944dfbbecc5'}, 'engineering_geo.py': {'base64': '...', 'sha256': '6a5b77cc7ae0de9839f0f668ff2e4a5a88cd3bda6e1ed753a4e4122a1395d7c7'}, 'evidence_hub.py': {'base64': 'IiIiT25lLW93bmVyIHRyYWluaW5nLWV2aWRlbmNlIHNuYXBzaG90cywgZXhwbGljaXQgdG9rZW4sIGJvdW5kZWQgcmV0cmllcy4NCg0KTm8gdG9rZW4gbG9va3VwIG91dHNpZGUgdGhlIGNhbGxlci1wcm92aWRlZCBLYWdnbGUgc2VjcmV0LiBFdmVyeSBjb29yZGluYXRvcg0KcmVxdWVzdCBpcyByZWNvcmRlZDsgd29ya2VycyBuZXZlciBtYWtlIEh1YiByZXF1ZXN0cy4gTm8gWklQL05QWiBwdWJsaWNhdGlvbi4NCiIiIg0KaW1wb3J0IGpzb24sb3MscmUsdGltZSx1dWlkDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmltcG9ydCByZXF1ZXN0cw0KZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IEhmQXBpLENvbW1pdE9wZXJhdGlvbkFkZA0KZnJvbSBjb21tb24gaW1wb3J0IGF0b21pYyxjYW5vbmljYWwscmVhZCxzaGENCg0KZGVmIGRlbGF5X2ZvcihleGMsYXR0ZW1wdCk6DQogcmVzcG9uc2U9Z2V0YXR0cihleGMsJ3Jlc3BvbnNlJyxOb25lKTtjb2RlPWdldGF0dHIocmVzcG9uc2UsJ3N0YXR1c19jb2RlJyxOb25lKQ0KIHRyYW5zcG9ydD1pc2luc3RhbmNlKGV4YywocmVxdWVzdHMuUmVxdWVzdEV4Y2VwdGlvbixDb25uZWN0aW9uRXJyb3IsVGltZW91dEVycm9yKSkNCiB0cnk6DQogIGltcG9ydCBodHRweA0KICB0cmFuc3BvcnQ9dHJhbnNwb3J0IG9yIGlzaW5zdGFuY2UoZXhjLGh0dHB4LlRyYW5zcG9ydEVycm9yKQ0KIGV4Y2VwdCBJbXBvcnRFcnJvcjpwYXNzDQogaWYgY29kZSBub3QgaW4gKDQyOSw1MDAsNTAyLDUwMyw1MDQpIGFuZCBub3QgKGNvZGUgaXMgTm9uZSBhbmQgdHJhbnNwb3J0KTpyZXR1cm4gTm9uZQ0KIGhlYWRlcnM9Z2V0YXR0cihyZXNwb25zZSwnaGVhZGVycycse30pIG9yIHt9DQogdHJ5OmRlbGF5PWZsb2F0KGhlYWRlcnMuZ2V0KCdSZXRyeS1BZnRlcicsJzAnKSkNCiBleGNlcHQgVmFsdWVFcnJvcjoNCiAgZnJvbSBlbWFpbC51dGlscyBpbXBvcnQgcGFyc2VkYXRlX3RvX2RhdGV0aW1lDQogIGRlbGF5PW1heCgwLixwYXJzZWRhdGVfdG9fZGF0ZXRpbWUoaGVhZGVyc1snUmV0cnktQWZ0ZXInXSkudGltZXN0YW1wKCktdGltZS50aW1lKCkpDQogbWF0Y2g9cmUuc2VhcmNoKHInXGJ0PShcZCspJyxoZWFkZXJzLmdldCgnUmF0ZUxpbWl0JywnJykpDQogcmV0dXJuIG1heChkZWxheSxmbG9hdChtYXRjaC5ncm91cCgxKSkgaWYgbWF0Y2ggZWxzZSAwLG1pbig2MCw1KjIqKmF0dGVtcHQpKQ0KZGVmIHdhaXQoZGVsYXkpOg0KIHByaW50KCdIRiBzZXJ2ZXIvYmFja29mZiBkZWxheTonLGRlbGF5LCdzZWNvbmRzJyxmbHVzaD1UcnVlKQ0KIHdoaWxlIGRlbGF5PjA6DQogIHZhbHVlPW1pbigzMC4sZGVsYXkpO3RpbWUuc2xlZXAodmFsdWUpO2RlbGF5LT12YWx1ZQ0KDQpjbGFzcyBIdWI6DQogUkVQTz0nU2hhbm11azQ2MjIvdHlyZS13ZWFyLXN0dWR5Jw0KIGRlZiBfX2luaXRfXyhzZWxmLGRlc3RpbmF0aW9uLHRva2VuLGxvZyk6DQogIGFzc2VydCB0b2tlbiwnRW5hYmxlIEthZ2dsZSBIRl9UT0tFTi4gTm8gYWx0ZXJuYXRlIGNyZWRlbnRpYWwgZmlsZXMgYXJlIHJlYWQuJw0KICBzZWxmLmRlc3RpbmF0aW9uPWRlc3RpbmF0aW9uO3NlbGYub3duZXI9dXVpZC51dWlkNCgpLmhleDtzZWxmLmxvZz1QYXRoKGxvZyk7c2VsZi5sYXN0X3ZlcmlmaWVkPU5vbmUNCiAgZnJvbSByYXRlX2J1ZGdldCBpbXBvcnQgY29uZmlndXJlLEJ1ZGdldFNlc3Npb24NCiAgYnVkZ2V0PWNvbmZpZ3VyZShzZWxmLmxvZy5wYXJlbnQucGFyZW50LHNlbGYubG9nLnBhcmVudC8nSFRUUF9SRVFVRVNUUy5qc29ubCcpDQogIHNlbGYuYXBpPUhmQXBpKHRva2VuPXRva2VuKTtzZWxmLnNlc3Npb249QnVkZ2V0U2Vzc2lvbihidWRnZXQsc2VsZi5sb2cucGFyZW50LydIVFRQX1JFUVVFU1RTLmpzb25sJyk7c2VsZi5zZXNzaW9uLnRydXN0X2Vudj1GYWxzZQ0KICBzZWxmLnNlc3Npb24uaGVhZGVyc1snQXV0aG9yaXphdGlvbiddPSdCZWFyZXIgJyt0b2tlbjtzZWxmLmNhbGxzPTANCiBkZWYgcmVjb3JkKHNlbGYsb3BlcmF0aW9uLCoqdmFsdWVzKToNCiAgc2VsZi5sb2cucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQ0KICBvYmo9ZGljdCh0aW1lX3V0Yz10aW1lLnN0cmZ0aW1lKCclWS0lbS0lZFQlSDolTTolU1onLHRpbWUuZ210aW1lKCkpLG9wZXJhdGlvbj1vcGVyYXRpb24sKip2YWx1ZXMpDQogIHdpdGggb3BlbihzZWxmLmxvZywnYWInKSBhcyBmOmYud3JpdGUoY2Fub25pY2FsKG9iaikrYidcbicpO2YuZmx1c2goKTtvcy5mc3luYyhmLmZpbGVubygpKQ0KIGRlZiBjYWxsKHNlbGYsZm4sb3BlcmF0aW9uKToNCiAgZm9yIGF0dGVtcHQgaW4gcmFuZ2UoNik6DQogICBzZWxmLmNhbGxzKz0xO2JlZ2luPXRpbWUubW9ub3RvbmljKCkNCiAgIHRyeToNCiAgICByZXN1bHQ9Zm4oKTtzZWxmLnJlY29yZChvcGVyYXRpb24sc3RhdHVzPSdvaycsYXR0ZW1wdD1hdHRlbXB0LHNlY29uZHM9dGltZS5tb25vdG9uaWMoKS1iZWdpbixjb29yZGluYXRvcl9jYWxscz1zZWxmLmNhbGxzKTtyZXR1cm4gcmVzdWx0DQogICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoNCiAgICBkZWxheT1kZWxheV9mb3IoZXhjLGF0dGVtcHQpDQogICAgc2VsZi5yZWNvcmQob3BlcmF0aW9uLHN0YXR1cz0nZXJyb3InLGF0dGVtcHQ9YXR0ZW1wdCxlcnJvcl90eXBlPXR5cGUoZXhjKS5fX25hbWVfXyxodHRwX3N0YXR1cz1nZXRhdHRyKGdldGF0dHIoZXhjLCdyZXNwb25zZScsTm9uZSksJ3N0YXR1c19jb2RlJyxOb25lKSxyZXRyeV9kZWxheT1kZWxheSkNCiAgICBpZiBkZWxheSBpcyBOb25lIG9yIGF0dGVtcHQ9PTU6cmFpc2UNCiAgICB3YWl0KGRlbGF5KQ0KIGRlZiBoZWFkKHNlbGYpOnJldHVybiBzZWxmLmNhbGwobGFtYmRhOnNlbGYuYXBpLnJlcG9faW5mbyhzZWxmLlJFUE8scmVwb190eXBlPSdkYXRhc2V0Jykuc2hhLCdyZXBvX2luZm8nKQ0KIGRlZiBnZXRfanNvbihzZWxmLG5hbWUscmV2aXNpb24pOg0KICBkZWYgbG9hZCgpOg0KICAgdXJsPWYnaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9kYXRhc2V0cy97c2VsZi5SRVBPfS9yZXNvbHZlL3tyZXZpc2lvbn0ve3NlbGYuZGVzdGluYXRpb259L3tuYW1lfScNCiAgIHdpdGggc2VsZi5zZXNzaW9uLmdldCh1cmwsdGltZW91dD0oMjAsMTIwKSkgYXMgcjoNCiAgICBpZiByLnN0YXR1c19jb2RlPT00MDQ6cmV0dXJuIE5vbmUNCiAgICByLnJhaXNlX2Zvcl9zdGF0dXMoKTtyZXR1cm4gci5qc29uKCkNCiAgcmV0dXJuIHNlbGYuY2FsbChsb2FkLCdyZXNvbHZlX2pzb246JytuYW1lKQ0KIGRlZiBjbGFpbShzZWxmKToNCiAgYXNzZXJ0IHNlbGYuY2FsbChsYW1iZGE6c2VsZi5hcGkud2hvYW1pKCksJ3dob2FtaScpWyduYW1lJ10ubG93ZXIoKT09J3NoYW5tdWs0NjIyJw0KICBmb3IgYXR0ZW1wdCBpbiByYW5nZSg4KToNCiAgIHBhcmVudD1zZWxmLmhlYWQoKTtvd25lcj1zZWxmLmdldF9qc29uKCdPV05FUi5qc29uJyxwYXJlbnQpDQogICBpZiBvd25lciBhbmQgb3duZXJbJ293bmVyJ109PXNlbGYub3duZXIgYW5kIG93bmVyWydleHBpcmVzJ10+dGltZS50aW1lKCk6DQogICAgc2VsZi5yZWNvcmQoJ2NsYWltJyxzdGF0dXM9J2xvc3RfcmVzcG9uc2VfYWNrbm93bGVkZ2VkJyk7cmV0dXJuDQogICBpZiBvd25lciBhbmQgb3duZXJbJ293bmVyJ10gbm90IGluIChOb25lLHNlbGYub3duZXIpIGFuZCBvd25lclsnZXhwaXJlcyddPnRpbWUudGltZSgpOnJhaXNlIFJ1bnRpbWVFcnJvcignQW5vdGhlciBsaXZlIGdhdGUgb3ducyB0aGlzIG5hbWVzcGFjZTsgc3RvcCBpdCBjbGVhbmx5IG9yIHdhaXQgZm9yIGl0cyA0NS1taW51dGUgbGVhc2UgZXhwaXJ5LicpDQogICB0cnk6DQogICAgc2VsZi5hcGkuY3JlYXRlX2NvbW1pdChzZWxmLlJFUE8scmVwb190eXBlPSdkYXRhc2V0JyxwYXJlbnRfY29tbWl0PXBhcmVudCwNCiAgICAgIG9wZXJhdGlvbnM9W0NvbW1pdE9wZXJhdGlvbkFkZChwYXRoX2luX3JlcG89c2VsZi5kZXN0aW5hdGlvbisnL09XTkVSLmpzb24nLHBhdGhfb3JfZmlsZW9iaj1jYW5vbmljYWwoZGljdChvd25lcj1zZWxmLm93bmVyLGV4cGlyZXM9dGltZS50aW1lKCkrMjcwMCkpKV0sDQogICAgICBjb21taXRfbWVzc2FnZT0nVFJFQUQtVFdPIEFUTEFTIGV2aWRlbmNlLWdhdGUgd3JpdGVyIGNsYWltJykNCiAgICBzZWxmLnJlY29yZCgnY2xhaW0nLHN0YXR1cz0ndmVyaWZpZWRfYnlfbmV4dF9wYXJlbnRfcmVhZCcpO3JldHVybg0KICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6DQogICAgY29kZT1nZXRhdHRyKGdldGF0dHIoZXhjLCdyZXNwb25zZScsTm9uZSksJ3N0YXR1c19jb2RlJyxOb25lKTtkZWxheT1kZWxheV9mb3IoZXhjLGF0dGVtcHQpDQogICAgc2VsZi5yZWNvcmQoJ2NsYWltJyxzdGF0dXM9J2Vycm9yJyxhdHRlbXB0PWF0dGVtcHQsZXJyb3JfdHlwZT10eXBlKGV4YykuX19uYW1lX18saHR0cF9zdGF0dXM9Y29kZSxyZXRyeV9kZWxheT1kZWxheSkNCiAgICBpZiBjb2RlIG5vdCBpbiAoNDA5LDQxMikgYW5kIChkZWxheSBpcyBOb25lIG9yIGF0dGVtcHQ9PTcpOnJhaXNlDQogICAgaWYgZGVsYXkgaXMgbm90IE5vbmU6d2FpdChkZWxheSkNCiAgcmFpc2UgUnVudGltZUVycm9yKCdVbmFibGUgdG8gY2xhaW0gZW5naW5lZXJpbmcgd3JpdGVyJykNCiBkZWYgcHVibGlzaChzZWxmLHNuYXBzaG90LHJlbGVhc2U9RmFsc2UpOg0KICBzbmFwc2hvdD1QYXRoKHNuYXBzaG90KTttYW5pZmVzdD1yZWFkKHNuYXBzaG90LydNQU5JRkVTVC5qc29uJyk7ZmlsZXM9ZGljdChtYW5pZmVzdFsnZmlsZXMnXSkNCiAgYXNzZXJ0IG5vdCBhbnkobi5lbmRzd2l0aCgoJy56aXAnLCcubnB6JykpIGZvciBuIGluIGZpbGVzKQ0KICBmaWxlc1snTUFOSUZFU1QuanNvbiddPWRpY3Qoc2hhMjU2PXNoYShzbmFwc2hvdC8nTUFOSUZFU1QuanNvbicpLGJ5dGVzPShzbmFwc2hvdC8nTUFOSUZFU1QuanNvbicpLnN0YXQoKS5zdF9zaXplKQ0KICBmb3IgYXR0ZW1wdCBpbiByYW5nZSg4KToNCiAgIHBhcmVudD1zZWxmLmhlYWQoKTtvd25lcj1zZWxmLmdldF9qc29uKCdPV05FUi5qc29uJyxwYXJlbnQpDQogICBhc3NlcnQgb3duZXIgYW5kIG93bmVyWydvd25lciddPT1zZWxmLm93bmVyLCdTdGFsZSB3cml0ZXIgZmVuY2VkIGJlZm9yZSBwdWJsaWNhdGlvbicNCiAgIHByZXZpb3VzPXNlbGYuZ2V0X2pzb24oJ01BTklGRVNULmpzb24nLHBhcmVudCkNCiAgIGNoYW5nZWQ9W24gZm9yIG4sbSBpbiBmaWxlcy5pdGVtcygpIGlmIG49PSdNQU5JRkVTVC5qc29uJyBvciBub3QgcHJldmlvdXMgb3IgcHJldmlvdXNbJ2ZpbGVzJ10uZ2V0KG4pIT1tXQ0KICAgb3BlcmF0aW9ucz1bQ29tbWl0T3BlcmF0aW9uQWRkKHBhdGhfaW5fcmVwbz1zZWxmLmRlc3RpbmF0aW9uKycvJytuLHBhdGhfb3JfZmlsZW9iaj1zdHIoc25hcHNob3QvbikpIGZvciBuIGluIGNoYW5nZWRdDQogICBvcGVyYXRpb25zLmFwcGVuZChDb21taXRPcGVyYXRpb25BZGQocGF0aF9pbl9yZXBvPXNlbGYuZGVzdGluYXRpb24rJy9PV05FUi5qc29uJyxwYXRoX29yX2ZpbGVvYmo9Y2Fub25pY2FsKGRpY3Qob3duZXI9Tm9uZSBpZiByZWxlYXNlIGVsc2Ugc2VsZi5vd25lcixleHBpcmVzPTAgaWYgcmVsZWFzZSBlbHNlIHRpbWUudGltZSgpKzI3MDApKSkpDQogICB0cnk6DQogICAgY29tbWl0PXNlbGYuYXBpLmNyZWF0ZV9jb21taXQoc2VsZi5SRVBPLHJlcG9fdHlwZT0nZGF0YXNldCcscGFyZW50X2NvbW1pdD1wYXJlbnQsb3BlcmF0aW9ucz1vcGVyYXRpb25zLA0KICAgICAgY29tbWl0X21lc3NhZ2U9J1RSRUFELVRXTyBBVExBUyBlbmdpbmVlcmluZyBnYXRlOiBkaXJlY3QgQ1NWIGFuZCBjb21wbGV0ZSBvcHRpbWl6ZXIvUk5HIHN0YXRlJykNCiAgICByZXZpc2lvbj1jb21taXQub2lkDQogICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoNCiAgICBjdXJyZW50PXNlbGYuaGVhZCgpO3JlbW90ZT1zZWxmLmdldF9qc29uKCdNQU5JRkVTVC5qc29uJyxjdXJyZW50KTthY3R1YWxfb3duZXI9c2VsZi5nZXRfanNvbignT1dORVIuanNvbicsY3VycmVudCkNCiAgICBpZiByZW1vdGU9PW1hbmlmZXN0IGFuZCBhY3R1YWxfb3duZXIgYW5kIGFjdHVhbF9vd25lclsnb3duZXInXT09KE5vbmUgaWYgcmVsZWFzZSBlbHNlIHNlbGYub3duZXIpOnJldmlzaW9uPWN1cnJlbnQNCiAgICBlbHNlOg0KICAgICBjb2RlPWdldGF0dHIoZ2V0YXR0cihleGMsJ3Jlc3BvbnNlJyxOb25lKSwnc3RhdHVzX2NvZGUnLE5vbmUpO2RlbGF5PWRlbGF5X2ZvcihleGMsYXR0ZW1wdCkNCiAgICAgaWYgY29kZSBub3QgaW4gKDQwOSw0MTIpIGFuZCAoZGVsYXkgaXMgTm9uZSBvciBhdHRlbXB0PT03KTpyYWlzZQ0KICAgICBpZiBkZWxheSBpcyBub3QgTm9uZTp3YWl0KGRlbGF5KQ0KICAgICBjb250aW51ZQ0KICAgaW5mb3M9c2VsZi5jYWxsKGxhbWJkYTpsaXN0KHNlbGYuYXBpLmxpc3RfcmVwb190cmVlKHNlbGYuUkVQTyxwYXRoX2luX3JlcG89c2VsZi5kZXN0aW5hdGlvbixyZXZpc2lvbj1yZXZpc2lvbixyZWN1cnNpdmU9VHJ1ZSxyZXBvX3R5cGU9J2RhdGFzZXQnKSksJ3ZlcmlmeV90cmVlJykNCiAgIGxvb2t1cD17di5wYXRoW2xlbihzZWxmLmRlc3RpbmF0aW9uKSsxOl06diBmb3IgdiBpbiBpbmZvcyBpZiBoYXNhdHRyKHYsJ3NpemUnKX0NCiAgIGltcG9ydCBoYXNobGliDQogICBmb3IgbmFtZSxzcGVjIGluIGZpbGVzLml0ZW1zKCk6DQogICAgaW5mbz1sb29rdXBbbmFtZV07YXNzZXJ0IGluZm8uc2l6ZT09c3BlY1snYnl0ZXMnXQ0KICAgIGlmIGluZm8ubGZzOmFzc2VydCBpbmZvLmxmcy5zaGEyNTY9PXNwZWNbJ3NoYTI1NiddLG5hbWUNCiAgICBlbHNlOg0KICAgICBib2R5PShzbmFwc2hvdC9uYW1lKS5yZWFkX2J5dGVzKCk7YXNzZXJ0IGluZm8uYmxvYl9pZD09aGFzaGxpYi5zaGExKGInYmxvYiAnK3N0cihsZW4oYm9keSkpLmVuY29kZSgpK2InXDAnK2JvZHkpLmhleGRpZ2VzdCgpLG5hbWUNCiAgIHNlbGYubGFzdF92ZXJpZmllZD1yZXZpc2lvbg0KICAgcmVzdWx0PWRpY3Qoc3RhdHVzPSdwdWJsaXNoZWRfYW5kX3ZlcmlmaWVkJyxyZXZpc2lvbj1yZXZpc2lvbixkZXN0aW5hdGlvbj1zZWxmLmRlc3RpbmF0aW9uLGZpbGVzPWxlbihmaWxlcyksemlwX3VwbG9hZHM9MCxvd25lcl9yZWxlYXNlZD1yZWxlYXNlKQ0KICAgc2VsZi5yZWNvcmQoJ3B1Ymxpc2gnLCoqcmVzdWx0KTtyZXR1cm4gcmVzdWx0DQogIHJhaXNlIFJ1bnRpbWVFcnJvcignUHVibGljYXRpb24gZXhoYXVzdGVkOyBsb2NhbCBjb21taXR0ZWQgdXBkYXRlcyByZW1haW4gcmV0YWluZWQuJykNCiBkZWYgZG93bmxvYWQoc2VsZixyZXZpc2lvbixmb2xkZXIscHJvdG9jb2wpOg0KICBtYW5pZmVzdD1zZWxmLmdldF9qc29uKCdNQU5JRkVTVC5qc29uJyxyZXZpc2lvbik7YXNzZXJ0IG1hbmlmZXN0IGFuZCBtYW5pZmVzdFsnY29udHJhY3QnXT09cHJvdG9jb2wNCiAgZm9sZGVyPVBhdGgoZm9sZGVyKTtmb2xkZXIubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpDQogIGZvciBuYW1lLHNwZWMgaW4gbWFuaWZlc3RbJ2ZpbGVzJ10uaXRlbXMoKToNCiAgIGFzc2VydCBub3QgUGF0aChuYW1lKS5pc19hYnNvbHV0ZSgpIGFuZCAnLi4nIG5vdCBpbiBQYXRoKG5hbWUpLnBhcnRzDQogICB0YXJnZXQ9Zm9sZGVyL25hbWU7dGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkNCiAgIGlmIHRhcmdldC5leGlzdHMoKSBhbmQgc2hhKHRhcmdldCk9PXNwZWNbJ3NoYTI1NiddOmNvbnRpbnVlDQogICBkZWYgdHJhbnNmZXIoKToNCiAgICB1cmw9ZidodHRwczovL2h1Z2dpbmdmYWNlLmNvL2RhdGFzZXRzL3tzZWxmLlJFUE99L3Jlc29sdmUve3JldmlzaW9ufS97c2VsZi5kZXN0aW5hdGlvbn0ve25hbWV9Jw0KICAgIHBhcnQ9dGFyZ2V0LndpdGhfbmFtZSh0YXJnZXQubmFtZSsnLnBhcnQnKQ0KICAgIHdpdGggc2VsZi5zZXNzaW9uLmdldCh1cmwsdGltZW91dD0oMjAsMTIwKSxzdHJlYW09VHJ1ZSkgYXMgcmVzcG9uc2U6DQogICAgIHJlc3BvbnNlLnJhaXNlX2Zvcl9zdGF0dXMoKQ0KICAgICB3aXRoIG9wZW4ocGFydCwnd2InKSBhcyBmOg0KICAgICAgZm9yIGJsb2NrIGluIHJlc3BvbnNlLml0ZXJfY29udGVudCg0KjEwMjQqKjIpOmYud3JpdGUoYmxvY2spDQogICAgcGFydC5yZXBsYWNlKHRhcmdldCkNCiAgIHNlbGYuY2FsbCh0cmFuc2ZlciwncmVzb2x2ZV9maWxlOicrbmFtZSkNCiAgIGFzc2VydCB0YXJnZXQuc3RhdCgpLnN0X3NpemU9PXNwZWNbJ2J5dGVzJ10gYW5kIHNoYSh0YXJnZXQpPT1zcGVjWydzaGEyNTYnXSxuYW1lDQogIGF0b21pYyhmb2xkZXIvJ01BTklGRVNULmpzb24nLG1hbmlmZXN0KQ0KICByZXR1cm4gZm9sZGVyDQo=', 'sha256': '03da70c79b10cfaf307bd29c81f90747d45440c7eb7e2a0d19f1feef980c7b9f'}, 'FILES.json': {'base64': 'eyJCVURHRVQuanNvbiI6eyJieXRlcyI6MTIzMCwic2hhMjU2IjoiNDdmZTgxZGM4ZTljZjU2NjEzZTZmOWJmMzIwMTgyMWIyYzQxMmU3ZmIzYzA3YWI5NTE1MzRhNzkxMmY1M2YxYiJ9LCJQTEFOLmpzb24iOnsiYnl0ZXMiOjY5NTUsInNoYTI1NiI6ImE1OGU3MDY0YTI2ZDQ0NzI4MTJjMDNjYzQ1MzMyY2JhMmYwYmZhNTZjZWU3NzdhYWY0YjViZGE5YzlhMzc0MGMifSwiU0NIRU1BUy5qc29uIjp7ImJ5dGVzIjoxNzMyNDIsInNoYTI1NiI6IjdkZGZiZDcxZDNhYTg5MmNhYmY3OTc4MjdiYzgwYWJhMmJkNTM2ODQ4ZGI3NDA5MjMyYTYzMjhiZDM3MDNhODAifSwiU09VUkNFX0NPTVBPTkVOVFMuY3N2Ijp7ImJ5dGVzIjoyMDA3NTMsInNoYTI1NiI6IjI1ZmMwZmQyY2U1NTkyMjk3YzNiMWZlMDkyOTk3MGRiOWM5NzA0MDIyN2IzYjRmYjcxYmUxMDdhNWUzZTI5M2YifSwiVkVORE9SRUQuanNvbiI6eyJieXRlcyI6MTU4NCwic2hhMjU2IjoiMTdmY2Y3YWJjMjhiY2RhNzFmOWRkOGUwNzg3MzI0Y2E3MWU0YTAyNzFhMzE5MWJkNTI2ZThhYTgzMjUzZDMwNyJ9LCJhbmNob3JfZ2VvLnB5Ijp7ImJ5dGVzIjoxNjIzLCJzaGEyNTYiOiJhMmRmYWE3Njc5YjAzNzg2ZGVmMzc0NWI3MDdhOTdmNTQwMzljYzI4ZjlhYjI1MTg5YmQzNDMwYjgyN2QyZDk4In0sImF0bGFzX2dlby5weSI6eyJieXRlcyI6MjI2Mywic2hhMjU2IjoiZjVlNWU2OWZkNTMxYzFkMTMxZjg0ZDdiODdiMmI5ODU4ZmM1OThiYjIyYTZiZGNjMDUwODg1ZmY5MzM5MmUzMSJ9LCJjaGVja3NfZ2VvLnB5Ijp7ImJ5dGVzIjo3NDQwLCJzaGEyNTYiOiJlOWY3YmI3ZmM4ZDIzMjJhOGEwMmNiZTQ5YzgwZjM2YTNlNzJiYzUyYjk0NDYzZDE1MWQyOTQ1ZjEwMzJjYjUzIn0sImNvbW1vbi5weSI6eyJieXRlcyI6NTExNiwic2hhMjU2IjoiOTIyYzgyYjgyNjM1MzEzY2U2MWVjYzI5OTI5OTBmNWJjMGI3ZjNjYjFkNmM4ZmNhMDA1MjdkZTcyYTM0YTIwNiJ9LCJjb29yZGluYXRvcl9nZW8ucHkiOnsiYnl0ZXMiOjE1MjQ2LCJzaGEyNTYiOiJlZDU3NTkwNDkwY2MyNWRjODQ5MGYxZmMzNGNkYjM1NGQ1ZTIxNWRiMDg0MWRjMGI5NDdhZGNlZDYzZWQ0OTVmIn0sImRhdGFfZ2VvLnB5Ijp7ImJ5dGVzIjozNjI2LCJzaGEyNTYiOiJhNWU2YWFiYzk1YWMzY2E4N2NiZmFkZDcxNzAxMTI4NzljNzY4ZGZmYzhhYzdhYzY5MThiOTk0NGRmYmJlY2M1In0sImVuZ2luZWVyaW5nX2dlby5weSI6eyJieXRlcyI6MjA3MTMsInNoYTI1NiI6IjZhNWI3N2NjN2FlMGRlOTgzOWYwZjY2OGZmMmU0YTVhODhjZDNiZGE2ZTFlZDc1M2E0ZTQxMjJhMTM5NWQ3YzcifSwiZXZpZGVuY2VfaHViLnB5Ijp7ImJ5dGVzIjo4Mjg4LCJzaGEyNTYiOiIwM2RhNzBjNzliMTBjZmFmMzA3YmQyOWM4MWY5MDc0N2Q0NTQ0MGM3ZWI3ZTJhMGQxOWYxZmVlZjk4MGM3YjlmIn0sImZyb3plbl9hbmNob3IucHkiOnsiYnl0ZXMiOjg1MDIsInNoYTI1NiI6ImQxNmNkNGVlZjc1Nzc5MDQ0ZGQ5NTkwMGMwNTMxZmVlNDc0NzA0Y2Y4OGY1MjMzMjZlZTA2OTExNTM0N2JhMDkifSwiZ2VvbWV0cnkucHkiOnsiYnl0ZXMiOjIzMzgsInNoYTI1NiI6IjA0YTgxNTFmNjY2Yzk2YWI3NDgyYzFhYTQ4YzJhMmVhN2UxNWQzNGRiMDk4OTNiZTNhN2FkMzFlNTA3ZDM2NmEifSwibGNfZmVhdHVyZXMucHkiOnsiYnl0ZXMiOjEyNzYsInNoYTI1NiI6ImQ0MTdlNTUyYjkxMDgzNmQ4ZGYyNzM0YmI1NDIzZjdkY2VlNzk1NmY0ODViMDIwNDY3MzlhZDFiZWY1NzQxZDYifSwibGVnYWN5X2RhdGEucHkiOnsiYnl0ZXMiOjI4NjUsInNoYTI1NiI6ImExNWJlZDBlMjU3NDk4MGMyYTIzMzlhOGE5MDA2MTNjYzkyOTJjNjM0MmQyZWU1ZTE2ZTFmNWEwNTY5OTU1YWUifSwibW9kZWxfZ2VvLnB5Ijp7ImJ5dGVzIjo3Nzc3LCJzaGEyNTYiOiIxMjRlMGY5OTk4ZTEyNDI1Y2M2Y2Q2YWJlNjJjNmQ0ZTk3Y2MxMDkyNmNhYmMzODkzNWE3ZDMyY2VmMWVhNTAyIn0sIm9iamVjdGl2ZV9nZW8ucHkiOnsiYnl0ZXMiOjYzMjAsInNoYTI1NiI6IjgyMzg3YjM1ZjgxYWY0Yjk3ZWZhN2Y0NTczY2M0NTNhMWI0YzFmZjVhZGFhMGUzZjFkM2U0YzVjZTU4YTExNWYifSwicGhhc2UyX3RyYWluaW5nX2RhdGEucHkiOnsiYnl0ZXMiOjcxNTQsInNoYTI1NiI6IjYwZjk3NTQwNzg5NDllY2FjNTcwNGI3M2VmOTMwN2I3Yjk1YzJjYzhhY2MxYTIwZGYyZjM0NWQwOGQ3ZTI1NzkifSwicHVibGlzaGVyLnB5Ijp7ImJ5dGVzIjo2MDg5LCJzaGEyNTYiOiIwZWE3N2Q1NTg0YTM5MDNiMDFmZGY3NDY2ZGFhOTVlYjdiZTEwMjFiNGU3MGFjNzZhNjcwNjUzMWEzZDU2MzY0In0sInJhdGVfYnVkZ2V0LnB5Ijp7ImJ5dGVzIjo3MzQ0LCJzaGEyNTYiOiI1NWE4ZGYxZWU5ZmI0OTgyNjY4NTg1MTRkNjU0Yzg5ZGRkNTI3NmQ3ZjhlNGRkN2JiOTZiYWE1NzI3N2U1MjM4In0sInJlcG9ydF9nZW8ucHkiOnsiYnl0ZXMiOjQwNTAsInNoYTI1NiI6ImQ4MWNiMmVjNjdlNjY4MTlkZjUyNmM3NzUzOTgwNjg1ZWFiMjdkODY5NTVhZDllOWEzOTllNDA5YzU5MTM1MWUifSwic2NoZWR1bGVfZ2VvLnB5Ijp7ImJ5dGVzIjoyNjk1LCJzaGEyNTYiOiJjNDZjZjMwODQ4YzQxNmQ4NjZiMzQyMzhkYzRhZDRhNGViMWNlYzQzOGIxNWQ5MDQ1OTA1MjEyYzM4OTkyNzcyIn0sInN0b3JlX2dlby5weSI6eyJieXRlcyI6NTY1Mywic2hhMjU2IjoiMmUyODI0ZDVkYWQ5YTlkYjY2NTQxMDk2NDFhOTE1Y2I2OWY5ZDFmZDgwODNjZDE0ODM1NTQ2MjQ1NTU2YTE4NCJ9LCJ3b3JrZXJfZ2VvLnB5Ijp7ImJ5dGVzIjoxMTYxLCJzaGEyNTYiOiI5YzZmYjAzM2MwNzBjMjllYzNiZWY2MmE5NjY4ZGJiN2FiYTIxZTNlMjBhNDkwNGRkMjc2ZDQzYjNiMjQyYzAyIn19', 'sha256': 'f69ad741497c7acc7199e64680c0c59aa14152f7b0d3b6c3481c82913597368a'}, 'frozen_anchor.py': {'base64': 'IiIiVFJFQUQtT05FIHIxOiBjdXN0b20gdHJhaW4tZnJvbS1zY3JhdGNoIG11bHRpLXRhc2sgZXZpZGVuY2UgbmV0d29yay4KCk5vIGRvd25sb2FkZWQgYmFja2JvbmUsIGRldGVjdG9yLCB0ZWFjaGVyIG9yIHBvc3QtaG9jIENBTSBkZXBlbmRlbmN5LgoiIiIKaW1wb3J0IG1hdGgKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgpmcm9tIHRvcmNoLm5uIGltcG9ydCBmdW5jdGlvbmFsIGFzIEYKCkdVSURFUyA9ICgzODQvMTUzNSwgNzY4LzE1MzUsIDExNTEvMTUzNSkKTU9ERUxfVkVSU0lPTiA9ICd0cmVhZC1vbmUtcjEnCgoKY2xhc3MgQ29udihubi5TZXF1ZW50aWFsKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjaW4sIGNvdXQsIGtlcm5lbD0zLCBzdHJpZGU9MSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXyhubi5Db252MmQoY2luLCBjb3V0LCBrZXJuZWwsIHN0cmlkZSwga2VybmVsLy8yLCBiaWFzPUZhbHNlKSwKICAgICAgICAgICAgICAgICAgICAgICAgIG5uLkdyb3VwTm9ybSg4LCBjb3V0KSwgbm4uU2lMVSgpKQoKCmNsYXNzIFRleHR1cmVCbG9jayhubi5Nb2R1bGUpOgogICAgIiIiTG9jYWwgZ3Jvb3ZlcyBhbmQgd2lkZXIgdGV4dHVyZSBjb250ZXh0LCB3aXRoIGV4cGxpY2l0IHJlc2lkdWFsIHNjYWxpbmcuIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2hhbm5lbHMsIGRpbGF0aW9uKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnNwYXRpYWwgPSBubi5Db252MmQoY2hhbm5lbHMsIGNoYW5uZWxzLCA1LCBwYWRkaW5nPTIqZGlsYXRpb24sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRpbGF0aW9uPWRpbGF0aW9uLCBncm91cHM9Y2hhbm5lbHMpCiAgICAgICAgc2VsZi5taXggPSBubi5TZXF1ZW50aWFsKG5uLkdyb3VwTm9ybSg4LCBjaGFubmVscyksIG5uLkNvbnYyZChjaGFubmVscywgMipjaGFubmVscywgMSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5uLlNpTFUoKSwgbm4uQ29udjJkKDIqY2hhbm5lbHMsIGNoYW5uZWxzLCAxKSkKICAgICAgICBzZWxmLnNjYWxlID0gbm4uUGFyYW1ldGVyKHRvcmNoLmZ1bGwoKDEsIGNoYW5uZWxzLCAxLCAxKSwgMC4xKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICByZXR1cm4geCArIHNlbGYuc2NhbGUqc2VsZi5taXgoc2VsZi5zcGF0aWFsKHgpKQoKCmNsYXNzIENvbnRleHRCbG9jayhubi5Nb2R1bGUpOgogICAgIiIiT3JkaW5hcnkgZXhwbGljaXQgYXR0ZW50aW9uIG1hdGg6IG5vIGZsYXNoL1NEUEEgYmFja2VuZCBhbWJpZ3VpdHkuIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2hhbm5lbHM9MTYwLCBoZWFkcz00KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmhlYWRzID0gaGVhZHMKICAgICAgICBzZWxmLm5vcm0gPSBubi5MYXllck5vcm0oY2hhbm5lbHMpCiAgICAgICAgc2VsZi5xa3YgPSBubi5MaW5lYXIoY2hhbm5lbHMsIGNoYW5uZWxzKjMpCiAgICAgICAgc2VsZi5wcm9qID0gbm4uTGluZWFyKGNoYW5uZWxzLCBjaGFubmVscykKICAgICAgICBzZWxmLmZmID0gbm4uU2VxdWVudGlhbChubi5MYXllck5vcm0oY2hhbm5lbHMpLCBubi5MaW5lYXIoY2hhbm5lbHMsIGNoYW5uZWxzKjIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5uLkdFTFUoKSwgbm4uTGluZWFyKGNoYW5uZWxzKjIsIGNoYW5uZWxzKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBiLCBjLCBoLCB3ID0geC5zaGFwZQogICAgICAgIHogPSB4LmZsYXR0ZW4oMikudHJhbnNwb3NlKDEsIDIpCiAgICAgICAgcSwgaywgdiA9IHNlbGYucWt2KHNlbGYubm9ybSh6KSkucmVzaGFwZShiLCBoKncsIDMsIHNlbGYuaGVhZHMsIGMvL3NlbGYuaGVhZHMpLnBlcm11dGUoMiwgMCwgMywgMSwgNCkKICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KHguZGV2aWNlLnR5cGUsIGVuYWJsZWQ9RmFsc2UpOgogICAgICAgICAgICB3ZWlnaHRzID0gKHEuZmxvYXQoKSBAIGsuZmxvYXQoKS50cmFuc3Bvc2UoLTEsIC0yKSkvbWF0aC5zcXJ0KGMvL3NlbGYuaGVhZHMpCiAgICAgICAgICAgIHkgPSB3ZWlnaHRzLnNvZnRtYXgoLTEpIEAgdi5mbG9hdCgpCiAgICAgICAgeSA9IHkudG8oei5kdHlwZSkudHJhbnNwb3NlKDEsIDIpLnJlc2hhcGUoYiwgaCp3LCBjKQogICAgICAgIHogPSB6ICsgc2VsZi5wcm9qKHkpCiAgICAgICAgeiA9IHogKyBzZWxmLmZmKHopCiAgICAgICAgcmV0dXJuIHoudHJhbnNwb3NlKDEsIDIpLnJlc2hhcGUoYiwgYywgaCwgdykKCgpjbGFzcyBUcmVhZE9uZShubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbmNlcHRzPTEyKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnN0ZW0gPSBubi5TZXF1ZW50aWFsKENvbnYoMywgMjQsIHN0cmlkZT0yKSwgQ29udigyNCwgMzIsIHN0cmlkZT0yKSkKICAgICAgICBzZWxmLnMxID0gbm4uU2VxdWVudGlhbChUZXh0dXJlQmxvY2soMzIsIDEpLCBUZXh0dXJlQmxvY2soMzIsIDIpKQogICAgICAgIHNlbGYuczIgPSBubi5TZXF1ZW50aWFsKENvbnYoMzIsIDY0LCBzdHJpZGU9MiksIFRleHR1cmVCbG9jayg2NCwgMSksIFRleHR1cmVCbG9jayg2NCwgMikpCiAgICAgICAgc2VsZi5zMyA9IG5uLlNlcXVlbnRpYWwoQ29udig2NCwgMTEyLCBzdHJpZGU9MiksIFRleHR1cmVCbG9jaygxMTIsIDEpLCBUZXh0dXJlQmxvY2soMTEyLCAyKSkKICAgICAgICBzZWxmLnM0ID0gbm4uU2VxdWVudGlhbChDb252KDExMiwgMTYwLCBzdHJpZGU9MiksIFRleHR1cmVCbG9jaygxNjAsIDEpKQogICAgICAgIHNlbGYucG9zaXRpb24gPSBubi5Db252MmQoMiwgMTYwLCAxKQogICAgICAgIHNlbGYuY29udGV4dCA9IG5uLlNlcXVlbnRpYWwoQ29udGV4dEJsb2NrKCksIENvbnRleHRCbG9jaygpKQogICAgICAgIHNlbGYuZDMsIHNlbGYuZDIsIHNlbGYuZDEgPSBDb252KDI3MiwgMTEyKSwgQ29udigxNzYsIDY0KSwgQ29udig5NiwgNDgpCiAgICAgICAgc2VsZi5zZWcgPSBubi5Db252MmQoNDgsIDIsIDEpCiAgICAgICAgc2VsZi5jb25jZXB0cyA9IG5uLkNvbnYyZCg0OCwgY29uY2VwdHMsIDEpCiAgICAgICAgc2VsZi5jbGFzc193ZWlnaHRzID0gbm4uUGFyYW1ldGVyKHRvcmNoLnJhbmRuKDMsIGNvbmNlcHRzKSowLjAzKQogICAgICAgIHNlbGYuY2xhc3NfYmlhcyA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcygzKSkKICAgICAgICBzZWxmLnBvaW50X2ZlYXR1cmVzID0gbm4uQ29udjJkKDQ4LCAxNiwgMSkKICAgICAgICBzZWxmLnBvaW50cyA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKDE2KjE2LCA5NiksIG5uLlNpTFUoKSwgbm4uTGluZWFyKDk2LCAyKSkKICAgICAgICBubi5pbml0Lnplcm9zXyhzZWxmLnBvaW50c1stMV0ud2VpZ2h0KQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBzZWxmLnBvaW50c1stMV0uYmlhcy5jb3B5Xyh0b3JjaC50ZW5zb3IoWy0xLjk1LCAxLjc5XSkpCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIHVwKHgsIHNraXApOgogICAgICAgICMgSW50ZWdlciByZXBlYXQvY3JvcCBrZWVwcyBiYWNrd2FyZCBkZXRlcm1pbmlzdGljIGF0IGFsbCBjb25maWd1cmVkIHNpemVzLgogICAgICAgIHggPSB4LnJlcGVhdF9pbnRlcmxlYXZlKDIsIC0yKS5yZXBlYXRfaW50ZXJsZWF2ZSgyLCAtMSkKICAgICAgICByZXR1cm4gdG9yY2guY2F0KCh4Wy4uLiwgOnNraXAuc2hhcGVbLTJdLCA6c2tpcC5zaGFwZVstMV1dLCBza2lwKSwgMSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpbWFnZSwgZ3VpZGVzPU5vbmUpOgogICAgICAgIGlmIGltYWdlLnNoYXBlWy0yXSAlIDMyIG9yIGltYWdlLnNoYXBlWy0xXSAlIDY0OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbnB1dCBoZWlnaHQgbXVzdCBkaXZpZGUgYnkgMzIgYW5kIHdpZHRoIGJ5IDY0LicpCiAgICAgICAgYSA9IHNlbGYuczEoc2VsZi5zdGVtKGltYWdlKSkKICAgICAgICBiID0gc2VsZi5zMihhKQogICAgICAgIGMgPSBzZWxmLnMzKGIpCiAgICAgICAgeiA9IHNlbGYuczQoYykKICAgICAgICB5eSwgeHggPSB0b3JjaC5tZXNoZ3JpZCh0b3JjaC5saW5zcGFjZSgtMSwgMSwgei5zaGFwZVstMl0sIGRldmljZT16LmRldmljZSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRvcmNoLmxpbnNwYWNlKC0xLCAxLCB6LnNoYXBlWy0xXSwgZGV2aWNlPXouZGV2aWNlKSwgaW5kZXhpbmc9J2lqJykKICAgICAgICBwb3NpdGlvbiA9IHRvcmNoLnN0YWNrKCh4eCwgeXkpKVtOb25lXS50byh6LmR0eXBlKQogICAgICAgIHogPSBzZWxmLmNvbnRleHQoeiArIHNlbGYucG9zaXRpb24ocG9zaXRpb24pKQogICAgICAgIGZlYXR1cmVzID0gc2VsZi5kMShzZWxmLnVwKHNlbGYuZDIoc2VsZi51cChzZWxmLmQzKHNlbGYudXAoeiwgYykpLCBiKSksIGEpKQogICAgICAgIHNlZ19sb2dpdHMgPSBzZWxmLnNlZyhmZWF0dXJlcykuZmxvYXQoKQogICAgICAgIG1hc2tzID0gc2VnX2xvZ2l0cy5zaWdtb2lkKCkKICAgICAgICAjIFNlZ21lbnRhdGlvbiBhbG9uZSBsZWFybnMgdGhlIGdhdGUuIENsYXNzaWZpY2F0aW9uIGNhbm5vdCBkaXJlY3RseSByZXNoYXBlIGl0LgogICAgICAgIGdhdGUgPSBtYXNrc1s6LCAxOjJdLmRldGFjaCgpCiAgICAgICAgZ2F0ZSA9IGdhdGUvZ2F0ZS5zdW0oKC0yLCAtMSksIGtlZXBkaW09VHJ1ZSkuY2xhbXBfbWluKDFlLTYpCiAgICAgICAgY29uY2VwdHMgPSBGLnNvZnRwbHVzKHNlbGYuY29uY2VwdHMoZmVhdHVyZXMpLmZsb2F0KCkpCiAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdChpbWFnZS5kZXZpY2UudHlwZSwgZW5hYmxlZD1GYWxzZSk6CiAgICAgICAgICAgIGNvbnRyaWJ1dGlvbiA9IHRvcmNoLmVpbnN1bSgnYmtodyxjay0+YmNodycsIGNvbmNlcHRzLCBzZWxmLmNsYXNzX3dlaWdodHMuZmxvYXQoKSkqZ2F0ZQogICAgICAgIGxvZ2l0cyA9IGNvbnRyaWJ1dGlvbi5zdW0oKC0yLCAtMSkpICsgc2VsZi5jbGFzc19iaWFzLmZsb2F0KCkKICAgICAgICBpZiBndWlkZXMgaXMgTm9uZToKICAgICAgICAgICAgZ3VpZGVzID0gaW1hZ2UubmV3X3RlbnNvcihHVUlERVMpW05vbmVdLmV4cGFuZChsZW4oaW1hZ2UpLCAtMSkKICAgICAgICBpbmRpY2VzID0gKGd1aWRlcyooZmVhdHVyZXMuc2hhcGVbLTJdLTEpKS5yb3VuZCgpLmxvbmcoKS5jbGFtcCgwLCBmZWF0dXJlcy5zaGFwZVstMl0tMSkKICAgICAgICBwZiA9IHNlbGYucG9pbnRfZmVhdHVyZXMoZmVhdHVyZXMpLmZsb2F0KCkKICAgICAgICByb3dzID0gdG9yY2guc3RhY2soW3BmW2ksIDosIGluZGljZXNbaV0sIDpdLnBlcm11dGUoMSwgMCwgMikgZm9yIGkgaW4gcmFuZ2UobGVuKGltYWdlKSldKQogICAgICAgIHJvd3MgPSByb3dzLnJlc2hhcGUobGVuKGltYWdlKSwgMywgMTYsIDE2LCBwZi5zaGFwZVstMV0vLzE2KS5tZWFuKC0xKS5mbGF0dGVuKDIpCiAgICAgICAgcmF3ID0gc2VsZi5wb2ludHMocm93cykKICAgICAgICBsZWZ0ID0gcmF3Wy4uLiwgMF0uc2lnbW9pZCgpKjAuOTkKICAgICAgICByaWdodCA9IGxlZnQgKyAoMS1sZWZ0KSpyYXdbLi4uLCAxXS5zaWdtb2lkKCkKICAgICAgICBwb2ludHMgPSB0b3JjaC5zdGFjaygobGVmdCwgcmlnaHQpLCAtMSkuZmxhdHRlbigxKQogICAgICAgIHJldHVybiBkaWN0KGxvZ2l0cz1sb2dpdHMsIHByb2JhYmlsaXRpZXM9bG9naXRzLnNvZnRtYXgoLTEpLCBzZWdfbG9naXRzPXNlZ19sb2dpdHMsCiAgICAgICAgICAgICAgICAgICAgbWFza3M9bWFza3MsIHBvaW50cz1wb2ludHMsIGNvbmNlcHRzPWNvbmNlcHRzLCBnYXRlPWdhdGUsCiAgICAgICAgICAgICAgICAgICAgY29udHJpYnV0aW9ucz1jb250cmlidXRpb24sIGJpYXM9c2VsZi5jbGFzc19iaWFzLmZsb2F0KCksIGd1aWRlcz1ndWlkZXMpCgoKZGVmIGxvc3NfdGVybXMob3V0LCBiYXRjaCk6CiAgICB5LCB2YWxpZCA9IGJhdGNoWydtYXNrcyddWzosIDoyXSwgMS1iYXRjaFsnbWFza3MnXVs6LCAyOjNdCiAgICAjIEZpeGVkIDR4NCBtZWFuIGFnZ3JlZ2F0aW9uIGF2b2lkcyBub25kZXRlcm1pbmlzdGljIGludGVycG9sYXRpb24gYmFja3dhcmQuCiAgICBuLCBfLCBoLCB3ID0geS5zaGFwZQogICAgeSA9IHkucmVzaGFwZShuLCAyLCBoLy80LCA0LCB3Ly80LCA0KS5tZWFuKCgzLCA1KSkKICAgIHZhbGlkID0gdmFsaWQucmVzaGFwZShuLCAxLCBoLy80LCA0LCB3Ly80LCA0KS5hbWluKCgzLCA1KSkKICAgIGJjZSA9IChGLmJpbmFyeV9jcm9zc19lbnRyb3B5X3dpdGhfbG9naXRzKG91dFsnc2VnX2xvZ2l0cyddLCB5LCByZWR1Y3Rpb249J25vbmUnKSp2YWxpZCkuc3VtKCkvKDIqdmFsaWQuc3VtKCkpLmNsYW1wX21pbigxKQogICAgcCA9IG91dFsnbWFza3MnXQogICAgZGljZSA9ICgoMioocCp5KnZhbGlkKS5zdW0oKC0yLCAtMSkpKzEpLygocCp2YWxpZCkuc3VtKCgtMiwgLTEpKSsoeSp2YWxpZCkuc3VtKCgtMiwgLTEpKSsxKSkubWVhbigpCiAgICBzZWdtZW50YXRpb24gPSBiY2UgKyAxLWRpY2UKICAgIGNsYXNzaWZpY2F0aW9uID0gRi5jcm9zc19lbnRyb3B5KG91dFsnbG9naXRzJ10sIGJhdGNoWydsYWJlbHMnXSwgbGFiZWxfc21vb3RoaW5nPTAuMDUpCiAgICB0YXJnZXQgPSBGLm9uZV9ob3QoYmF0Y2hbJ2xhYmVscyddLCAzKS5mbG9hdCgpCiAgICBwcm9iYWJpbGl0eV9jZGYgPSB0b3JjaC5zdGFjaygob3V0Wydwcm9iYWJpbGl0aWVzJ11bOiwgMF0sIG91dFsncHJvYmFiaWxpdGllcyddWzosIDBdK291dFsncHJvYmFiaWxpdGllcyddWzosIDFdKSwgLTEpCiAgICB0YXJnZXRfY2RmID0gdG9yY2guc3RhY2soKHRhcmdldFs6LCAwXSwgdGFyZ2V0WzosIDBdK3RhcmdldFs6LCAxXSksIC0xKQogICAgb3JkaW5hbCA9IChwcm9iYWJpbGl0eV9jZGYtdGFyZ2V0X2NkZikuc3F1YXJlKCkubWVhbigpCiAgICBnZW9tZXRyeSA9IChGLnNtb290aF9sMV9sb3NzKG91dFsncG9pbnRzJ10sIGJhdGNoWydwb2ludHMnXSwgcmVkdWN0aW9uPSdub25lJywgYmV0YT0uMDIpLm1lYW4oLTEpKmJhdGNoWydwb2ludF93ZWlnaHQnXSkuc3VtKCkvYmF0Y2hbJ3BvaW50X3dlaWdodCddLnN1bSgpLmNsYW1wX21pbigxKQogICAgY29udGFpbm1lbnQgPSAoRi5yZWx1KHBbOiwgMToyXS1wWzosIDoxXSkqdmFsaWQpLnN1bSgpL3ZhbGlkLnN1bSgpLmNsYW1wX21pbigxKQogICAgIyBHZW9tZXRyeSBhbmQgc2VnbWVudGF0aW9uIHNob3VsZCBkZXNjcmliZSB0aGUgc2FtZSBndWlkZS1yb3cgaW50ZXJ2YWwuCiAgICBpZHggPSAob3V0WydndWlkZXMnXSoocC5zaGFwZVstMl0tMSkpLnJvdW5kKCkubG9uZygpCiAgICByb3dzID0gdG9yY2guc3RhY2soW3BbaSwgMSwgaWR4W2ldXSBmb3IgaSBpbiByYW5nZShuKV0pCiAgICByb3dfdmFsaWQgPSB0b3JjaC5zdGFjayhbdmFsaWRbaSwgMCwgaWR4W2ldXSBmb3IgaSBpbiByYW5nZShuKV0pCiAgICByZWZlcmVuY2UgPSB0b3JjaC5zdGFjayhbeVtpLCAxLCBpZHhbaV1dIGZvciBpIGluIHJhbmdlKG4pXSkKICAgIHVzYWJsZSA9ICgocmVmZXJlbmNlWy4uLiwgMF0gPCAuMSkgJiAocmVmZXJlbmNlWy4uLiwgLTFdIDwgLjEpICYgKHJlZmVyZW5jZS5zdW0oLTEpID4gMikpLmZsb2F0KCkKICAgIHggPSB0b3JjaC5saW5zcGFjZSgwLCAxLCBwLnNoYXBlWy0xXSwgZGV2aWNlPXAuZGV2aWNlKVtOb25lLCBOb25lXQogICAgcSA9IG91dFsncG9pbnRzJ10ucmVzaGFwZShuLCAzLCAyKQogICAgaW50ZXJ2YWwgPSAoKHgtcVsuLi4sIDoxXSkvLjAxNSkuc2lnbW9pZCgpKigocVsuLi4sIDE6XS14KS8uMDE1KS5zaWdtb2lkKCkKICAgIGFsaWdubWVudCA9ICgoaW50ZXJ2YWwtcm93cy5kZXRhY2goKSkuYWJzKCkqcm93X3ZhbGlkKnVzYWJsZVsuLi4sIE5vbmVdKS5zdW0oKS8ocm93X3ZhbGlkKnVzYWJsZVsuLi4sIE5vbmVdKS5zdW0oKS5jbGFtcF9taW4oMSkKICAgIHRvdGFsID0gY2xhc3NpZmljYXRpb24gKyAuMipvcmRpbmFsICsgc2VnbWVudGF0aW9uICsgMipnZW9tZXRyeSArIC4xKmNvbnRhaW5tZW50ICsgLjEqYWxpZ25tZW50CiAgICByZXR1cm4gZGljdCh0b3RhbD10b3RhbCwgY2xhc3NpZmljYXRpb249Y2xhc3NpZmljYXRpb24sIG9yZGluYWw9b3JkaW5hbCwgc2VnbWVudGF0aW9uPXNlZ21lbnRhdGlvbiwKICAgICAgICAgICAgICAgIGdlb21ldHJ5PWdlb21ldHJ5LCBjb250YWlubWVudD1jb250YWlubWVudCwgYWxpZ25tZW50PWFsaWdubWVudCkKCgpkZWYgYm94ZXMoYmluYXJ5KToKICAgICIiIk1hc2stZGVyaXZlZCBib3gsIG5vdCBhIHNlcGFyYXRlbHkgdHJhaW5lZCBvYmplY3QgZGV0ZWN0b3IuIiIiCiAgICByZXN1bHQgPSBbXQogICAgZm9yIG1hc2sgaW4gYmluYXJ5OgogICAgICAgIHlzLCB4cyA9IHRvcmNoLndoZXJlKG1hc2spCiAgICAgICAgcmVzdWx0LmFwcGVuZChOb25lIGlmIG5vdCBsZW4oeHMpIGVsc2UgW2ludCh4cy5taW4oKSksIGludCh5cy5taW4oKSksIGludCh4cy5tYXgoKSkrMSwgaW50KHlzLm1heCgpKSsxXSkKICAgIHJldHVybiByZXN1bHQK', 'sha256': 'd16cd4eef75779044dd95900c0531fee474704cf88f523326ee069115347ba09'}, 'geometry.py': {'base64': 'IiIiVmlzaWJsZSAyRCB0cmVhZCBjb29yZGluYXRlcyBmcm9tIGZyb3plbiBwcmVkaWN0ZWQgbWFza3M7IG5vIHRydXRoLW1hc2sgaW5wdXQuIiIiCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLm5uIGltcG9ydCBmdW5jdGlvbmFsIGFzIEYKSEVJR0hUPTUxMjtXSURUSD0yNTY7UEFUQ0g9NjQKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGNhbm9uaWNhbGl6ZShyZ2IsbWFza3MpOgogYXNzZXJ0IHJnYi5uZGltPT00IGFuZCB0dXBsZShyZ2Iuc2hhcGVbMTpdKT09KDMsNTEyLDM4NCkgYW5kIHR1cGxlKG1hc2tzLnNoYXBlWzE6XSk9PSgyLDEyOCw5NikKIHJnYj1yZ2IuZGV0YWNoKCk7YmluYXJ5PShtYXNrcz49LjUpLnJlcGVhdF9pbnRlcmxlYXZlKDQsLTIpLnJlcGVhdF9pbnRlcmxlYXZlKDQsLTEpO3N1cHBvcnQ9YmluYXJ5WzosMF0mYmluYXJ5WzosMV0KIHhzPXRvcmNoLmFyYW5nZSgzODQsZGV2aWNlPXJnYi5kZXZpY2UpW05vbmUsTm9uZV07bGVmdD10b3JjaC53aGVyZShzdXBwb3J0LHhzLDM4NCkuYW1pbigtMSk7cmlnaHQ9dG9yY2gud2hlcmUoc3VwcG9ydCx4cywtMSkuYW1heCgtMSk7cm93X3ZhbGlkPXJpZ2h0LWxlZnQrMT49MTYKIGdyaWRzPVtdO2RpYWdub3N0aWNzPVtdO3NhbXBsZWRfdmFsaWQ9W10KIGZvciBpIGluIHJhbmdlKGxlbihyZ2IpKToKICByb3dzPXRvcmNoLndoZXJlKHJvd192YWxpZFtpXSlbMF0KICBpZiBub3QgbGVuKHJvd3MpOgogICBncmlkcy5hcHBlbmQodG9yY2guemVyb3MoKEhFSUdIVCxXSURUSCwyKSxkZXZpY2U9cmdiLmRldmljZSkpO3NhbXBsZWRfdmFsaWQuYXBwZW5kKHRvcmNoLnplcm9zKEhFSUdIVCxkZXZpY2U9cmdiLmRldmljZSxkdHlwZT10b3JjaC5ib29sKSk7ZGlhZ25vc3RpY3MuYXBwZW5kKGRpY3Qoc3VwcG9ydGVkPUZhbHNlLHNvdXJjZV9maXJzdF9yb3c9Tm9uZSxzb3VyY2VfbGFzdF9yb3c9Tm9uZSxjbGlwcGVkX3Jvd19mcmFjdGlvbj1Ob25lKSk7Y29udGludWUKICB5eT10b3JjaC5saW5zcGFjZShpbnQocm93c1swXSksaW50KHJvd3NbLTFdKSxIRUlHSFQsZGV2aWNlPXJnYi5kZXZpY2UpO2luZGljZXM9eXkucm91bmQoKS5sb25nKCk7b2s9cm93X3ZhbGlkW2ksaW5kaWNlc107bD1sZWZ0W2ksaW5kaWNlc10uZmxvYXQoKTtyPXJpZ2h0W2ksaW5kaWNlc10uZmxvYXQoKTt1PXRvcmNoLmxpbnNwYWNlKDAsMSxXSURUSCxkZXZpY2U9cmdiLmRldmljZSkKICB4eD1sWzosTm9uZV0rdVtOb25lXSooci1sKVs6LE5vbmVdO2dyaWQ9dG9yY2guc3RhY2soKDIqeHgvMzgzLTEsKDIqeXkvNTExLTEpWzosTm9uZV0uZXhwYW5kKC0xLFdJRFRIKSksLTEpO2dyaWRbfm9rXT0wO2dyaWRzLmFwcGVuZChncmlkKTtzYW1wbGVkX3ZhbGlkLmFwcGVuZChvaykKICBjbGlwcGVkPShsZWZ0W2kscm93c109PTApfChyaWdodFtpLHJvd3NdPT0zODMpO2RpYWdub3N0aWNzLmFwcGVuZChkaWN0KHN1cHBvcnRlZD1UcnVlLHNvdXJjZV9maXJzdF9yb3c9aW50KHJvd3NbMF0pLHNvdXJjZV9sYXN0X3Jvdz1pbnQocm93c1stMV0pLGNsaXBwZWRfcm93X2ZyYWN0aW9uPWZsb2F0KGNsaXBwZWQuZmxvYXQoKS5tZWFuKCkpKSkKIGdyaWQ9dG9yY2guc3RhY2soZ3JpZHMpO3ZhbGlkPUYuZ3JpZF9zYW1wbGUoc3VwcG9ydFs6LE5vbmVdLmZsb2F0KCksZ3JpZCxtb2RlPSduZWFyZXN0JyxwYWRkaW5nX21vZGU9J3plcm9zJyxhbGlnbl9jb3JuZXJzPVRydWUpCiB2YWxpZCo9dG9yY2guc3RhY2soc2FtcGxlZF92YWxpZClbOixOb25lLDosTm9uZV0KIGF0bGFzPUYuZ3JpZF9zYW1wbGUocmdiLGdyaWQsbW9kZT0nYmlsaW5lYXInLHBhZGRpbmdfbW9kZT0nemVyb3MnLGFsaWduX2Nvcm5lcnM9VHJ1ZSkqdmFsaWQKIHJldHVybiBkaWN0KHJnYj1hdGxhcyx2YWxpZD12YWxpZCxncmlkPWdyaWQsZGlhZ25vc3RpY3M9ZGlhZ25vc3RpY3MpCgpkZWYgcGF0Y2hlcyhhdGxhcyk6CiBiLGMsaCx3PWF0bGFzLnNoYXBlO2Fzc2VydCAoaCx3KT09KEhFSUdIVCxXSURUSCkKIHJldHVybiBhdGxhcy51bmZvbGQoMixQQVRDSCxQQVRDSCkudW5mb2xkKDMsUEFUQ0gsUEFUQ0gpLnBlcm11dGUoMCwyLDMsMSw0LDUpLnJlc2hhcGUoYiwzMixjLFBBVENILFBBVENIKQoKZGVmIGJhbmRfd2VpZ2h0cyh2YWxpZCk6CiBjb3ZlcmFnZT1wYXRjaGVzKHZhbGlkKS5tZWFuKCgtMywtMiwtMSkpO3Jvd3M9Y292ZXJhZ2UucmVzaGFwZSgtMSw4LDQpO3Jvd19zdW09cm93cy5zdW0oLTEsa2VlcGRpbT1UcnVlKTthY3RpdmU9KHJvd19zdW0+MCkuc3VtKDEsa2VlcGRpbT1UcnVlKS5jbGFtcF9taW4oMSkKIHJldHVybiAocm93cy9yb3dfc3VtLmNsYW1wX21pbigxZS05KS9hY3RpdmUpLnJlc2hhcGUoLTEsMzIpLGNvdmVyYWdlCg==', 'sha256': '04a8151f666c96ab7482c1aa48c2a2ea7e15d34db09893be3a7ad31e507d366a'}, 'lc_features.py': {'base64': 'IiIiRml4ZWQgdHJhaW4tb25seSBsb2NhbC1jb250cmFzdCBkaWFnbm9zdGljcyBhbmQgZGV0ZXJtaW5pc3RpYyBjYXNlIGRpdmVyc2l0eS4iIiINCmltcG9ydCBtYXRoDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCB0b3JjaA0KZnJvbSB0b3JjaC5ubiBpbXBvcnQgZnVuY3Rpb25hbCBhcyBGDQoNCg0KZGVmIGNvbnRyYXN0KHRpbGVzLHZhbGlkLHJ1bGVzKToNCiAgICAiIiJNYXNrZWQgbG9jYWwgbWVhbi9zdGQ7IGZvdXIgZXhwbGljaXRseSBub24tZGVwdGggdGV4dHVyZSBjaGFubmVscy4NCg0KICAgIENoYW5uZWxzOiBjbGlwcGVkIG5vcm1hbGl6ZWQgbHVtaW5hbmNlLCBTb2JlbCB4L3kgb2YgdGhhdCBsdW1pbmFuY2UsDQogICAgZ3JhZGllbnQgbWFnbml0dWRlLiBJbnZhbGlkIHZhbHVlcyBzdGF5IHplcm8uIFRoZSBzdGQgZmxvb3IsIGNsaXBwaW5nIGFuZA0KICAgIGlucHV0IHNhdHVyYXRpb24gbGltaXQgZXhhY3QgbXVsdGlwbGljYXRpdmUtYnJpZ2h0bmVzcyBlcXVpdmFyaWFuY2UuDQogICAgIiIiDQogICAgZ3JheT10aWxlcy5tZWFuKDEsa2VlcGRpbT1UcnVlKQ0KICAgIHdpbmRvdz1ydWxlc1snbG9jYWxfY29udHJhc3Rfd2luZG93J107cGFkPXdpbmRvdy8vMg0KICAgIG1hc3M9Ri5hdmdfcG9vbDJkKHZhbGlkLHdpbmRvdywxLHBhZCkuY2xhbXBfbWluKDFlLTgpDQogICAgbWVhbj1GLmF2Z19wb29sMmQoZ3JheSp2YWxpZCx3aW5kb3csMSxwYWQpL21hc3MNCiAgICB2YXJpYW5jZT0oRi5hdmdfcG9vbDJkKGdyYXkuc3F1YXJlKCkqdmFsaWQsd2luZG93LDEscGFkKS9tYXNzLW1lYW4uc3F1YXJlKCkpLmNsYW1wX21pbigwKQ0KICAgIHN0ZD12YXJpYW5jZS5zcXJ0KCkuY2xhbXBfbWluKHJ1bGVzWydsb2NhbF9jb250cmFzdF9zdGRfZmxvb3InXSkNCiAgICB6PSgoZ3JheS1tZWFuKS9zdGQpLmNsYW1wKC1ydWxlc1snbG9jYWxfY29udHJhc3RfY2xpcCddLHJ1bGVzWydsb2NhbF9jb250cmFzdF9jbGlwJ10pL3J1bGVzWydsb2NhbF9jb250cmFzdF9jbGlwJ10qdmFsaWQNCiAgICBrZXJuZWw9dGlsZXMubmV3X3RlbnNvcihbWy0xLiwwLDFdLFstMiwwLDJdLFstMSwwLDFdXSlbTm9uZSxOb25lXS84DQogICAgZHg9Ri5jb252MmQoeixrZXJuZWwscGFkZGluZz0xKSp2YWxpZA0KICAgIGR5PUYuY29udjJkKHosa2VybmVsLnRyYW5zcG9zZSgtMSwtMikscGFkZGluZz0xKSp2YWxpZA0KICAgIHJldHVybiB0b3JjaC5jYXQoKHosZHgsZHksKGR4LnNxdWFyZSgpK2R5LnNxdWFyZSgpKzFlLTEyKS5zcXJ0KCkqdmFsaWQpLDEpDQoNCg==', 'sha256': 'd417e552b910836d8df2734bb5423f7dcee7956f485b02046739ad1bef5741d6'}, 'legacy_data.py': {'base64': 'IiIiUmV1c2UgdGhlIGZyb3plbiBzYWZlIFBoYXNlIDIgb3ZlcmxheSwgbmV2ZXIgaXRzIHRyYWluZWQgd2VpZ2h0cy4iIiIKaW1wb3J0IG1hdGgKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCmltcG9ydCBwaGFzZTJfdHJhaW5pbmdfZGF0YSBhcyBsZWdhY3kKZnJvbSBjb21tb24gaW1wb3J0IGRpZ2VzdCwgcmVhZCwgc2hhCgpIVyA9ICg1MTIsIDM4NCkKQkFUQ0ggPSA4CkVQT0NIUyA9IDYwClNFRURTID0gKDEsIDIsIDMpCgoKZGVmIHBsYW4ocm9vdCk6CiAgICByZXN1bHQgPSBsZWdhY3kub3ZlcmxheShyb290KQogICAgaHVtYW4gPSB7clsnaW1hZ2VfaWQnXTogciBmb3IgciBpbiByZWFkKFBhdGgocm9vdCkvJ2dlb21ldHJ5L29yaWdpbmFsX2h1bWFuX3BvaW50cy5qc29uJyl9CiAgICB3ZWFrID0ge3JbJ2ltYWdlX2lkJ106IHIgZm9yIHIgaW4gcmVhZChQYXRoKHJvb3QpLydnZW9tZXRyeS9uZXdfcG9pbnRfcHJvcG9zYWxzLmpzb24nKX0KICAgIGZvciByb3cgaW4gcmVzdWx0Wydyb3dzJ106CiAgICAgICAgcmVjb3JkID0gaHVtYW4uZ2V0KHJvd1snaW1hZ2VfaWQnXSkgb3Igd2Vhay5nZXQocm93WydpbWFnZV9pZCddKQogICAgICAgIHJvd1snZ3VpZGVfZnJhY3Rpb25zJ10gPSAoW3JlY29yZFsncG9pbnRzJ11baV1bJ3knXS8ocm93WydoZWlnaHQnXS0xKSBmb3IgaSBpbiAoMCwgMiwgNCldIGlmIHJlY29yZCBlbHNlIFszODQvMTUzNSwgNzY4LzE1MzUsIDExNTEvMTUzNV0pCiAgICBhc3NlcnQgcmVzdWx0Wydjb3VudHMnXSA9PSB7J3RyYWluJzogMzg2LCAndmFsaWRhdGlvbic6IDgxLCAndGVzdCc6IDEwM30KICAgIGFzc2VydCBsZW4oc2V0KHJlc3VsdFsnZ3JvdXBzJ11bJ3RyYWluJ10pICYgc2V0KHJlc3VsdFsnZ3JvdXBzJ11bJ3Rlc3QnXSkpID09IDAKICAgIHJldHVybiByZXN1bHQKCgpkZWYgc2NoZWR1bGUocCwgc2VlZCwgZXBvY2gpOgogICAgcm93cyA9IFtyIGZvciByIGluIHBbJ3Jvd3MnXSBpZiByWydyb2xlJ10gPT0gJ3RyYWluJ10KICAgIHBvb2xzID0ge30KICAgIGZvciBpLCByIGluIGVudW1lcmF0ZShyb3dzKToKICAgICAgICBncm91cCA9IHJbJ3BoeXNpY2FsX3R5cmVfaWQnXSBvciByWydzb3VyY2VfdmlkZW8nXQogICAgICAgIHBvb2xzLnNldGRlZmF1bHQoclsnY2xhc3NfaW5kZXgnXSwge30pLnNldGRlZmF1bHQoclsnZG9tYWluJ10sIHt9KS5zZXRkZWZhdWx0KGdyb3VwLCBbXSkuYXBwZW5kKGkpCiAgICBycyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKjEwMDAwMDMrZXBvY2gpCiAgICByZXN1bHQgPSBbXQogICAgZm9yIF8gaW4gcmFuZ2UobWF0aC5jZWlsKGxlbihyb3dzKS9CQVRDSCkqQkFUQ0gpOgogICAgICAgIGNscyA9IGludChycy5jaG9pY2Uoc29ydGVkKHBvb2xzKSkpCiAgICAgICAgZG9tYWluID0gc3RyKHJzLmNob2ljZShzb3J0ZWQocG9vbHNbY2xzXSkpKQogICAgICAgIGdyb3VwID0gc3RyKHJzLmNob2ljZShzb3J0ZWQocG9vbHNbY2xzXVtkb21haW5dKSkpCiAgICAgICAgcmVzdWx0LmFwcGVuZChpbnQocnMuY2hvaWNlKHBvb2xzW2Nsc11bZG9tYWluXVtncm91cF0pKSkKICAgIHJldHVybiByb3dzLCBbcmVzdWx0W2k6aStCQVRDSF0gZm9yIGkgaW4gcmFuZ2UoMCwgbGVuKHJlc3VsdCksIEJBVENIKV0KCgpkZWYgYmF0Y2gocm9vdCwgcm93cywgZGV2aWNlPSdjcHUnLCBzZWVkPU5vbmUsIGVwb2NoPTAsIGN1cnNvcj0wLCBodz1IVyk6CiAgICBhcnJheXMgPSBbbGVnYWN5LnNhbXBsZShyb290LCByLCBodywgTm9uZSBpZiBzZWVkIGlzIE5vbmUgZWxzZSBzZWVkKjEwMDAwMDE5K2Vwb2NoKjEwMDAwMytjdXJzb3IqOTcraSwgJ3NlZ2Zvcm1lcicpIGZvciBpLCByIGluIGVudW1lcmF0ZShyb3dzKV0KICAgIGltYWdlID0gdG9yY2gudGVuc29yKG5wLnN0YWNrKFt4WzBdIGZvciB4IGluIGFycmF5c10pLCBkZXZpY2U9ZGV2aWNlKQogICAgcmV0dXJuIGRpY3QoaW1nPShpbWFnZS1pbWFnZS5uZXdfdGVuc29yKFsuNDg1LCAuNDU2LCAuNDA2XSlbTm9uZSwgOiwgTm9uZSwgTm9uZV0pL2ltYWdlLm5ld190ZW5zb3IoWy4yMjksIC4yMjQsIC4yMjVdKVtOb25lLCA6LCBOb25lLCBOb25lXSwKICAgICAgICAgICAgICAgIG1hc2tzPXRvcmNoLnRlbnNvcihucC5zdGFjayhbeFsxXSBmb3IgeCBpbiBhcnJheXNdKSwgZGV2aWNlPWRldmljZSwgZHR5cGU9dG9yY2guZmxvYXQzMiksCiAgICAgICAgICAgICAgICBwb2ludHM9dG9yY2gudGVuc29yKG5wLnN0YWNrKFt4WzJdIGZvciB4IGluIGFycmF5c10pLCBkZXZpY2U9ZGV2aWNlKSwKICAgICAgICAgICAgICAgIGxhYmVscz10b3JjaC50ZW5zb3IoW3JbJ2NsYXNzX2luZGV4J10gZm9yIHIgaW4gcm93c10sIGRldmljZT1kZXZpY2UpLAogICAgICAgICAgICAgICAgcG9pbnRfd2VpZ2h0PWltYWdlLm5ld190ZW5zb3IoW3JbJ3BvaW50X3dlaWdodCddIGZvciByIGluIHJvd3NdKSwKICAgICAgICAgICAgICAgIGd1aWRlcz1pbWFnZS5uZXdfdGVuc29yKFtyWydndWlkZV9mcmFjdGlvbnMnXSBmb3IgciBpbiByb3dzXSkpCgoKZGVmIGNvdW50ZXJmYWN0dWFsKGIpOgogICAgIiIiU3dhcCBvbmx5IGxhYmVsbGVkIG91dHNpZGUtdHlyZSBwaXhlbHMgZHVyaW5nIHRyYWluaW5nOyBubyB0ZXN0IGxhYmVscyBlbnRlciBpbmZlcmVuY2UuIiIiCiAgICBmb3JlZ3JvdW5kID0gKGJbJ21hc2tzJ11bOiwgOjFdID4gLjUpIHwgKGJbJ21hc2tzJ11bOiwgMToyXSA+IC41KSB8IChiWydtYXNrcyddWzosIDI6M10gPiAuNSkKICAgIHJldHVybiB0b3JjaC53aGVyZShmb3JlZ3JvdW5kLCBiWydpbWcnXSwgYlsnaW1nJ10ucm9sbCgxLCAwKSkK', 'sha256': 'a15bed0e2574980c2a2339a8a900613cc9292c6342d2ee5e16e1f5a0569955ae'}, 'model_geo.py': {'base64': 'IiIiVFJFQUQtVFdPL0dFTyB0cmFpbmFibGUgbW9kdWxlczsgdGhlIGZyb3plbiBhbmNob3IgaXMgZGVsaWJlcmF0ZWx5IG91dHNpZGUgdGhpcyBtb2R1bGUuIiIiCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KZnJvbSB0b3JjaC5ubiBpbXBvcnQgZnVuY3Rpb25hbCBhcyBGCmZyb20gYXRsYXNfZ2VvIGltcG9ydCBjYW5vbmljYWxpemUsIHBhdGNoZXMKZnJvbSBkYXRhX2dlbyBpbXBvcnQgcGhvdG9tZXRyaWMKZnJvbSBsY19mZWF0dXJlcyBpbXBvcnQgY29udHJhc3QKCgpkZWYgb3JkZXJlZF9wb2ludHMocmF3KToKICAgIGxlZnQgPSAuOTkgKiByYXdbLi4uLCAwXS5zaWdtb2lkKCkKICAgIHJpZ2h0ID0gbGVmdCArICgxIC0gbGVmdCkgKiByYXdbLi4uLCAxXS5zaWdtb2lkKCkKICAgIHJldHVybiB0b3JjaC5zdGFjaygobGVmdCwgcmlnaHQpLCAtMSkuZmxhdHRlbigxKQoKCmNsYXNzIEF4aWFsQmxvY2sobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnNwYXRpYWwgPSBubi5Nb2R1bGVMaXN0KFtubi5Db252MmQoOTYsIDk2LCBrLCBwYWRkaW5nPShrWzBdLy8yLCBrWzFdLy8yKSwgZ3JvdXBzPTk2KQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBrIGluICgoMywgMyksICg1LCAxKSwgKDEsIDMpKV0pCiAgICAgICAgc2VsZi5taXggPSBubi5TZXF1ZW50aWFsKG5uLkdyb3VwTm9ybSg4LCA5NiksIG5uLkNvbnYyZCg5NiwgMTkyLCAxKSwgbm4uU2lMVSgpLCBubi5Db252MmQoMTkyLCA5NiwgMSkpCiAgICAgICAgc2VsZi5zY2FsZSA9IG5uLlBhcmFtZXRlcih0b3JjaC5mdWxsKCgxLCA5NiwgMSwgMSksIC4xKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB0b2tlbnMpOgogICAgICAgIHggPSB0b2tlbnMudHJhbnNwb3NlKDEsIDIpLnJlc2hhcGUoLTEsIDk2LCA4LCA0KQogICAgICAgIHkgPSB4ICsgc2VsZi5zY2FsZSAqIHNlbGYubWl4KHN1bShsYXllcih4KSBmb3IgbGF5ZXIgaW4gc2VsZi5zcGF0aWFsKSkKICAgICAgICByZXR1cm4geS5mbGF0dGVuKDIpLnRyYW5zcG9zZSgxLCAyKQoKCmNsYXNzIEdlbyhubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubWFza19hZGFwdGVyID0gbm4uU2VxdWVudGlhbChubi5Db252MmQoNDgsIDI0LCAzLCBwYWRkaW5nPTEpLCBubi5Hcm91cE5vcm0oOCwgMjQpLCBubi5TaUxVKCksIG5uLkNvbnYyZCgyNCwgMiwgMSkpCiAgICAgICAgc2VsZi5wb2ludF9hZGFwdGVyID0gbm4uU2VxdWVudGlhbChubi5MaW5lYXIoMjU2LCA2NCksIG5uLlNpTFUoKSwgbm4uTGluZWFyKDY0LCAyKSkKICAgICAgICBmb3IgbGF5ZXIgaW4gKHNlbGYubWFza19hZGFwdGVyWy0xXSwgc2VsZi5wb2ludF9hZGFwdGVyWy0xXSk6CiAgICAgICAgICAgIG5uLmluaXQuemVyb3NfKGxheWVyLndlaWdodCkKICAgICAgICAgICAgbm4uaW5pdC56ZXJvc18obGF5ZXIuYmlhcykKICAgICAgICBlbmNvZGVyID0gW10KICAgICAgICBmb3IgY2luLCBjb3V0IGluIHppcCgoNCwgMjQsIDMyLCA2NCksICgyNCwgMzIsIDY0LCA5NikpOgogICAgICAgICAgICBlbmNvZGVyLmV4dGVuZChbbm4uQ29udjJkKGNpbiwgY291dCwgMywgc3RyaWRlPTIsIHBhZGRpbmc9MSksIG5uLkdyb3VwTm9ybSg4LCBjb3V0KSwgbm4uU2lMVSgpXSkKICAgICAgICBzZWxmLmVuY29kZXIgPSBubi5TZXF1ZW50aWFsKCplbmNvZGVyKQogICAgICAgIHNlbGYubWV0YWRhdGEgPSBubi5MaW5lYXIoNCwgOTYpCiAgICAgICAgc2VsZi5jb250ZXh0ID0gbm4uU2VxdWVudGlhbChBeGlhbEJsb2NrKCksIEF4aWFsQmxvY2soKSkKICAgICAgICBzZWxmLm1hc2tfdG9rZW4gPSBubi5QYXJhbWV0ZXIodG9yY2guemVyb3MoOTYpKQogICAgICAgIHNlbGYucHJvamVjdG9yID0gbm4uU2VxdWVudGlhbChubi5MaW5lYXIoOTYsIDEyOCksIG5uLlNpTFUoKSwgbm4uTGluZWFyKDEyOCwgNjQpKQogICAgICAgIHNlbGYuZGVjb2RlciA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKDk2LCAxMjgpLCBubi5TaUxVKCksIG5uLkxpbmVhcigxMjgsIDEwMjQpKQogICAgICAgIHNlbGYucXVhbGl0eSA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKDY0LCAzMiksIG5uLlNpTFUoKSwgbm4uTGluZWFyKDMyLCAzKSwgbm4uU2lnbW9pZCgpKQoKICAgIGRlZiBncm91cHMoc2VsZik6CiAgICAgICAgZ2VvbWV0cnkgPSBbKnNlbGYubWFza19hZGFwdGVyLnBhcmFtZXRlcnMoKSwgKnNlbGYucG9pbnRfYWRhcHRlci5wYXJhbWV0ZXJzKCldCiAgICAgICAgcXVhbGl0eSA9IGxpc3Qoc2VsZi5xdWFsaXR5LnBhcmFtZXRlcnMoKSkKICAgICAgICBleGNsdWRlZCA9IHtpZChwKSBmb3IgcCBpbiBnZW9tZXRyeSArIHF1YWxpdHl9CiAgICAgICAgdGV4dHVyZSA9IFtwIGZvciBwIGluIHNlbGYucGFyYW1ldGVycygpIGlmIGlkKHApIG5vdCBpbiBleGNsdWRlZF0KICAgICAgICByZXR1cm4geydnZW9tZXRyeSc6IGdlb21ldHJ5LCAndGV4dHVyZSc6IHRleHR1cmUsICdxdWFsaXR5JzogcXVhbGl0eX0KCiAgICBkZWYgcmVmaW5lKHNlbGYsIGJhc2UpOgogICAgICAgIGxvZ2l0cyA9IGJhc2VbJ2FuY2hvcl9sb2dpdHMnXSArIHNlbGYubWFza19hZGFwdGVyKGJhc2VbJ2ZlYXR1cmVzJ10pLnRhbmgoKQogICAgICAgIHJhdyA9IGJhc2VbJ3JhdyddICsgLjI1ICogc2VsZi5wb2ludF9hZGFwdGVyKGJhc2VbJ3BvaW50X2ZlYXR1cmVzJ10pLnRhbmgoKQogICAgICAgIHJldHVybiBkaWN0KHNlZ19sb2dpdHM9bG9naXRzLCBtYXNrcz1sb2dpdHMuc2lnbW9pZCgpLCBwb2ludHM9b3JkZXJlZF9wb2ludHMocmF3KSwgcmF3PXJhdykKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgcXVhbGl0eV9pbnB1dChzZWxmLCBiYXNlLCByZWZpbmVkLCBhdGxhcywgZ3VpZGVzKToKICAgICAgICBwID0gcmVmaW5lZFsnbWFza3MnXS5kZXRhY2goKQogICAgICAgIGJpbmFyeSA9IHAgPj0gLjUKICAgICAgICBhcmVhcyA9IHAubWVhbigoLTIsIC0xKSkKICAgICAgICB6ID0gcC5jbGFtcCgxZS02LCAxLTFlLTYpCiAgICAgICAgZW50cm9weSA9IC0oeip6LmxvZygpKygxLXopKigxLXopLmxvZygpKS5tZWFuKCgtMiwgLTEpKQogICAgICAgIGVtcHR5ID0gKH5iaW5hcnkuZmxhdHRlbigyKS5hbnkoLTEpKS5mbG9hdCgpCiAgICAgICAgZWRnZSA9IGJpbmFyeVsuLi4sIDBdIHwgYmluYXJ5Wy4uLiwgLTFdCiAgICAgICAgY2xpcHBlZCA9IGVkZ2UuZmxvYXQoKS5zdW0oLTEpIC8gYmluYXJ5LmFueSgtMSkuZmxvYXQoKS5zdW0oLTEpLmNsYW1wX21pbigxKQogICAgICAgIGluZGV4ID0gKGd1aWRlcyAqIDEyNykucm91bmQoKS5sb25nKCkuY2xhbXAoMCwgMTI3KQogICAgICAgIHJvd3MgPSB0b3JjaC5zdGFjayhbYmluYXJ5W2ksIDAsIGluZGV4W2ldXSAmIGJpbmFyeVtpLCAxLCBpbmRleFtpXV0gZm9yIGkgaW4gcmFuZ2UobGVuKHApKV0pCiAgICAgICAgeHMgPSB0b3JjaC5hcmFuZ2UoOTYsIGRldmljZT1wLmRldmljZSlbTm9uZSwgTm9uZV0KICAgICAgICBhdmFpbGFibGUgPSByb3dzLmFueSgtMSkKICAgICAgICBsZWZ0ID0gdG9yY2gud2hlcmUocm93cywgeHMsIDk2KS5hbWluKC0xKS5mbG9hdCgpLzk1CiAgICAgICAgcmlnaHQgPSB0b3JjaC53aGVyZShyb3dzLCB4cywgLTEpLmFtYXgoLTEpLmZsb2F0KCkvOTUKICAgICAgICBsaW1pdHMgPSB0b3JjaC5zdGFjaygobGVmdCwgcmlnaHQpLCAtMSkKICAgICAgICBnYXBzID0gKGxpbWl0cy1yZWZpbmVkWydwb2ludHMnXS5kZXRhY2goKS5yZXNoYXBlKC0xLCAzLCAyKSkuYWJzKCkubWVhbigtMSkKICAgICAgICBnYXBzID0gdG9yY2gud2hlcmUoYXZhaWxhYmxlLCBnYXBzLCB0b3JjaC56ZXJvc19saWtlKGdhcHMpKQogICAgICAgIGNvdiA9IGF0bGFzWydjb3ZlcmFnZSddCiAgICAgICAgc2NhbGFycyA9IHRvcmNoLmNhdCgoYXJlYXMsIGVudHJvcHksIGVtcHR5LCBjbGlwcGVkLCBnYXBzLCBhdmFpbGFibGUuZmxvYXQoKSwgY292Lm1lYW4oLTEsIGtlZXBkaW09VHJ1ZSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKGNvdi5yZXNoYXBlKC0xLCA4LCA0KS5zdW0oLTEpPjApLmZsb2F0KCkubWVhbigtMSwga2VlcGRpbT1UcnVlKSksIC0xKQogICAgICAgIGFzc2VydCBzY2FsYXJzLnNoYXBlWy0xXSA9PSAxNgogICAgICAgIHJldHVybiB0b3JjaC5jYXQoKGJhc2VbJ2ZlYXR1cmVzJ10ubWVhbigoLTIsIC0xKSksIHNjYWxhcnMpLCAtMSkuZGV0YWNoKCkKCiAgICBkZWYgY29udGV4dHVhbGl6ZShzZWxmLCBlbmNvZGVkLCBtZXRhZGF0YSwgbWFza2VkPU5vbmUpOgogICAgICAgIHRva2VucyA9IGVuY29kZWQgaWYgbWFza2VkIGlzIE5vbmUgZWxzZSB0b3JjaC53aGVyZShtYXNrZWRbLi4uLCBOb25lXSwgc2VsZi5tYXNrX3Rva2VuLCBlbmNvZGVkKQogICAgICAgIHJldHVybiBzZWxmLmNvbnRleHQodG9rZW5zICsgbWV0YWRhdGEpCgogICAgZGVmIHRleHR1cmVfdmlldyhzZWxmLCBhdGxhcywgc2VlZHMpOgogICAgICAgIG4gPSBsZW4oc2VlZHMpCiAgICAgICAgcm5ncyA9IFtucC5yYW5kb20uZGVmYXVsdF9ybmcoaW50KHMpKSBmb3IgcyBpbiBzZWVkc10KICAgICAgICBmYWN0b3JzID0gYXRsYXNbJ3JnYiddLm5ld190ZW5zb3IobnAuc3RhY2soW3JuZy51bmlmb3JtKC44LCAxLjIsIDMpIGZvciBybmcgaW4gcm5nc10pKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICByZ2IgPSBhdGxhc1sncmdiJ10KICAgICAgICAgICAgdmFsaWRfdGlsZXMgPSBwYXRjaGVzKGF0bGFzWyd2YWxpZCddKQogICAgICAgICAgICAjIENvbnRyYXN0IG1lYW5zIGFyZSB0aWxlLWxvY2FsIHRvbzsgYSBnbG9iYWwgYXRsYXMgbWVhbiB3b3VsZCBsZXQKICAgICAgICAgICAgIyBoaWRkZW4gcGl4ZWwgY2hhbmdlcyBwZXJ0dXJiIHZpc2libGUgaW5wdXRzIGJlZm9yZSBsb2NhbCBjb250cmFzdC4KICAgICAgICAgICAgdGlsZXMgPSBwYXRjaGVzKHJnYikKICAgICAgICAgICAgdGlsZXMgPSBwaG90b21ldHJpYyh0aWxlcy5mbGF0dGVuKDAsIDEpLCBmYWN0b3JzWzosIE5vbmVdLmV4cGFuZCgtMSwgMzIsIC0xKS5yZXNoYXBlKC0xLCAzKSkucmVzaGFwZShuLCAzMiwgMywgNjQsIDY0KQogICAgICAgICAgICBsYyA9IGNvbnRyYXN0KHRpbGVzLmZsYXR0ZW4oMCwgMSksIHZhbGlkX3RpbGVzLmZsYXR0ZW4oMCwgMSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgeydsb2NhbF9jb250cmFzdF93aW5kb3cnOiA5LCAnbG9jYWxfY29udHJhc3Rfc3RkX2Zsb29yJzogLjAyLCAnbG9jYWxfY29udHJhc3RfY2xpcCc6IDN9KQogICAgICAgICAgICBsYyA9IGxjLnJlc2hhcGUobiwgMzIsIDQsIDY0LCA2NCkuZGV0YWNoKCkKICAgICAgICAgICAgdGFyZ2V0cyA9IEYuYXZnX3Bvb2wyZChsYy5mbGF0dGVuKDAsIDEpLCA0KS5yZXNoYXBlKG4sIDMyLCA0LCAxNiwgMTYpCiAgICAgICAgICAgIGxvc3NfdmFsaWQgPSBwYXRjaGVzKGF0bGFzWyd2YWxpZCddICogKDEtYXRsYXNbJ2lnbm9yZWQnXSkpCiAgICAgICAgICAgIHdlaWdodCA9IC1GLm1heF9wb29sMmQoLWxvc3NfdmFsaWQuZmxhdHRlbigwLCAxKSwgNCkKICAgICAgICAgICAgd2VpZ2h0ID0gd2VpZ2h0LnJlc2hhcGUobiwgMzIsIDEsIDE2LCAxNikKICAgICAgICAgICAgc3VwcG9ydGVkID0gYXRsYXNbJ2NvdmVyYWdlJ10gPj0gLjI1CiAgICAgICAgICAgIGNob3NlbiA9IHRvcmNoLnplcm9zKChuLCAzMiksIGR0eXBlPXRvcmNoLmJvb2wsIGRldmljZT1yZ2IuZGV2aWNlKQogICAgICAgICAgICBmb3IgaSwgcm5nIGluIGVudW1lcmF0ZShybmdzKToKICAgICAgICAgICAgICAgIGZvciBiYW5kIGluIHJhbmdlKDgpOgogICAgICAgICAgICAgICAgICAgIGVsaWdpYmxlID0gdG9yY2gud2hlcmUoc3VwcG9ydGVkW2ksIGJhbmQqNDpiYW5kKjQrNF0pWzBdLmNwdSgpLnRvbGlzdCgpCiAgICAgICAgICAgICAgICAgICAgaWYgbGVuKGVsaWdpYmxlKSA+PSAyOgogICAgICAgICAgICAgICAgICAgICAgICBjaG9zZW5baSwgYmFuZCo0K2ludChybmcuY2hvaWNlKGVsaWdpYmxlKSldID0gVHJ1ZQogICAgICAgICAgICBlbm91Z2ggPSAoc3VwcG9ydGVkLnN1bSgtMSkgPj0gNCkgJiAoc3VwcG9ydGVkLnJlc2hhcGUobiwgOCwgNCkuYW55KC0xKS5zdW0oLTEpID49IDIpCiAgICAgICAgICAgIGVub3VnaCAmPSAoc3VwcG9ydGVkICYgfmNob3Nlbikuc3VtKC0xKSA+PSAyCiAgICAgICAgICAgIGVub3VnaCAmPSAod2VpZ2h0ICogY2hvc2VuWy4uLiwgTm9uZSwgTm9uZSwgTm9uZV0pLnN1bSgoMSwgMiwgMywgNCkpID4gMAogICAgICAgICAgICB5eSwgeHggPSB0b3JjaC5tZXNoZ3JpZCh0b3JjaC5saW5zcGFjZSgtMSwgMSwgOCwgZGV2aWNlPXJnYi5kZXZpY2UpLCB0b3JjaC5saW5zcGFjZSgtMSwgMSwgNCwgZGV2aWNlPXJnYi5kZXZpY2UpLCBpbmRleGluZz0naWonKQogICAgICAgICAgICBjb29yZGluYXRlcyA9IHRvcmNoLnN0YWNrKCh5eSwgeHgpLCAtMSkucmVzaGFwZSgxLCAzMiwgMikuZXhwYW5kKG4sIC0xLCAtMSkKICAgICAgICAgICAgbWV0YV9pbnB1dCA9IHRvcmNoLmNhdCgoY29vcmRpbmF0ZXMsIGF0bGFzWydjb3ZlcmFnZSddWy4uLiwgTm9uZV0sIGF0bGFzWydtYXNzJ11bLi4uLCBOb25lXSksIC0xKQogICAgICAgIGVuY29kZWQgPSBzZWxmLmVuY29kZXIobGMuZmxhdHRlbigwLCAxKSkubWVhbigoLTIsIC0xKSkucmVzaGFwZShuLCAzMiwgOTYpCiAgICAgICAgbWV0YWRhdGEgPSBzZWxmLm1ldGFkYXRhKG1ldGFfaW5wdXQpCiAgICAgICAgZnVsbCA9IHNlbGYuY29udGV4dHVhbGl6ZShlbmNvZGVkLCBtZXRhZGF0YSkKICAgICAgICBtYXNrZWQgPSBzZWxmLmNvbnRleHR1YWxpemUoZW5jb2RlZCwgbWV0YWRhdGEsIGNob3NlbikKICAgICAgICBlbWJlZGRpbmcgPSAoZnVsbCAqIGF0bGFzWydtYXNzJ11bLi4uLCBOb25lXSkuc3VtKDEpCiAgICAgICAgcHJlZGljdGlvbiA9IHNlbGYuZGVjb2RlcihtYXNrZWQpLnJlc2hhcGUobiwgMzIsIDQsIDE2LCAxNikKICAgICAgICByZXR1cm4gZGljdChlbmNvZGVkPWVuY29kZWQsIGZ1bGw9ZnVsbCwgbWFza2VkX3Rva2Vucz1tYXNrZWQsIGVtYmVkZGluZz1lbWJlZGRpbmcsCiAgICAgICAgICAgICAgICAgICAgcHJvamVjdGVkPXNlbGYucHJvamVjdG9yKGVtYmVkZGluZyksIHByZWRpY3Rpb249cHJlZGljdGlvbiwgdGFyZ2V0PXRhcmdldHMsIHdlaWdodD13ZWlnaHQsCiAgICAgICAgICAgICAgICAgICAgc2VsZWN0ZWQ9Y2hvc2VuLCBlbGlnaWJsZT1lbm91Z2gsIGZhY3RvcnM9ZmFjdG9ycywgbWV0YWRhdGE9bWV0YWRhdGEsIGxjPWxjKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGJhc2UsIGJhdGNoLCBzZWVkcyk6CiAgICAgICAgcmVmaW5lZCA9IHNlbGYucmVmaW5lKGJhc2UpCiAgICAgICAgYXRsYXMgPSBjYW5vbmljYWxpemUoYmF0Y2hbJ3JnYiddLCByZWZpbmVkWydtYXNrcyddLCBiYXRjaFsnbWFza3MnXVs6LCAyOjNdKQogICAgICAgIHEgPSBzZWxmLnF1YWxpdHkoc2VsZi5xdWFsaXR5X2lucHV0KGJhc2UsIHJlZmluZWQsIGF0bGFzLCBiYXRjaFsnZ3VpZGVzJ10pKQogICAgICAgIHZpZXdzID0gW3NlbGYudGV4dHVyZV92aWV3KGF0bGFzLCBbaW50KHMpKjIraSBmb3IgcyBpbiBzZWVkc10pIGZvciBpIGluIHJhbmdlKDIpXQogICAgICAgIHJldHVybiBkaWN0KHJlZmluZWQ9cmVmaW5lZCwgYXRsYXM9YXRsYXMsIHF1YWxpdHk9cSwgdmlld3M9dmlld3MpCg==', 'sha256': '124e0f9998e12425cc6cd6abe62c6d4e97cc10926cabc38935a7d32cef1ea502'}, 'objective_geo.py': {'base64': 'IiIiRXhwbGljaXQgcmVkdWN0aW9ucyBmb3IgZ2VvbWV0cnksIGh1bWFuLW9ubHkgZXJyb3IgdGFyZ2V0cyBhbmQgVklDUmVnIHRleHR1cmUgbG9zcy4iIiIKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2gubm4gaW1wb3J0IGZ1bmN0aW9uYWwgYXMgRgoKCmRlZiBjb2Fyc2VfdHJ1dGgobWFza3MpOgogICAgbiwgXywgaCwgdyA9IG1hc2tzLnNoYXBlCiAgICB0YXJnZXQgPSBtYXNrc1s6LCA6Ml0ucmVzaGFwZShuLCAyLCBoLy80LCA0LCB3Ly80LCA0KS5tZWFuKCgzLCA1KSkKICAgIHZhbGlkID0gKDEtbWFza3NbOiwgMjozXSkucmVzaGFwZShuLCAxLCBoLy80LCA0LCB3Ly80LCA0KS5hbWluKCgzLCA1KSkKICAgIHJldHVybiB0YXJnZXQsIHZhbGlkCgoKZGVmIGZ1bGxfZ2VvbWV0cnkocmVmaW5lZCwgYmF0Y2gpOgogICAgcHJlZCA9IChyZWZpbmVkWydtYXNrcyddLmRldGFjaCgpID49IC41KS5yZXBlYXRfaW50ZXJsZWF2ZSg0LCAtMikucmVwZWF0X2ludGVybGVhdmUoNCwgLTEpCiAgICB0cnV0aCwgdmFsaWQgPSBiYXRjaFsnbWFza3MnXVs6LCA6Ml0gPiAuNSwgYmF0Y2hbJ21hc2tzJ11bOiwgMjozXSA8IC41CiAgICB0cCA9IChwcmVkICYgdHJ1dGggJiB2YWxpZCkuc3VtKCgtMiwgLTEpKQogICAgZnAgPSAocHJlZCAmIH50cnV0aCAmIHZhbGlkKS5zdW0oKC0yLCAtMSkpCiAgICBmbiA9ICh+cHJlZCAmIHRydXRoICYgdmFsaWQpLnN1bSgoLTIsIC0xKSkKICAgIGRlbm9taW5hdG9yID0gMip0cCtmcCtmbgogICAgZGljZSA9IHRvcmNoLndoZXJlKGRlbm9taW5hdG9yID4gMCwgMip0cC5mbG9hdCgpL2Rlbm9taW5hdG9yLmNsYW1wX21pbigxKSwgdG9yY2gub25lc19saWtlKHRwKS5mbG9hdCgpKQogICAgcG9pbnRfZXJyb3IgPSAocmVmaW5lZFsncG9pbnRzJ10uZGV0YWNoKCktYmF0Y2hbJ3BvaW50cyddKS5hYnMoKS5tZWFuKC0xKQogICAgcmV0dXJuIGRpY3QodHA9dHAsIGZwPWZwLCBmbj1mbiwgZGljZT1kaWNlLCBwb2ludF9lcnJvcj1wb2ludF9lcnJvciwKICAgICAgICAgICAgICAgIHRhcmdldF92YWxpZD12YWxpZC5zdW0oKC0yLCAtMSkpLmV4cGFuZCgtMSwgMik+MCkKCgpkZWYgZ2VvbWV0cnlfbG9zcyhyZWZpbmVkLCBiYXNlLCBiYXRjaCk6CiAgICBwLCBsb2dpdHMgPSByZWZpbmVkWydtYXNrcyddLCByZWZpbmVkWydzZWdfbG9naXRzJ10KICAgIHksIHZhbGlkID0gY29hcnNlX3RydXRoKGJhdGNoWydtYXNrcyddKQogICAgYmNlX251bSA9IChGLmJpbmFyeV9jcm9zc19lbnRyb3B5X3dpdGhfbG9naXRzKGxvZ2l0cywgeSwgcmVkdWN0aW9uPSdub25lJykqdmFsaWQpLnN1bSgpCiAgICBiY2VfZGVuID0gMip2YWxpZC5zdW0oKQogICAgYmNlID0gYmNlX251bS9iY2VfZGVuLmNsYW1wX21pbigxKQogICAgZGljZSA9ICgyKihwKnkqdmFsaWQpLnN1bSgoLTIsIC0xKSkrMSkvKChwKnZhbGlkKS5zdW0oKC0yLCAtMSkpKyh5KnZhbGlkKS5zdW0oKC0yLCAtMSkpKzEpCiAgICBzZWdtZW50YXRpb24gPSBiY2UrMS1kaWNlLm1lYW4oKQogICAgcG9pbnRfc3VtID0gKEYuc21vb3RoX2wxX2xvc3MocmVmaW5lZFsncG9pbnRzJ10sIGJhdGNoWydwb2ludHMnXSwgcmVkdWN0aW9uPSdub25lJywgYmV0YT0uMDIpLm1lYW4oLTEpKmJhdGNoWydwb2ludF93ZWlnaHQnXSkuc3VtKCkKICAgIHBvaW50X2RlbiA9IGJhdGNoWydwb2ludF93ZWlnaHQnXS5zdW0oKQogICAgcG9pbnQgPSBwb2ludF9zdW0vcG9pbnRfZGVuLmNsYW1wX21pbigxZS0xMikKICAgIGNvbnRhaW5tZW50X3N1bSA9IChGLnJlbHUocFs6LCAxOjJdLXBbOiwgOjFdKSp2YWxpZCkuc3VtKCkKICAgIGNvbnRhaW5tZW50X2RlbiA9IHZhbGlkLnN1bSgpCiAgICBjb250YWlubWVudCA9IGNvbnRhaW5tZW50X3N1bS9jb250YWlubWVudF9kZW4uY2xhbXBfbWluKDEpCiAgICBpbmRleCA9IChiYXRjaFsnZ3VpZGVzJ10qKHAuc2hhcGVbLTJdLTEpKS5yb3VuZCgpLmxvbmcoKQogICAgcm93cyA9IHRvcmNoLnN0YWNrKFtwW2ksIDEsIGluZGV4W2ldXSBmb3IgaSBpbiByYW5nZShsZW4ocCkpXSkKICAgIHJvd192YWxpZCA9IHRvcmNoLnN0YWNrKFt2YWxpZFtpLCAwLCBpbmRleFtpXV0gZm9yIGkgaW4gcmFuZ2UobGVuKHApKV0pCiAgICByZWZlcmVuY2UgPSB0b3JjaC5zdGFjayhbeVtpLCAxLCBpbmRleFtpXV0gZm9yIGkgaW4gcmFuZ2UobGVuKHApKV0pCiAgICB1c2FibGUgPSAocmVmZXJlbmNlWy4uLiwgMF08LjEpICYgKHJlZmVyZW5jZVsuLi4sIC0xXTwuMSkgJiAocmVmZXJlbmNlLnN1bSgtMSk+MikKICAgIHggPSB0b3JjaC5saW5zcGFjZSgwLCAxLCBwLnNoYXBlWy0xXSwgZGV2aWNlPXAuZGV2aWNlKVtOb25lLCBOb25lXQogICAgcSA9IHJlZmluZWRbJ3BvaW50cyddLnJlc2hhcGUoLTEsIDMsIDIpCiAgICBpbnRlcnZhbCA9ICgoeC1xWy4uLiwgOjFdKS8uMDE1KS5zaWdtb2lkKCkqKChxWy4uLiwgMTpdLXgpLy4wMTUpLnNpZ21vaWQoKQogICAgd2VpZ2h0cyA9IHJvd192YWxpZCp1c2FibGVbLi4uLCBOb25lXQogICAgcm93X3N1bSA9ICgoaW50ZXJ2YWwtcm93cy5kZXRhY2goKSkuYWJzKCkqd2VpZ2h0cykuc3VtKCkKICAgIHJvd19kZW4gPSB3ZWlnaHRzLnN1bSgpCiAgICByb3dfbG9zcyA9IHJvd19zdW0vcm93X2Rlbi5jbGFtcF9taW4oMSkKICAgIGRyaWZ0X21hc2sgPSAoKHAtYmFzZVsnYW5jaG9yX2xvZ2l0cyddLnNpZ21vaWQoKSkuc3F1YXJlKCkqdmFsaWQpLnN1bSgpL2JjZV9kZW4uY2xhbXBfbWluKDEpCiAgICBkcmlmdF9wb2ludHMgPSAocmVmaW5lZFsncG9pbnRzJ10tYmFzZVsnYW5jaG9yX3BvaW50cyddKS5zcXVhcmUoKS5tZWFuKCkKICAgIGRyaWZ0ID0gZHJpZnRfbWFzaytkcmlmdF9wb2ludHMKICAgIHRvdGFsID0gc2VnbWVudGF0aW9uKzIqcG9pbnQrLjEqY29udGFpbm1lbnQrLjEqcm93X2xvc3MrLjEqZHJpZnQKICAgIHJldHVybiB0b3RhbCwgZGljdChiY2VfbnVtZXJhdG9yPWJjZV9udW0sIGJjZV9kZW5vbWluYXRvcj1iY2VfZGVuLCBzb2Z0X2RpY2VfbG9zcz0xLWRpY2UubWVhbigpLAogICAgICAgICAgICAgICAgICAgICAgcG9pbnRfbnVtZXJhdG9yPXBvaW50X3N1bSwgcG9pbnRfZGVub21pbmF0b3I9cG9pbnRfZGVuLAogICAgICAgICAgICAgICAgICAgICAgY29udGFpbm1lbnRfbnVtZXJhdG9yPWNvbnRhaW5tZW50X3N1bSwgY29udGFpbm1lbnRfZGVub21pbmF0b3I9Y29udGFpbm1lbnRfZGVuLAogICAgICAgICAgICAgICAgICAgICAgcm93X251bWVyYXRvcj1yb3dfc3VtLCByb3dfZGVub21pbmF0b3I9cm93X2RlbiwgYW5jaG9yX2RyaWZ0PWRyaWZ0LCBnZW9tZXRyeT10b3RhbCkKCgpkZWYgcXVhbGl0eV9sb3NzKHByZWRpY3Rpb24sIHJlZmluZWQsIGJhdGNoKToKICAgIG1ldHJpY3MgPSBmdWxsX2dlb21ldHJ5KHJlZmluZWQsIGJhdGNoKQogICAgdGFyZ2V0cyA9IHRvcmNoLmNhdCgoMS1tZXRyaWNzWydkaWNlJ10sIG1ldHJpY3NbJ3BvaW50X2Vycm9yJ11bOiwgTm9uZV0pLCAtMSkuZGV0YWNoKCkKICAgIGF2YWlsYWJpbGl0eSA9IHRvcmNoLmNhdCgobWV0cmljc1sndGFyZ2V0X3ZhbGlkJ10sIGJhdGNoWydodW1hbiddWzosIE5vbmVdKSwgLTEpCiAgICBudW1lcmF0b3IgPSAoRi5zbW9vdGhfbDFfbG9zcyhwcmVkaWN0aW9uLCB0YXJnZXRzLCByZWR1Y3Rpb249J25vbmUnLCBiZXRhPS4wMikqYXZhaWxhYmlsaXR5KS5zdW0oKQogICAgZGVub21pbmF0b3IgPSBhdmFpbGFiaWxpdHkuc3VtKCkKICAgIHJldHVybiBudW1lcmF0b3IvZGVub21pbmF0b3IuY2xhbXBfbWluKDEpLCBkaWN0KHRhcmdldHM9dGFyZ2V0cywgYXZhaWxhYmxlPWF2YWlsYWJpbGl0eSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBudW1lcmF0b3I9bnVtZXJhdG9yLCBkZW5vbWluYXRvcj1kZW5vbWluYXRvcikKCgpkZWYgdmFyaWFuY2VfY292YXJpYW5jZSh6KToKICAgIGNlbnRlcmVkID0gei16Lm1lYW4oMCkKICAgIGNvdmFyaWFuY2UgPSBjZW50ZXJlZC5UQGNlbnRlcmVkLyhsZW4oeiktMSkKICAgIHN0ZCA9IChjb3ZhcmlhbmNlLmRpYWcoKSsxZS00KS5zcXJ0KCkKICAgIHZhcmlhbmNlID0gRi5yZWx1KDEtc3RkKS5tZWFuKCkKICAgIG9mZiA9IGNvdmFyaWFuY2UgLSB0b3JjaC5kaWFnX2VtYmVkKGNvdmFyaWFuY2UuZGlhZygpKQogICAgcmV0dXJuIHZhcmlhbmNlLCBvZmYuc3F1YXJlKCkuc3VtKCkvei5zaGFwZVstMV0KCgpkZWYgdGV4dHVyZV9sb3NzKHZpZXdzLCBjb21wb25lbnRzLCBlcG9jaCk6CiAgICBhLCBiID0gdmlld3MKICAgIGVsaWdpYmxlID0gYVsnZWxpZ2libGUnXSAmIGJbJ2VsaWdpYmxlJ10KICAgIGluZGljZXMgPSB0b3JjaC53aGVyZShlbGlnaWJsZSlbMF0uY3B1KCkudG9saXN0KCkKICAgIGNvdW50LCBjb21wb25lbnRfY291bnQgPSBsZW4oaW5kaWNlcyksIGxlbih7Y29tcG9uZW50c1tpXSBmb3IgaSBpbiBpbmRpY2VzfSkKICAgIHZhbGlkID0gY291bnQgPj0gNCBhbmQgY29tcG9uZW50X2NvdW50ID49IDIKICAgIHplcm8gPSBhWydwcm9qZWN0ZWQnXS5zdW0oKSowCiAgICB2YWx1ZXMgPSBkaWN0KGVsaWdpYmxlX2ltYWdlcz1jb3VudCwgZWxpZ2libGVfY29tcG9uZW50cz1jb21wb25lbnRfY291bnQsIHRleHR1cmVfc3RlcD12YWxpZCwKICAgICAgICAgICAgICAgICAgcmFtcD1taW4oMS4sIGVwb2NoLzUpLCByZWFzb249JycgaWYgdmFsaWQgZWxzZSAnZmV3ZXJfdGhhbl80X2ltYWdlc19vcl8yX2NvbXBvbmVudHMnKQogICAgaWYgbm90IHZhbGlkOgogICAgICAgIHZhbHVlcy51cGRhdGUoaW52YXJpYW5jZT16ZXJvLCB2YXJpYW5jZV9hPXplcm8sIHZhcmlhbmNlX2I9emVybywgY292YXJpYW5jZV9hPXplcm8sCiAgICAgICAgICAgICAgICAgICAgICBjb3ZhcmlhbmNlX2I9emVybywgbWFza2VkX251bWVyYXRvcj16ZXJvLCBtYXNrZWRfZGVub21pbmF0b3I9emVybywgbWFza2VkX2h1YmVyPXplcm8sIHRleHR1cmU9emVybykKICAgICAgICByZXR1cm4gemVybywgdmFsdWVzCiAgICB6YSwgemIgPSBhWydwcm9qZWN0ZWQnXVtlbGlnaWJsZV0sIGJbJ3Byb2plY3RlZCddW2VsaWdpYmxlXQogICAgaW52ID0gKHphLXpiKS5zcXVhcmUoKS5tZWFuKCkKICAgIHZhLCBjYSA9IHZhcmlhbmNlX2NvdmFyaWFuY2UoemEpCiAgICB2YiwgY2IgPSB2YXJpYW5jZV9jb3ZhcmlhbmNlKHpiKQogICAgbnVtZXJhdG9ycywgZGVub21pbmF0b3JzID0gW10sIFtdCiAgICBmb3IgdmlldyBpbiB2aWV3czoKICAgICAgICB3ZWlnaHRzID0gdmlld1snd2VpZ2h0J10qdmlld1snc2VsZWN0ZWQnXVsuLi4sIE5vbmUsIE5vbmUsIE5vbmVdKmVsaWdpYmxlWzosIE5vbmUsIE5vbmUsIE5vbmUsIE5vbmVdCiAgICAgICAgbnVtZXJhdG9yID0gKEYuc21vb3RoX2wxX2xvc3Modmlld1sncHJlZGljdGlvbiddLCB2aWV3Wyd0YXJnZXQnXSwgYmV0YT0uMSwgcmVkdWN0aW9uPSdub25lJykqd2VpZ2h0cykuc3VtKCkKICAgICAgICBkZW5vbWluYXRvciA9IHdlaWdodHMuc3VtKCkqNAogICAgICAgIG51bWVyYXRvcnMuYXBwZW5kKG51bWVyYXRvcikKICAgICAgICBkZW5vbWluYXRvcnMuYXBwZW5kKGRlbm9taW5hdG9yKQogICAgbnVtZXJhdG9yLCBkZW5vbWluYXRvciA9IHN1bShudW1lcmF0b3JzKSwgc3VtKGRlbm9taW5hdG9ycykKICAgIGFzc2VydCBmbG9hdChkZW5vbWluYXRvcikgPiAwCiAgICBtYXNrZWQgPSBudW1lcmF0b3IvZGVub21pbmF0b3IKICAgIHRvdGFsID0gMjUqaW52KzI1Kih2YSt2YikrY2ErY2IrLjUqbWFza2VkCiAgICB2YWx1ZXMudXBkYXRlKGludmFyaWFuY2U9aW52LCB2YXJpYW5jZV9hPXZhLCB2YXJpYW5jZV9iPXZiLCBjb3ZhcmlhbmNlX2E9Y2EsIGNvdmFyaWFuY2VfYj1jYiwKICAgICAgICAgICAgICAgICAgbWFza2VkX251bWVyYXRvcj1udW1lcmF0b3IsIG1hc2tlZF9kZW5vbWluYXRvcj1kZW5vbWluYXRvciwgbWFza2VkX2h1YmVyPW1hc2tlZCwgdGV4dHVyZT10b3RhbCkKICAgIHJldHVybiB2YWx1ZXNbJ3JhbXAnXSp0b3RhbCwgdmFsdWVzCgoKZGVmIGxvc3NlcyhvdXQsIGJhc2UsIGJhdGNoLCBzZWVkcywgZXBvY2gpOgogICAgZ2VvLCBncyA9IGdlb21ldHJ5X2xvc3Mob3V0WydyZWZpbmVkJ10sIGJhc2UsIGJhdGNoKQogICAgcXVhbGl0eSwgcXMgPSBxdWFsaXR5X2xvc3Mob3V0WydxdWFsaXR5J10sIG91dFsncmVmaW5lZCddLCBiYXRjaCkKICAgIHRleHR1cmUsIHRzID0gdGV4dHVyZV9sb3NzKG91dFsndmlld3MnXSwgW3JbJ2NvbXBvbmVudCddIGZvciByIGluIGJhdGNoWydyb3dzJ11dLCBlcG9jaCkKICAgIHJldHVybiBnZW8rLjI1KnF1YWxpdHkrdGV4dHVyZSwgeydnZW9tZXRyeSc6IGdzLCAncXVhbGl0eSc6IHFzLCAndGV4dHVyZSc6IHRzfQo=', 'sha256': '82387b35f81af4b97efa7f4573cc453a1b4c1ff5adaa0e3f1d3e4c5ce58a115f'}, 'phase2_training_data.py': {'base64': 'IiIiVmVyc2lvbmVkIHRyYWluaW5nIG92ZXJsYXk7IHNvdXJjZS12MSBpcyBhbHdheXMgcmVhZC1vbmx5LiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aApmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBQSUwgaW1wb3J0IEltYWdlLCBJbWFnZUVuaGFuY2UsIEltYWdlRmlsdGVyCgpSRUxFQVNFX1NIQSA9ICcwNmM2YzlkMDAzYWZmODA4YmFiZjdlYmI1Nzg2ZjQ0MWU5NTA2NGViMDAwZjVlNWVkNWIxNTVkMGVlOWU3YzZhJwpSRVZJU0lPTiA9ICdwaGFzZTItdHJhaW5pbmctcjEnCk1PREVMUyA9IFsnbW9iaWxlbmV0djQnLCAncmVzbmV0NTAnLCAnc2VnZm9ybWVyJywgJ3lvbG8yNm0nLCAnaHJuZXQnXQpTUEVDUyA9IHsKICdtb2JpbGVuZXR2NCc6IGRpY3QobW9kZWw9J21vYmlsZW5ldHY0X2NvbnZfbWVkaXVtLmU1MDBfcjI1Nl9pbjFrJywgcmVwbz0ndGltbS9tb2JpbGVuZXR2NF9jb252X21lZGl1bS5lNTAwX3IyNTZfaW4xaycsIHJldmlzaW9uPSdhZDY2ODk4YzA0NWMxYjUyMjNlYTNmMmMwODMwYjc0Y2YyZTc1YmFjJywgZmlsZT0nbW9kZWwuc2FmZXRlbnNvcnMnLCBodz1bMzg0LDM4NF0sIGJhdGNoPTgpLAogJ3Jlc25ldDUwJzogZGljdChtb2RlbD0ncmVzbmV0NTAuYTFfaW4xaycsIHJlcG89J3RpbW0vcmVzbmV0NTAuYTFfaW4xaycsIHJldmlzaW9uPSc3NjcyNjg2MDNjYTBjYjBiZmUzMjZmYTg3Mjc3ZjE5YzQxOTU2NmVmJywgZmlsZT0nbW9kZWwuc2FmZXRlbnNvcnMnLCBodz1bMzg0LDM4NF0sIGJhdGNoPTgpLAogJ3NlZ2Zvcm1lcic6IGRpY3QobW9kZWw9J252aWRpYS9taXQtYjAnLCByZXBvPSdudmlkaWEvbWl0LWIwJywgcmV2aXNpb249JzgwOTgzYTQxM2MzMGQzNmEzOWMyMDIwMzk3NGFlNzgwNzgzNWUyYjQnLCBmaWxlPSdweXRvcmNoX21vZGVsLmJpbicsIGh3PVs1MTIsMzg0XSwgYmF0Y2g9MiksCiAnaHJuZXQnOiBkaWN0KG1vZGVsPSdocm5ldF93MTgubXNfYXVnX2luMWsnLCByZXBvPSd0aW1tL2hybmV0X3cxOC5tc19hdWdfaW4xaycsIHJldmlzaW9uPSc3ZTJjNTU4Mzc2OWY1NDUxNGZkODdlM2JhOWRlNDA4ZTMzZWFiYTBmJywgZmlsZT0nbW9kZWwuc2FmZXRlbnNvcnMnLCBodz1bNTEyLDM4NF0sIGJhdGNoPTIpLAogJ3lvbG8yNm0nOiBkaWN0KG1vZGVsPSd5b2xvMjZtLXNlZy55YW1sJywgdXJsPSdodHRwczovL2dpdGh1Yi5jb20vdWx0cmFseXRpY3MvYXNzZXRzL3JlbGVhc2VzL2Rvd25sb2FkL3Y4LjQuMC95b2xvMjZtLXNlZy5wdCcsIHNoYTI1Nj0nMTZiNjM2ZjA0ZThmYjZhMzI1YjMzNzBmMjJkYzVlNTUzNWZmNDczZTM4NGY0ZDA0MWZkMjhkNzg4ZjZlZTlmNScsIGh3PVs1MTIsMzg0XSwgYmF0Y2g9MiksCn0KCmRlZiBjYW5vbmljYWwoeCk6IHJldHVybiBqc29uLmR1bXBzKHgsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgnLCcsICc6JykpLmVuY29kZSgpCmRlZiBkaWdlc3QoeCk6IHJldHVybiBoYXNobGliLnNoYTI1NihjYW5vbmljYWwoeCkpLmhleGRpZ2VzdCgpCmRlZiBzaGEocGF0aCk6CiBoPWhhc2hsaWIuc2hhMjU2KCkKIHdpdGggb3BlbihwYXRoLCdyYicpIGFzIGY6CiAgZm9yIGIgaW4gaXRlcihsYW1iZGE6Zi5yZWFkKDEwMjQqMTAyNCksYicnKTpoLnVwZGF0ZShiKQogcmV0dXJuIGguaGV4ZGlnZXN0KCkKZGVmIHJlYWQocGF0aCk6IHJldHVybiBqc29uLmxvYWRzKFBhdGgocGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpKQpkZWYgd3JpdGUocGF0aCxkYXRhKToKIHA9UGF0aChwYXRoKTtwLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKIHRtcD1wLndpdGhfc3VmZml4KHAuc3VmZml4KycudG1wJyk7dG1wLndyaXRlX2J5dGVzKGNhbm9uaWNhbChkYXRhKSk7dG1wLnJlcGxhY2UocCkKCmRlZiBsb2NhdGUocm9vdD0nJyk6CiBpZiByb290OiBwYXRocz1bUGF0aChyb290KV0KIGVsc2U6IHBhdGhzPVtwLnBhcmVudCBmb3IgcCBpbiBQYXRoKCcva2FnZ2xlL2lucHV0Jykucmdsb2IoJ1ZFUlNJT04uanNvbicpIGlmIHJlYWQocCkuZ2V0KCdkYXRhc2V0X3RpdGxlJyk9PSdUaXJlIERhdGFzZXQgUHJlcGFyZWQgcGhhc2UyJ10KIGlmIGxlbihwYXRocykhPTE6cmFpc2UgVmFsdWVFcnJvcignQXR0YWNoIGV4YWN0bHkgb25lIGV4cGFuZGVkIHRpcmUtZGF0YXNldC1wcmVwYXJlZC1waGFzZTIgaW5wdXQsIG9yIHNldCBEQVRBX1JPT1QuJykKIHJvb3Q9cGF0aHNbMF0ucmVzb2x2ZSgpCiBpZiBzaGEocm9vdC8nU0hBMjU2U1VNUy50eHQnKSE9UkVMRUFTRV9TSEE6cmFpc2UgVmFsdWVFcnJvcignV3JvbmcgZGF0YXNldCByZWxlYXNlOyBkbyBub3QgYnlwYXNzIHRoZSBjaGVja3N1bS4nKQogcmV0dXJuIHJvb3QKCmRlZiBvdmVybGF5KHJvb3QpOgogcm93cz1yZWFkKFBhdGgocm9vdCkvJ21hbmlmZXN0cy9pbWFnZXMuanNvbicpCiAjIEFsbCBwb3NzaWJsZSBtYXRjaGVzIG9mIGV2ZXJ5IHVucmVzb2x2ZWQgdmlkZW8gYXJlIHJlc3RyaWN0ZWQgdG8gVFJBSU4uCiAjIEhvbGRvdXRzIGFyZSB3aG9sZSBrbm93biBsb3cvaGlnaCB0eXJlcywgc2VsZWN0ZWQgYnkgY291bnRzLCBuZXZlciBieSBtb2RlbCByZXN1bHRzLgogb2xkPVtyIGZvciByIGluIHJvd3MgaWYgclsnZG9tYWluJ109PSdvcmlnaW5hbF9waG90byddCiBjb3VudHM9Q291bnRlcihyWydwaHlzaWNhbF90eXJlX2lkJ10gZm9yIHIgaW4gb2xkKQogY2xhc3Nlcz17clsncGh5c2ljYWxfdHlyZV9pZCddOnJbJ2NsYXNzX2luZGV4J10gZm9yIHIgaW4gb2xkfQogZ3JvdXBzPWRpY3QodHJhaW49W10sdmFsaWRhdGlvbj1bXSx0ZXN0PVtdKQogZm9yIGNscyBpbiAoMCwyKToKICBjYW5kaWRhdGVzPXNvcnRlZCgoZyBmb3IgZyBpbiBjb3VudHMgaWYgY2xhc3Nlc1tnXT09Y2xzIGFuZCBjb3VudHNbZ10+PTIwKSxrZXk9bGFtYmRhIGc6KC1jb3VudHNbZ10sZykpCiAgYXNzZXJ0IGxlbihjYW5kaWRhdGVzKT49MwogIGdyb3Vwc1sndmFsaWRhdGlvbiddLmFwcGVuZChjYW5kaWRhdGVzWzJdKTtncm91cHNbJ3Rlc3QnXS5hcHBlbmQoY2FuZGlkYXRlc1sxXSkKIGdyb3Vwc1sndHJhaW4nXT1zb3J0ZWQoc2V0KGNvdW50cyktc2V0KGdyb3Vwc1sndmFsaWRhdGlvbiddKS1zZXQoZ3JvdXBzWyd0ZXN0J10pKQogYXNzZXJ0IGFsbChnIGluIGdyb3Vwc1sndHJhaW4nXSBmb3IgZyBpbiBjb3VudHMgaWYgY2xhc3Nlc1tnXT09MSkKIGh1bWFuPXtyWydpbWFnZV9pZCddOnIgZm9yIHIgaW4gcmVhZChQYXRoKHJvb3QpLydnZW9tZXRyeS9vcmlnaW5hbF9odW1hbl9wb2ludHMuanNvbicpfQogcHJvcG9zZWQ9e3JbJ2ltYWdlX2lkJ106ciBmb3IgciBpbiByZWFkKFBhdGgocm9vdCkvJ2dlb21ldHJ5L25ld19wb2ludF9wcm9wb3NhbHMuanNvbicpfQogZm9yIHIgaW4gcm93czoKICByWydyb2xlJ109J3RyYWluJyBpZiByWydkb21haW4nXT09J25ld192aWRlb19mcmFtZScgZWxzZSBuZXh0KGsgZm9yIGssdiBpbiBncm91cHMuaXRlbXMoKSBpZiByWydwaHlzaWNhbF90eXJlX2lkJ10gaW4gdikKICBwPWh1bWFuLmdldChyWydpbWFnZV9pZCddKSBvciBwcm9wb3NlZC5nZXQoclsnaW1hZ2VfaWQnXSkKICByWydwb2ludF93ZWlnaHQnXT0xLiBpZiByWydpbWFnZV9pZCddIGluIGh1bWFuIGVsc2UgLjI1IGlmIHAgZWxzZSAwLgogIHJbJ3BvaW50cyddPVtxWyd4J10vKHJbJ3dpZHRoJ10tMSkgZm9yIHEgaW4gcFsncG9pbnRzJ11dIGlmIHAgZWxzZSBOb25lCiAgclsncG9pbnRfa2luZCddPSdodW1hbicgaWYgclsnaW1hZ2VfaWQnXSBpbiBodW1hbiBlbHNlICdwb2x5Z29uX3dlYWsnIGlmIHAgZWxzZSAnbm9uZScKICBpZiBwOmFzc2VydCBhbGwocVsneCddIGlzIG5vdCBOb25lIGZvciBxIGluIHBbJ3BvaW50cyddKQogYXNzZXJ0IGxlbihyb3dzKT09NTcwIGFuZCBsZW4oe3JbJ2ltYWdlX3NoYTI1NiddIGZvciByIGluIHJvd3N9KT09NTcwCiBhc3NlcnQgbm90IGFueShyWydkb21haW4nXT09J25ld192aWRlb19mcmFtZScgYW5kIHJbJ3JvbGUnXSE9J3RyYWluJyBmb3IgciBpbiByb3dzKQogcmV0dXJuIGRpY3QocmV2aXNpb249UkVWSVNJT04sc291cmNlX2NoZWNrc3VtPVJFTEVBU0VfU0hBLGdyb3Vwcz1ncm91cHMscm93cz1yb3dzLAogIGNvdW50cz1kaWN0KENvdW50ZXIoclsncm9sZSddIGZvciByIGluIHJvd3MpKSwKICBnZW9tZXRyeV9jb3VudHM9ZGljdChDb3VudGVyKHJbJ3JvbGUnXSBmb3IgciBpbiByb3dzIGlmIHJbJ3BvaW50cyddIGlzIG5vdCBOb25lKSksCiAgZGVjaXNpb25zPWRpY3QoaWRlbnRpdHk9J0FsbCB0aHJlZSBvcmlnaW5hbCBtaWQgdHlyZXMgYW5kIGFsbCBuZXcgZnJhbWVzIGFyZSB0cmFpbmluZy1vbmx5OyB1bmtub3duIGV4YWN0IG1hcHBpbmcgaXMgbm90IGludmVudGVkLicsCiAgIG1hc2tzPSdTZWdGb3JtZXIgaWdub3JlcyBjb250cmFkaWN0b3J5IHBpeGVscy4gWU9MTyB0cmFpbmluZyB0eXJlID0gcmF3IHR5cmUgT1IgcmF3IHRyZWFkOyB0cmVhZCB1bmNoYW5nZWQuIERlcml2ZWQgcG9saWN5LCBub3QgaHVtYW4gY29ycmVjdGlvbi4nLAogICBnZW9tZXRyeT0nTmV3IHBvbHlnb24tZGVyaXZlZCBwb2ludHMgYXJlIHdlYWsgdHJhaW5pbmcgdGFyZ2V0cyB3aXRoIHdlaWdodCAwLjI1OyB2YWxpZGF0aW9uL3Rlc3QgdXNlIG9ubHkgZXhpc3RpbmcgaHVtYW4gcG9pbnRzLicsCiAgIGV2YWx1YXRpb249J0hlbGQtb3V0IG9sZCBsb3cvaGlnaCB0eXJlcyBvbmx5LiBObyB1bnNlZW4tbWlkLCB1bnNlZW4tdmlkZW8sIHRocmVlLWNsYXNzLWdlbmVyYWxpc2F0aW9uIG9yIHNhZmV0eSBjbGFpbS4nKSkKCmRlZiB0cmFpbl9yb3dzKHBsYW4sam9iKToKIHJldHVybiBbciBmb3IgciBpbiBwbGFuWydyb3dzJ10gaWYgclsncm9sZSddPT0ndHJhaW4nIGFuZCAoam9iWydjb25kaXRpb24nXT09J2NvbWJpbmVkJyBvciByWydkb21haW4nXT09J29yaWdpbmFsX3Bob3RvJykgYW5kIChqb2JbJ21vZGVsJ10hPSdocm5ldCcgb3IgclsncG9pbnRzJ10gaXMgbm90IE5vbmUpXQoKZGVmIHNjaGVkdWxlKHBsYW4sam9iLGVwb2NoKToKICIiIlN0YXRlbGVzcyBjbGFzcy1iYWxhbmNlZCBzYW1wbGVyLCB0aGVuIGRvbWFpbi1iYWxhbmNlZCB3aXRoaW4gZWFjaCBjbGFzcy4iIiIKIHJvd3M9dHJhaW5fcm93cyhwbGFuLGpvYik7cnM9bnAucmFuZG9tLmRlZmF1bHRfcm5nKGpvYlsnc2VlZCddKjEwMDAwMDMrZXBvY2gpCiBwb29scz17fQogZm9yIGksciBpbiBlbnVtZXJhdGUocm93cyk6CiAgZ3JvdXA9clsncGh5c2ljYWxfdHlyZV9pZCddIG9yIHJbJ3NvdXJjZV92aWRlbyddCiAgcG9vbHMuc2V0ZGVmYXVsdChyWydjbGFzc19pbmRleCddLHt9KS5zZXRkZWZhdWx0KHJbJ2RvbWFpbiddLHt9KS5zZXRkZWZhdWx0KGdyb3VwLFtdKS5hcHBlbmQoaSkKICMgU2FtZSBvcHRpbWl6ZXItdXBkYXRlIGJ1ZGdldCBmb3Igb2xkLW9ubHkgYW5kIGNvbWJpbmVkIHdpdGhpbiBlYWNoIGFyY2hpdGVjdHVyZS4KIHJlZmVyZW5jZT10cmFpbl9yb3dzKHBsYW4sZGljdChqb2IsY29uZGl0aW9uPSdjb21iaW5lZCcpKQogYmF0Y2g9U1BFQ1Nbam9iWydtb2RlbCddXVsnYmF0Y2gnXTtzdGVwcz1tYXRoLmNlaWwobGVuKHJlZmVyZW5jZSkvYmF0Y2gpCiBvcmRlcj1bXQogZm9yIF8gaW4gcmFuZ2Uoc3RlcHMqYmF0Y2gpOgogIGNscz1pbnQocnMuY2hvaWNlKHNvcnRlZChwb29scykpKTtkb21haW5zPXBvb2xzW2Nsc107ZG9tYWluPXN0cihycy5jaG9pY2Uoc29ydGVkKGRvbWFpbnMpKSkKICBiYWdzPWRvbWFpbnNbZG9tYWluXTtncm91cD1zdHIocnMuY2hvaWNlKHNvcnRlZChiYWdzKSkpO29yZGVyLmFwcGVuZChpbnQocnMuY2hvaWNlKGJhZ3NbZ3JvdXBdKSkpCiByZXR1cm4gcm93cyxbb3JkZXJbaTppK2JhdGNoXSBmb3IgaSBpbiByYW5nZSgwLGxlbihvcmRlciksYmF0Y2gpXQoKZGVmIHNhbXBsZShyb290LHJvdyxodyxzZWVkPU5vbmUsbW9kZWw9J3NlZ2Zvcm1lcicpOgogIiIiU3luY2hyb25vdXMgZGV0ZXJtaW5pc3RpYyBDUFUgdHJhbnNmb3Jtczsgbm8gaGlkZGVuIHByZWZldGNoL1JORyBzdGF0ZS4iIiIKIHdpdGggSW1hZ2Uub3BlbihQYXRoKHJvb3QpL3Jvd1snaW1hZ2VfcGF0aCddKSBhcyBpbTppbT1pbS5jb252ZXJ0KCdSR0InKS5yZXNpemUodHVwbGUoaHdbOjotMV0pLEltYWdlLlJlc2FtcGxpbmcuQklMSU5FQVIpCiBtYXNrcz1bXQogZm9yIGtleSBpbiAoJ3R5cmVfbWFzaycsJ3RyZWFkX21hc2snLCdpZ25vcmVfbWFzaycpOgogIHdpdGggSW1hZ2Uub3BlbihQYXRoKHJvb3QpL3Jvd1trZXldKSBhcyBtOm1hc2tzLmFwcGVuZChucC5hc2FycmF5KG0ucmVzaXplKHR1cGxlKGh3Wzo6LTFdKSxJbWFnZS5SZXNhbXBsaW5nLk5FQVJFU1QpKT4wKQogbWFza3M9bnAuc3RhY2sobWFza3MpO3BvaW50cz1ucC5hcnJheShyb3dbJ3BvaW50cyddIG9yIFswLl0qNixkdHlwZT1ucC5mbG9hdDMyKQogaWYgc2VlZCBpcyBub3QgTm9uZToKICBycz1ucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkKICBpZiBycy5yYW5kb20oKTwuNToKICAgaW09aW0udHJhbnNwb3NlKEltYWdlLlRyYW5zcG9zZS5GTElQX0xFRlRfUklHSFQpO21hc2tzPW1hc2tzWzosOiw6Oi0xXS5jb3B5KCk7cG9pbnRzPTEtcG9pbnRzW1sxLDAsMywyLDUsNF1dCiAgaW09SW1hZ2VFbmhhbmNlLkJyaWdodG5lc3MoaW0pLmVuaGFuY2UoZmxvYXQocnMudW5pZm9ybSguOCwxLjIpKSkKICBpbT1JbWFnZUVuaGFuY2UuQ29udHJhc3QoaW0pLmVuaGFuY2UoZmxvYXQocnMudW5pZm9ybSguODUsMS4xNSkpKQogIGlmIHJzLnJhbmRvbSgpPC4yOmltPWltLmZpbHRlcihJbWFnZUZpbHRlci5HYXVzc2lhbkJsdXIoZmxvYXQocnMudW5pZm9ybSguMiwuOCkpKSkKIGlmIG1vZGVsPT0neW9sbzI2bSc6bWFza3NbMF18PW1hc2tzWzFdCiByZXR1cm4gbnAuYXNhcnJheShpbSxkdHlwZT1ucC5mbG9hdDMyKS50cmFuc3Bvc2UoMiwwLDEpLmNvcHkoKS8yNTUuLG1hc2tzLmNvcHkoKSxwb2ludHMKCmRlZiBqb2JzKGNvbmRpdGlvbnM9KCdjb21iaW5lZCcsKSxzZWVkcz0oMSwyLDMpKToKIHJldHVybiBbZGljdChtb2RlbD1tLGNvbmRpdGlvbj1jLHNlZWQ9cyxpZD1mJ3ttfS17Y30tc2VlZHtzfScpIGZvciBtIGluIE1PREVMUyBmb3IgYyBpbiBjb25kaXRpb25zIGZvciBzIGluIHNlZWRzXQo=', 'sha256': '60f9754078949ecac5704b73ef9307b7b95c2cc8acc1a20df2f345d08d7e2579'}, 'PLAN.json': {'base64': 'ew0KICAiYW5jaG9yIjogew0KICAgICJlcG9jaCI6IDIwLA0KICAgICJmcm96ZW4iOiB0cnVlLA0KICAgICJtb2RlbCI6ICJUUkVBRC1PTkUgcjEiLA0KICAgICJwYXJhbWV0ZXJzIjogMTM3MjE1OSwNCiAgICAic2VlZCI6IDMsDQogICAgInNvdXJjZV9zaGEyNTYiOiAiZDE2Y2Q0ZWVmNzU3NzkwNDRkZDk1OTAwYzA1MzFmZWU0NzQ3MDRjZjg4ZjUyMzMyNmVlMDY5MTE1MzQ3YmEwOSIsDQogICAgInVybCI6ICJodHRwczovL2h1Z2dpbmdmYWNlLmNvL2RhdGFzZXRzL1NoYW5tdWs0NjIyL3R5cmUtd2Vhci1zdHVkeS9yZXNvbHZlL2EwMjk2ZjE4ZTAyYjEyZDI4ZjdlNTEzNzc3ODNkNjg1MzgwMzIyMDkvcGhhc2UzL3RyZWFkLW9uZS1yMS8zNzFjNDBjMGM2YWFlZGFmNWM1OTcwZjc3MjFhMjg0YjA0MjQ5NGRiN2JiMzMwMDY1ZjZjYmRlOGZjM2EwYjhjL3JlcG9ydC93ZWlnaHRzLnB0IiwNCiAgICAid2VpZ2h0c19zaGEyNTYiOiAiYTViODcyZWQyZTRjYmZiM2I5NzgyYjBkZjU2M2UyNjA2NjMxNzQ0Y2QwMjc0ZmVlMmRhNDJhODMyZjk4NWY2NSINCiAgfSwNCiAgImFyY2hpdGVjdHVyZSI6IHsNCiAgICAiRU1BX3RlYWNoZXIiOiBmYWxzZSwNCiAgICAiYWN0dWFsX3BhcmFtZXRlcl9jb3VudF9wZW5kaW5nIjogdHJ1ZSwNCiAgICAiYXRsYXNfYWxpZ25fY29ybmVycyI6IHRydWUsDQogICAgImF0bGFzX2h3IjogWw0KICAgICAgNTEyLA0KICAgICAgMjU2DQogICAgXSwNCiAgICAiYXRsYXNfbWluX3NvdXJjZV9yb3dfd2lkdGgiOiAxNiwNCiAgICAiYXhpYWxfYmxvY2tzIjogMiwNCiAgICAiY2xhc3NpZmljYXRpb25fbG9zcyI6IGZhbHNlLA0KICAgICJnZW9tZXRyeV9mZWF0dXJlX3NoYXBlIjogWw0KICAgICAgNDgsDQogICAgICAxMjgsDQogICAgICA5Ng0KICAgIF0sDQogICAgImd1aWRlX21hc2tfZ2FwX2ZsYWciOiAwLjA1LA0KICAgICJndWlkZXMiOiBbDQogICAgICAwLjI1MDE2Mjg2NjQ0OTUxMTQsDQogICAgICAwLjUwMDMyNTczMjg5OTAyMjgsDQogICAgICAwLjc0OTgzNzEzMzU1MDQ4ODYNCiAgICBdLA0KICAgICJpbnB1dF9odyI6IFsNCiAgICAgIDUxMiwNCiAgICAgIDM4NA0KICAgIF0sDQogICAgIm1hc2tfYWRhcHRlcl9jaGFubmVscyI6IFsNCiAgICAgIDQ4LA0KICAgICAgMjQsDQogICAgICAyDQogICAgXSwNCiAgICAibWFza19sb2dpdF9kZWx0YV9ib3VuZCI6IDEuMCwNCiAgICAibWFza190aHJlc2hvbGQiOiAwLjUsDQogICAgIm1hc2tlZF9kZWNvZGVyX2NoYW5uZWxzIjogWw0KICAgICAgOTYsDQogICAgICAxMjgsDQogICAgICAxMDI0DQogICAgXSwNCiAgICAibWFza2VkX3RhcmdldF9zaGFwZSI6IFsNCiAgICAgIDQsDQogICAgICAxNiwNCiAgICAgIDE2DQogICAgXSwNCiAgICAibWFza2VkX3RpbGVfbWluX2NvdmVyYWdlIjogMC4yNSwNCiAgICAibWluaW11bV9TU0xfc3VwcG9ydGVkX2JhbmRzIjogMiwNCiAgICAibWluaW11bV9TU0xfc3VwcG9ydGVkX3RpbGVzIjogNCwNCiAgICAicGF0Y2hfZ3JpZCI6IFsNCiAgICAgIDgsDQogICAgICA0DQogICAgXSwNCiAgICAicGF0Y2hfaHciOiBbDQogICAgICA2NCwNCiAgICAgIDY0DQogICAgXSwNCiAgICAicG9pbnRfYWRhcHRlcl9jaGFubmVscyI6IFsNCiAgICAgIDI1NiwNCiAgICAgIDY0LA0KICAgICAgMg0KICAgIF0sDQogICAgInBvaW50X3Jhd19kZWx0YV9ib3VuZCI6IDAuMjUsDQogICAgInByb2plY3Rvcl9jaGFubmVscyI6IFsNCiAgICAgIDk2LA0KICAgICAgMTI4LA0KICAgICAgNjQNCiAgICBdLA0KICAgICJxdWFsaXR5X2NoYW5uZWxzIjogWw0KICAgICAgNjQsDQogICAgICAzMiwNCiAgICAgIDMNCiAgICBdLA0KICAgICJzaGFyZWRfdHJhaW5hYmxlX2dlb21ldHJ5X3RleHR1cmVfcGFyYW1ldGVycyI6IGZhbHNlLA0KICAgICJ0YXJnZXRfbWF4X3RvdGFsX3BhcmFtZXRlcnMiOiAzMDAwMDAwLA0KICAgICJ0YXJnZXRfbWF4X3RyYWluYWJsZV9wYXJhbWV0ZXJzIjogMTAwMDAwMCwNCiAgICAidGV4dHVyZV9lbmNvZGVyX2NoYW5uZWxzIjogWw0KICAgICAgMjQsDQogICAgICAzMiwNCiAgICAgIDY0LA0KICAgICAgOTYNCiAgICBdLA0KICAgICJ0ZXh0dXJlX2lucHV0X2NoYW5uZWxzIjogNCwNCiAgICAidG9rZW5fZGltZW5zaW9ucyI6IDk2LA0KICAgICJ0cmFuc2Zvcm1lciI6IGZhbHNlDQogIH0sDQogICJkYXRhIjogew0KICAgICJjaGVja3N1bV9maWxlX3NoYTI1NiI6ICIwNmM2YzlkMDAzYWZmODA4YmFiZjdlYmI1Nzg2ZjQ0MWU5NTA2NGViMDAwZjVlNWVkNWIxNTVkMGVlOWU3YzZhIiwNCiAgICAiY29uc2VydmF0aXZlX3RyYWluX2NvbXBvbmVudHMiOiBbDQogICAgICAxLA0KICAgICAgMSwNCiAgICAgIDQNCiAgICBdLA0KICAgICJoaXN0b3JpY2FsX3Rlc3QiOiAxMDMsDQogICAgImltYWdlcyI6IDU3MCwNCiAgICAia2FnZ2xlX2lucHV0IjogInNoYW5tdWs0NjIyL3RpcmUtZGF0YXNldC1wcmVwYXJlZC1waGFzZTIiLA0KICAgICJrbm93bl90cmFpbl90eXJlcyI6IFsNCiAgICAgIDEsDQogICAgICAzLA0KICAgICAgNA0KICAgIF0sDQogICAgIm92ZXJsYXlfc2hhMjU2IjogImY4MWE4MGQyMGI2MGJhNTA1ZDk0ZTMxYzI2NjE1ZTdlNzE3MWE5MWE4NTA2OTQ4YmU2YzM5MmI2ODEwN2EwMzkiLA0KICAgICJ0ZXN0X2h1bWFuX2dlb21ldHJ5X2ltYWdlcyI6IDI1LA0KICAgICJ0cmFpbiI6IDM4NiwNCiAgICAidHJhaW5faHVtYW5fZ2VvbWV0cnlfaW1hZ2VzIjogNzAsDQogICAgInRyYWluX25vX3BvaW50X3RhcmdldF9pbWFnZXMiOiAxNjQsDQogICAgInRyYWluX3dlYWtfZ2VvbWV0cnlfaW1hZ2VzIjogMTUyLA0KICAgICJ2YWxpZGF0aW9uIjogODEsDQogICAgInZhbGlkYXRpb25faHVtYW5fZ2VvbWV0cnlfaW1hZ2VzIjogMjUNCiAgfSwNCiAgImRhdGVfQXNpYV9DYWxjdXR0YSI6ICIyMDI2LTExLTAxIiwNCiAgImZhbWlseSI6ICJUUkVBRC1UV08vR0VPIiwNCiAgImZpdCI6IHsNCiAgICAiYmF0Y2giOiA4LA0KICAgICJiYXRjaGVzX3Blcl9lcG9jaCI6IDQ5LA0KICAgICJiZXRhcyI6IFsNCiAgICAgIDAuOSwNCiAgICAgIDAuOTk5DQogICAgXSwNCiAgICAiY2FuZGlkYXRlX2ZpdHMiOiAxLA0KICAgICJkcmF3cyI6IDIzNTIwLA0KICAgICJlcG9jaHMiOiA2MCwNCiAgICAiZXBzIjogMWUtMDgsDQogICAgImZvcmVhY2giOiBmYWxzZSwNCiAgICAiZnVzZWQiOiBmYWxzZSwNCiAgICAiZ2VvbWV0cnlfbHIiOiAwLjAwMDEsDQogICAgImxyX3NjaGVkdWxlIjogImNvc2luZSBieSBjb21wbGV0ZWQgZ2xvYmFsIHVwZGF0ZSIsDQogICAgIm1pbmltdW1fU1NMX2NvbXBvbmVudHNfcGVyX2JhdGNoIjogMiwNCiAgICAibWluaW11bV9TU0xfaW1hZ2VzX3Blcl9iYXRjaCI6IDQsDQogICAgIm9wdGltaXplciI6ICJBZGFtVyIsDQogICAgInByZWNpc2lvbiI6ICJGUDMyIiwNCiAgICAicXVhbGl0eV9sciI6IDAuMDAwMywNCiAgICAic2NoZWR1bGUiOiAiY29tcGxldGUzODYtaW1hZ2UgY292ZXJhZ2UgcGx1czYgZG9jdW1lbnRlZCBwYWRkaW5nIGRyYXdzIHBlciBlcG9jaDsgYXQgbGVhc3Qgb25lIGxhcmdlc3QtY29tcG9uZW50IGFuZCBvbmUgb3RoZXItY29tcG9uZW50IGltYWdlIHBlciBiYXRjaCIsDQogICAgInNlZWQiOiAxLA0KICAgICJzZXBhcmF0ZV9ncm91cF9jbGlwX25vcm0iOiAxLjAsDQogICAgInRleHR1cmVfbHIiOiAwLjAwMSwNCiAgICAidGV4dHVyZV9yYW1wX2Vwb2NocyI6IDUsDQogICAgInRleHR1cmVfdmlld3MiOiA0NzA0MCwNCiAgICAidXBkYXRlcyI6IDI5NDAsDQogICAgIndlaWdodF9kZWNheSI6IDAuMDAwMQ0KICB9LA0KICAibG9zc2VzIjogew0KICAgICJWSUNfY292YXJpYW5jZV93ZWlnaHQiOiAxLjAsDQogICAgIlZJQ19pbnZhcmlhbmNlX3dlaWdodCI6IDI1LjAsDQogICAgIlZJQ192YXJpYW5jZV9lcHNpbG9uIjogMC4wMDAxLA0KICAgICJWSUNfdmFyaWFuY2VfZmxvb3IiOiAxLjAsDQogICAgIlZJQ192YXJpYW5jZV93ZWlnaHQiOiAyNS4wLA0KICAgICJhbmNob3JfZHJpZnRfd2VpZ2h0IjogMC4xLA0KICAgICJjb250YWlubWVudF93ZWlnaHQiOiAwLjEsDQogICAgIm1hc2tlZF9odWJlcl9iZXRhIjogMC4xLA0KICAgICJtYXNrZWRfcHJlZGljdGlvbl93ZWlnaHQiOiAwLjUsDQogICAgInBvaW50X3Ntb290aF9sMV9iZXRhIjogMC4wMiwNCiAgICAicG9pbnRfd2VpZ2h0IjogMi4wLA0KICAgICJxdWFsaXR5X3Ntb290aF9sMV9iZXRhIjogMC4wMiwNCiAgICAicXVhbGl0eV93ZWlnaHQiOiAwLjI1LA0KICAgICJyb3dfYWdyZWVtZW50X3dlaWdodCI6IDAuMSwNCiAgICAic2VnbWVudGF0aW9uIjogIm1hc2tlZCBCQ0UgKyBzb2Z0IERpY2UsIG1lYW4gb2YyIHJlZ2lvbnMiLA0KICAgICJ3ZWFrX3BvaW50X3dlaWdodCI6IDAuMjUNCiAgfSwNCiAgIm5vdGVib29rcyI6IHsNCiAgICAiTkIxNiI6ICJwbGFubmVkIEdFTyBwcmVwYXJhdGlvbiwgbWF0aC9yZWNvdmVyeS9zdG9yYWdlIGFuZCBhY3R1YWwgZHVhbC1UNCBnYXRlOyBkaXNwb3NhYmxlIGVuZ2luZWVyaW5nIHVwZGF0ZXMgb25seSIsDQogICAgIk5CMTciOiAicGxhbm5lZCBzaW5nbGUgcmVzdW1hYmxlIEdFTyBtZWFzdXJlZCBwaWxvdCIsDQogICAgIk5CMTgiOiAicGxhbm5lZCBjb21wbGV0ZWQtcnVuIHJldmlldy9leHBvcnQgaWYgTkIxNyBjb21wbGV0ZXM7IG5vIGF1dG9tYXRpYyBleHRyYSBtb2RlbCBmaXQiDQogIH0sDQogICJwdWJsaWNhdGlvbiI6IHsNCiAgICAiY2F0Y2hhYmxlX3N0b3BfYXR0ZW1wdF9pbW1lZGlhdGUiOiB0cnVlLA0KICAgICJsb2NhbF9jb21taXRfYXR0ZW1wdHNfcGVyX2hvdXIiOiA5NiwNCiAgICAibWFqb3JfY2VsbF9wdXNoIjogdHJ1ZSwNCiAgICAibmFtZXNwYWNlIjogInBoYXNlMy90cmVhZC10d28vZ2VvLXIxIiwNCiAgICAibmF0aXZlX2Vwb2NoX2NoZWNrcG9pbnRzIjogNjAsDQogICAgIm5vcm1hbF9wdXNoX3NlY29uZHMiOiAxODAwLA0KICAgICJucHoiOiBmYWxzZSwNCiAgICAicHJpdmF0ZV9IRl9TREsiOiAiMC4zNi4yIHVudGlsIHNlcGFyYXRlbHkgdGVzdGVkIHRyYW5zcG9ydCByZXZpc2lvbiIsDQogICAgInJlcG9zaXRvcnkiOiAiU2hhbm11azQ2MjIvdHlyZS13ZWFyLXN0dWR5IiwNCiAgICAic2VjcmV0IjogIkhGX1RPS0VOIiwNCiAgICAic2VydmVyX2hlYWRlcnNfYXV0aG9yaXRhdGl2ZSI6IHRydWUsDQogICAgInppcCI6IGZhbHNlDQogIH0sDQogICJyZXNvdXJjZXMiOiB7DQogICAgIkREUCI6IGZhbHNlLA0KICAgICJhc3N1bWVkX291dHB1dF9ieXRlcyI6IDIwMDAwMDAwMDAwLA0KICAgICJmcmVlX3Jlc2VydmVfYnl0ZXMiOiAyMDAwMDAwMDAwLA0KICAgICJmcm96ZW5fZmVhdHVyZV9hbmRfZXZhbHVhdGlvbl9ncHUiOiAiY3VkYToxIiwNCiAgICAiZ2VvbWV0cnlfbWljcm9iYXRjaCI6IDEsDQogICAgImxpdmVfbWVhc3VyZW1lbnRfcmVxdWlyZWQiOiB0cnVlLA0KICAgICJsb2dpY2FsX2FydGlmYWN0X2J1ZGdldF9ieXRlcyI6IDE1MDAwMDAwMDAwLA0KICAgICJtYXhpbXVtX3Byb2plY3RlZF9wZWFrX2J5dGVzIjogMTgwMDAwMDAwMDAsDQogICAgIm9ic2VydmF0aW9uX25hdGl2ZV9mb3JtYXQiOiAidW5jb21wcmVzc2VkIE5QWSBzaGFyZHMsIGF0IG1vc3Q4Y29tcGxldGUgdXBkYXRlczsgYWN0aXZlIHJhdyBhcHBlbmQgc3RyZWFtcyB3aXRoIGNvbW1pdHRlZCBieXRlIHByZWZpeGVzIGFuZCBpbmRleGVkIHNoYXBlcyIsDQogICAgInNjcmF0Y2hfMVRCX3JlcXVpcmVkIjogZmFsc2UsDQogICAgInN0b3JhZ2VfdW5jZXJ0YWludHlfZmFjdG9yIjogMS4yLA0KICAgICJ0cmFpbmluZ19ncHUiOiAiY3VkYTowIiwNCiAgICAid2FsbF90aW1lX2VzdGltYXRlIjogInBlbmRpbmcgYWN0dWFsIGZpeHR1cmU7IG5vIHNwZWVkIGNsYWltIg0KICB9LA0KICAicmV2aXNpb24iOiAiZ2VvLXIxLXBsYW4tdjEiLA0KICAic2NvcGUiOiB7DQogICAgImFjdHVhbF9LYWdnbGVfZ2F0ZV9wYXNzZWQiOiBmYWxzZSwNCiAgICAiZXhpc3RpbmdfcGhvdG9zX29ubHkiOiB0cnVlLA0KICAgICJtZWFzdXJlZF90cmFpbmluZ19zdGFydGVkIjogZmFsc2UsDQogICAgIm5ld19pbmRlcGVuZGVudF90eXJlcyI6IDAsDQogICAgIm5ld19tb2RlbF9mb3J3YXJkc19wZXJmb3JtZWQiOiAwLA0KICAgICJuZXdfb3B0aW1pemVyX3VwZGF0ZXNfcGVyZm9ybWVkIjogMCwNCiAgICAid29ya3N0YXRpb25fY2hhbmdlcyI6IGZhbHNlDQogIH0sDQogICJzZWxlY3Rpb24iOiB7DQogICAgImh1bWFuX3BvaW50X01BRV9jZWlsaW5nX2RlbHRhIjogMC4wMDEsDQogICAgImludmFsaWRfYXRsYXNfZnJhY3Rpb25fY2VpbGluZ19kZWx0YSI6IDAuMDEsDQogICAgIm1hc2tfZGljZV9mbG9vcl9kZWx0YSI6IC0wLjAwMiwNCiAgICAibWlsZWFnZV9jbGFzc2lmaWNhdGlvbl9zZWxlY3Rpb25fbWV0cmljIjogZmFsc2UsDQogICAgIm5vX2VsaWdpYmxlX2NoZWNrcG9pbnQiOiAicmVqZWN0IGNhbmRpZGF0ZTsgcmV0YWluIGZpbmFsIGFuZCBhbGwgZXZpZGVuY2U7IGRvIG5vdCBzdWJzdGl0dXRlIGhpc3RvcmljYWwtdGVzdCB3aW5uZXIiLA0KICAgICJvYmplY3RpdmUiOiAibWFza2VkX0h1YmVyIC8gdHJhaW5pbmdfbWVhbl9wcmVkaWN0b3JfSHViZXIgKzAuMSpwYWlyZWRfY29zaW5lX2Rpc3RhbmNlOyBsb3dlciBpcyBiZXR0ZXI7IGVhcmxpZXN0IHRpZSIsDQogICAgInByb2plY3Rvcl9lZmZlY3RpdmVfcmFua19mbG9vciI6IDQsDQogICAgInByb2plY3Rvcl9zdGRfbWVkaWFuX2Zsb29yIjogMC4xLA0KICAgICJxdWFsaXR5X2hlYWRfc2VsZWN0aW9uX21ldHJpYyI6IGZhbHNlLA0KICAgICJzdWNjZXNzX21hc2tlZF9sb3NzX3JlbGF0aXZlX2dhaW4iOiAwLjA1LA0KICAgICJzdWNjZXNzX21pbmltdW1fU1NMX2RyYXdfY292ZXJhZ2UiOiAwLjksDQogICAgInN1Y2Nlc3NfcGFpcmVkX2Nvc2luZV9kaXN0YW5jZSI6IDAuMDUNCiAgfSwNCiAgInN0YXR1cyI6ICJkZXNpZ25fY29tcGxldGVfaW1wbGVtZW50YXRpb25fcGVuZGluZyINCn0NCg==', 'sha256': 'a58e7064a26d4472812c03cc45332cba2f0bfa56cee777aaf4b5bda9c9a3740c'}, 'publisher.py': {'base64': 'IiIiQm91bmRlZCBpbW11dGFibGUgZGlyZWN0LWZpbGUgc3RhZ2luZzsgb25lIGZpbmFsIG1hbmlmZXN0IHNlbGVjdHMgYSBzbmFwc2hvdC4iIiINCmltcG9ydCBoYXNobGliLGpzb24sdGltZSx1dWlkDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBDb21taXRPcGVyYXRpb25BZGQNCmZyb20gZXZpZGVuY2VfaHViIGltcG9ydCBIdWIsZGVsYXlfZm9yLHdhaXQNCmZyb20gY29tbW9uIGltcG9ydCBhdG9taWMsY2Fub25pY2FsLGRpZ2VzdCxyZWFkLHNoYQ0KDQpjbGFzcyBQdWJsaXNoZXIoSHViKToNCiBkZWYgb3duZWQoc2VsZik6DQogIHBhcmVudD1zZWxmLmhlYWQoKTtvd25lcj1zZWxmLmdldF9qc29uKCdPV05FUi5qc29uJyxwYXJlbnQpDQogIGFzc2VydCBvd25lciBhbmQgb3duZXJbJ293bmVyJ109PXNlbGYub3duZXIsJ1B1YmxpY2F0aW9uIGZlbmNlZDogYW5vdGhlciB3cml0ZXIgb3ducyB0aGlzIG5hbWVzcGFjZScNCiAgcmV0dXJuIHBhcmVudA0KIGRlZiBjb21taXQoc2VsZixvcGVyYXRpb25zLG1hcmtlcl9uYW1lLG1hcmtlcl92YWx1ZSxtZXNzYWdlLHJlbGVhc2U9RmFsc2UpOg0KICBmb3IgYXR0ZW1wdCBpbiByYW5nZSg4KToNCiAgIHBhcmVudD1zZWxmLm93bmVkKCkNCiAgIGlmIHNlbGYuZ2V0X2pzb24obWFya2VyX25hbWUscGFyZW50KT09bWFya2VyX3ZhbHVlOg0KICAgIHNlbGYucmVjb3JkKCdhbWJpZ3VvdXNfb3JfcmVwbGF5ZWRfY29tbWl0JyxzdGF0dXM9J2Fja25vd2xlZGdlZCcscmV2aXNpb249cGFyZW50LG1hcmtlcj1tYXJrZXJfbmFtZSk7cmV0dXJuIHBhcmVudA0KICAgb3duZXI9Y2Fub25pY2FsKGRpY3Qob3duZXI9Tm9uZSBpZiByZWxlYXNlIGVsc2Ugc2VsZi5vd25lcixleHBpcmVzPTAgaWYgcmVsZWFzZSBlbHNlIHRpbWUudGltZSgpKzI3MDApKQ0KICAgb3BzPW9wZXJhdGlvbnMrW0NvbW1pdE9wZXJhdGlvbkFkZChwYXRoX2luX3JlcG89c2VsZi5kZXN0aW5hdGlvbisnLycrbWFya2VyX25hbWUscGF0aF9vcl9maWxlb2JqPWNhbm9uaWNhbChtYXJrZXJfdmFsdWUpKSxDb21taXRPcGVyYXRpb25BZGQocGF0aF9pbl9yZXBvPXNlbGYuZGVzdGluYXRpb24rJy9PV05FUi5qc29uJyxwYXRoX29yX2ZpbGVvYmo9b3duZXIpXQ0KICAgdHJ5Og0KICAgIHJlc3VsdD1zZWxmLmFwaS5jcmVhdGVfY29tbWl0KHNlbGYuUkVQTyxyZXBvX3R5cGU9J2RhdGFzZXQnLHBhcmVudF9jb21taXQ9cGFyZW50LG9wZXJhdGlvbnM9b3BzLGNvbW1pdF9tZXNzYWdlPW1lc3NhZ2UpDQogICAgc2VsZi5yZWNvcmQoJ2NvbW1pdCcsc3RhdHVzPSdvaycscmV2aXNpb249cmVzdWx0Lm9pZCxmaWxlcz1sZW4ob3BzKSk7cmV0dXJuIHJlc3VsdC5vaWQNCiAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOg0KICAgIGN1cnJlbnQ9c2VsZi5oZWFkKCkNCiAgICBpZiBzZWxmLmdldF9qc29uKG1hcmtlcl9uYW1lLGN1cnJlbnQpPT1tYXJrZXJfdmFsdWU6DQogICAgIHNlbGYucmVjb3JkKCdhbWJpZ3VvdXNfY29tbWl0JyxzdGF0dXM9J2Fja25vd2xlZGdlZCcscmV2aXNpb249Y3VycmVudCk7cmV0dXJuIGN1cnJlbnQNCiAgICBjb2RlPWdldGF0dHIoZ2V0YXR0cihleGMsJ3Jlc3BvbnNlJyxOb25lKSwnc3RhdHVzX2NvZGUnLE5vbmUpO2RlbGF5PWRlbGF5X2ZvcihleGMsYXR0ZW1wdCkNCiAgICBzZWxmLnJlY29yZCgnY29tbWl0JyxzdGF0dXM9J2Vycm9yJyxlcnJvcl90eXBlPXR5cGUoZXhjKS5fX25hbWVfXyxodHRwX3N0YXR1cz1jb2RlLHJldHJ5X2RlbGF5PWRlbGF5KQ0KICAgIGlmIGNvZGUgbm90IGluICg0MDksNDEyKSBhbmQgKGRlbGF5IGlzIE5vbmUgb3IgYXR0ZW1wdD09Nyk6cmFpc2UNCiAgICBpZiBkZWxheSBpcyBub3QgTm9uZTp3YWl0KGRlbGF5KQ0KICByYWlzZSBSdW50aW1lRXJyb3IoJ0JvdW5kZWQgcHVibGljYXRpb24gYXR0ZW1wdHMgZXhoYXVzdGVkOyBuYXRpdmUvbG9jYWwgZXZpZGVuY2UgcmV0YWluZWQnKQ0KIGRlZiBwdWJsaXNoKHNlbGYsc25hcHNob3QscmVsZWFzZT1GYWxzZSk6DQogIHNuYXBzaG90PVBhdGgoc25hcHNob3QpO2xvY2FsPXJlYWQoc25hcHNob3QvJ01BTklGRVNULmpzb24nKTtyZW1vdGVfZmlsZXM9e30NCiAgZm9yIG5hbWUsdmFsdWUgaW4gbG9jYWxbJ2ZpbGVzJ10uaXRlbXMoKToNCiAgIGFzc2VydCBub3QgbmFtZS5lbmRzd2l0aCgoJy56aXAnLCcubnB6JywnLnNxbGl0ZScpKQ0KICAgYXNzZXJ0IHNoYShzbmFwc2hvdC9uYW1lKT09dmFsdWVbJ3NoYTI1NiddIGFuZCAoc25hcHNob3QvbmFtZSkuc3RhdCgpLnN0X3NpemU9PXZhbHVlWydieXRlcyddDQogICBwPVBhdGgobmFtZSk7b2JqZWN0X25hbWU9KHAucGFyZW50LyhwLnN0ZW0rJy0nK3ZhbHVlWydzaGEyNTYnXVs6MjBdK3Auc3VmZml4KSkuYXNfcG9zaXgoKQ0KICAgcmVtb3RlX2ZpbGVzW25hbWVdPWRpY3QodmFsdWUsb2JqZWN0PSdldmlkZW5jZS8nK29iamVjdF9uYW1lKQ0KICBtYW5pZmVzdD1kaWN0KGNvbnRyYWN0PWxvY2FsWydjb250cmFjdCddLGZvcm1hdD0naW1tdXRhYmxlLWRpcmVjdC1maWxlcy12MScsZmlsZXM9cmVtb3RlX2ZpbGVzKQ0KICBwYXJlbnQ9c2VsZi5vd25lZCgpO3ByZXZpb3VzPXNlbGYuZ2V0X2pzb24oJ01BTklGRVNULmpzb24nLHBhcmVudCkNCiAgbmVlZGVkPVtuIGZvciBuLHYgaW4gcmVtb3RlX2ZpbGVzLml0ZW1zKCkgaWYgbm90IHByZXZpb3VzIG9yIHByZXZpb3VzWydmaWxlcyddLmdldChuKSE9dl0NCiAgZm9yIG9mZnNldCBpbiByYW5nZSgwLGxlbihuZWVkZWQpLDgwKToNCiAgIGdyb3VwPW5lZWRlZFtvZmZzZXQ6b2Zmc2V0KzgwXTttYXJrZXI9ZGljdChjb250cmFjdD1sb2NhbFsnY29udHJhY3QnXSxmaWxlcz17bjpyZW1vdGVfZmlsZXNbbl0gZm9yIG4gaW4gZ3JvdXB9KTtrZXk9J0JBVENIRVMvJytkaWdlc3QobWFya2VyKSsnLmpzb24nDQogICBvcGVyYXRpb25zPVtDb21taXRPcGVyYXRpb25BZGQocGF0aF9pbl9yZXBvPXNlbGYuZGVzdGluYXRpb24rJy8nK3JlbW90ZV9maWxlc1tuXVsnb2JqZWN0J10scGF0aF9vcl9maWxlb2JqPXN0cihzbmFwc2hvdC9uKSkgZm9yIG4gaW4gZ3JvdXBdDQogICBzZWxmLmNvbW1pdChvcGVyYXRpb25zLGtleSxtYXJrZXIsJ1RSRUFELVRXTyBBVExBUyBpbW11dGFibGUgZGlyZWN0IENTVi9uYXRpdmUgb2JqZWN0cycpDQogIHN0YWdlZD1zZWxmLm93bmVkKCkNCiAgZW50cmllcz1zZWxmLmNhbGwobGFtYmRhOmxpc3Qoc2VsZi5hcGkubGlzdF9yZXBvX3RyZWUoc2VsZi5SRVBPLHBhdGhfaW5fcmVwbz1zZWxmLmRlc3RpbmF0aW9uLHJldmlzaW9uPXN0YWdlZCxyZWN1cnNpdmU9VHJ1ZSxyZXBvX3R5cGU9J2RhdGFzZXQnKSksJ3ZlcmlmeV9pbW11dGFibGVfdHJlZScpDQogIGxvb2t1cD17di5wYXRoW2xlbihzZWxmLmRlc3RpbmF0aW9uKSsxOl06diBmb3IgdiBpbiBlbnRyaWVzIGlmIGhhc2F0dHIodiwnc2l6ZScpfQ0KICBmb3IgbmFtZSx2YWx1ZSBpbiByZW1vdGVfZmlsZXMuaXRlbXMoKToNCiAgIG9iaj1sb29rdXBbdmFsdWVbJ29iamVjdCddXTthc3NlcnQgb2JqLnNpemU9PXZhbHVlWydieXRlcyddDQogICBpZiBvYmoubGZzOmFzc2VydCBvYmoubGZzLnNoYTI1Nj09dmFsdWVbJ3NoYTI1NiddLG5hbWUNCiAgIGVsc2U6DQogICAgYm9keT0oc25hcHNob3QvbmFtZSkucmVhZF9ieXRlcygpO2Fzc2VydCBvYmouYmxvYl9pZD09aGFzaGxpYi5zaGExKGInYmxvYiAnK3N0cihsZW4oYm9keSkpLmVuY29kZSgpK2InXDAnK2JvZHkpLmhleGRpZ2VzdCgpLG5hbWUNCiAgIyBPbmx5IHZlcmlmaWVkIGltbXV0YWJsZSBvYmplY3RzIGNhbiBiZSBzZWxlY3RlZCBieSB0aGUgZmluYWwgbWFuaWZlc3QuDQogIG1hcmtlcj1kaWN0KGNvbnRyYWN0PWxvY2FsWydjb250cmFjdCddLG1hbmlmZXN0X3NoYTI1Nj1kaWdlc3QobWFuaWZlc3QpLG93bmVyX3JlbGVhc2VkPXJlbGVhc2Usd3JpdGVyPXNlbGYub3duZXIscHVibGljYXRpb25fbm9uY2U9dXVpZC51dWlkNCgpLmhleCkNCiAga2V5PSdQVUJMSUNBVElPTlMvJytkaWdlc3QobWFya2VyKSsnLmpzb24nDQogIG9wZXJhdGlvbnM9W0NvbW1pdE9wZXJhdGlvbkFkZChwYXRoX2luX3JlcG89c2VsZi5kZXN0aW5hdGlvbisnL01BTklGRVNULmpzb24nLHBhdGhfb3JfZmlsZW9iaj1jYW5vbmljYWwobWFuaWZlc3QpKV0NCiAgcmV2aXNpb249c2VsZi5jb21taXQob3BlcmF0aW9ucyxrZXksbWFya2VyLCdUUkVBRC1UV08gQVRMQVMgc2VsZWN0IHZlcmlmaWVkIHJlY292ZXJ5IHNuYXBzaG90JyxyZWxlYXNlPXJlbGVhc2UpDQogIGFzc2VydCBzZWxmLmdldF9qc29uKCdNQU5JRkVTVC5qc29uJyxyZXZpc2lvbik9PW1hbmlmZXN0DQogIGFjdHVhbF9vd25lcj1zZWxmLmdldF9qc29uKCdPV05FUi5qc29uJyxyZXZpc2lvbik7YXNzZXJ0IGFjdHVhbF9vd25lclsnb3duZXInXT09KE5vbmUgaWYgcmVsZWFzZSBlbHNlIHNlbGYub3duZXIpDQogIHNlbGYubGFzdF92ZXJpZmllZD1yZXZpc2lvbg0KICByZXN1bHQ9ZGljdChzdGF0dXM9J3B1Ymxpc2hlZF9hbmRfdmVyaWZpZWQnLHJldmlzaW9uPXJldmlzaW9uLGRlc3RpbmF0aW9uPXNlbGYuZGVzdGluYXRpb24sZmlsZXM9bGVuKHJlbW90ZV9maWxlcykrMSx1cGxvYWRlZF9vYmplY3RzPWxlbihuZWVkZWQpLHppcF91cGxvYWRzPTAsb3duZXJfcmVsZWFzZWQ9cmVsZWFzZSkNCiAgc2VsZi5yZWNvcmQoJ3B1Ymxpc2gnLCoqcmVzdWx0KTtyZXR1cm4gcmVzdWx0DQogZGVmIGRvd25sb2FkKHNlbGYscmV2aXNpb24sZm9sZGVyLHByb3RvY29sKToNCiAgbWFuaWZlc3Q9c2VsZi5nZXRfanNvbignTUFOSUZFU1QuanNvbicscmV2aXNpb24pO2Fzc2VydCBtYW5pZmVzdCBhbmQgbWFuaWZlc3RbJ2NvbnRyYWN0J109PXByb3RvY29sIGFuZCBtYW5pZmVzdFsnZm9ybWF0J109PSdpbW11dGFibGUtZGlyZWN0LWZpbGVzLXYxJw0KICBmb2xkZXI9UGF0aChmb2xkZXIpO2ZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSk7bG9jYWw9e30NCiAgZm9yIG5hbWUsdmFsdWUgaW4gbWFuaWZlc3RbJ2ZpbGVzJ10uaXRlbXMoKToNCiAgIGlmIGhhc2F0dHIoc2VsZiwndHJhbnNmZXJfdGljaycpOnNlbGYudHJhbnNmZXJfdGljaygpDQogICBhc3NlcnQgbm90IFBhdGgobmFtZSkuaXNfYWJzb2x1dGUoKSBhbmQgJy4uJyBub3QgaW4gUGF0aChuYW1lKS5wYXJ0cyBhbmQgJy4uJyBub3QgaW4gUGF0aCh2YWx1ZVsnb2JqZWN0J10pLnBhcnRzDQogICB0YXJnZXQ9Zm9sZGVyL25hbWU7dGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkNCiAgIGlmIG5vdCB0YXJnZXQuZXhpc3RzKCkgb3Igc2hhKHRhcmdldCkhPXZhbHVlWydzaGEyNTYnXToNCiAgICBkZWYgdHJhbnNmZXIoKToNCiAgICAgd2l0aCBzZWxmLnNlc3Npb24uZ2V0KGYnaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9kYXRhc2V0cy97c2VsZi5SRVBPfS9yZXNvbHZlL3tyZXZpc2lvbn0ve3NlbGYuZGVzdGluYXRpb259L3t2YWx1ZVsib2JqZWN0Il19Jyx0aW1lb3V0PSgyMCwxNTApLHN0cmVhbT1UcnVlKSBhcyByZXNwb25zZToNCiAgICAgIHJlc3BvbnNlLnJhaXNlX2Zvcl9zdGF0dXMoKQ0KICAgICAgd2l0aCBvcGVuKHRhcmdldC53aXRoX25hbWUodGFyZ2V0Lm5hbWUrJy5wYXJ0JyksJ3diJykgYXMgZjoNCiAgICAgICBmb3IgYmxvY2sgaW4gcmVzcG9uc2UuaXRlcl9jb250ZW50KDQqMTAyNCoqMik6Zi53cml0ZShibG9jaykNCiAgICAgdGFyZ2V0LndpdGhfbmFtZSh0YXJnZXQubmFtZSsnLnBhcnQnKS5yZXBsYWNlKHRhcmdldCkNCiAgICBzZWxmLmNhbGwodHJhbnNmZXIsJ3Jlc29sdmVfaW1tdXRhYmxlOicrbmFtZSkNCiAgIGFzc2VydCB0YXJnZXQuc3RhdCgpLnN0X3NpemU9PXZhbHVlWydieXRlcyddIGFuZCBzaGEodGFyZ2V0KT09dmFsdWVbJ3NoYTI1NiddLG5hbWUNCiAgIGxvY2FsW25hbWVdPXtrOnYgZm9yIGssdiBpbiB2YWx1ZS5pdGVtcygpIGlmIGshPSdvYmplY3QnfQ0KICBhdG9taWMoZm9sZGVyLydSRU1PVEVfTUFOSUZFU1QuanNvbicsbWFuaWZlc3QpO2F0b21pYyhmb2xkZXIvJ01BTklGRVNULmpzb24nLGRpY3QoY29udHJhY3Q9cHJvdG9jb2wsZmlsZXM9bG9jYWwpKTtyZXR1cm4gZm9sZGVyDQo=', 'sha256': '0ea77d5584a3903b01fdf7466daa95eb7be1021b4e70ac76a6706531a3d56364'}, 'rate_budget.py': {'base64': 'IiIiU2hhcmVkIGxvY2FsIEhUVFAgYnVkZ2V0cyBwbHVzIHNlcnZlciBiYWNrb2ZmLCBpbmNsdWRpbmcgU0RLIHRyYW5zcG9ydCBjYWxscy4NCg0KQ3Jvc3MtaG9zdCBhY2NvdW50IHRyYWZmaWMgY2Fubm90IGJlIGF0b21pY2FsbHkgY291bnRlZCBieSB0aGlzIHNlc3Npb247IHNlcnZlcg0KaGVhZGVycyByZW1haW4gYXV0aG9yaXRhdGl2ZS4gT25seSBzYWZlIHJlcXVlc3QgY2F0ZWdvcmllcy9zdGF0dXMgYXJlIGxvZ2dlZC4NCiIiIg0KaW1wb3J0IHJlDQppbXBvcnQgbWF0aA0KaW1wb3J0IHRpbWUNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSB1cmxsaWIucGFyc2UgaW1wb3J0IHVybHBhcnNlDQoNCmltcG9ydCByZXF1ZXN0cw0KZnJvbSBmaWxlbG9jayBpbXBvcnQgRmlsZUxvY2sNCmZyb20gY29tbW9uIGltcG9ydCBhdG9taWMsIGV2ZW50LCByZWFkDQoNCg0KUE9MSUNZPXsnYXBpJzooMzAwLDk2KSwncmVzb2x2ZXInOigzMDAsNDAwKSwndHJhbnNmZXInOigzMDAsMTAwMCksJ3JlcG9fY29tbWl0JzooMzYwMCw5Nil9DQoNCmNsYXNzIEJ1ZGdldDoNCiAgICBkZWYgX19pbml0X18oc2VsZiwgcm9vdCk6DQogICAgICAgIHNlbGYucm9vdCA9IFBhdGgocm9vdCkNCiAgICAgICAgc2VsZi5yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICAgICAgc2VsZi5wYXRoID0gc2VsZi5yb290IC8gJ0hGX0hUVFBfQlVER0VULmpzb24nDQogICAgICAgIHNlbGYubG9jayA9IEZpbGVMb2NrKHN0cihzZWxmLnBhdGgpICsgJy5sb2NrJywgdGltZW91dD02MCkNCg0KICAgIEBzdGF0aWNtZXRob2QNCiAgICBkZWYgY2F0ZWdvcnkodXJsKToNCiAgICAgICAgcGFyc2VkID0gdXJscGFyc2UodXJsKQ0KICAgICAgICBpZiAnL3Jlc29sdmUvJyBpbiBwYXJzZWQucGF0aDoNCiAgICAgICAgICAgIHJldHVybiAncmVzb2x2ZXInDQogICAgICAgIHJldHVybiAnYXBpJyBpZiBwYXJzZWQuaG9zdG5hbWUgPT0gJ2h1Z2dpbmdmYWNlLmNvJyBlbHNlICd0cmFuc2ZlcicNCg0KICAgIGRlZiByZXNlcnZlKHNlbGYsIGJ1Y2tldCk6DQogICAgICAgIGFzc2VydCBidWNrZXQgaW4gUE9MSUNZDQogICAgICAgIGRlZmF1bHRfd2luZG93LCBkZWZhdWx0X2xpbWl0ID0gUE9MSUNZW2J1Y2tldF0NCiAgICAgICAgbmV4dF9ub3RpY2UgPSAwLg0KICAgICAgICB3aGlsZSBUcnVlOg0KICAgICAgICAgICAgaW5zdGFudCA9IHRpbWUudGltZSgpDQogICAgICAgICAgICB3aXRoIHNlbGYubG9jazoNCiAgICAgICAgICAgICAgICBkYXRhID0gcmVhZChzZWxmLnBhdGgpIGlmIHNlbGYucGF0aC5leGlzdHMoKSBlbHNlIHt9DQogICAgICAgICAgICAgICAgc3RhdGUgPSBkYXRhLnNldGRlZmF1bHQoYnVja2V0LCBkaWN0KHRpbWVzPVtdLCBibG9ja2VkX3VudGlsPTApKQ0KICAgICAgICAgICAgICAgIG9ic2VydmVkID0gc3RhdGUuZ2V0KCdzZXJ2ZXJfcG9saWN5Jywge30pDQogICAgICAgICAgICAgICAgd2luZG93ID0gbWF4KGRlZmF1bHRfd2luZG93LCBpbnQob2JzZXJ2ZWQuZ2V0KCd3aW5kb3dfc2Vjb25kcycsIGRlZmF1bHRfd2luZG93KSkpDQogICAgICAgICAgICAgICAgbGltaXQgPSBtaW4oZGVmYXVsdF9saW1pdCwgbWF4KDEsIG1hdGguZmxvb3IoLjggKiBpbnQob2JzZXJ2ZWQuZ2V0KCdxdW90YScsIGRlZmF1bHRfbGltaXQgKiAyKSkpKSkNCiAgICAgICAgICAgICAgICBzdGF0ZVsndGltZXMnXSA9IFtzdGFtcCBmb3Igc3RhbXAgaW4gc3RhdGVbJ3RpbWVzJ10gaWYgc3RhbXAgPiBpbnN0YW50IC0gd2luZG93XQ0KICAgICAgICAgICAgICAgIHJlbWFpbmluZyA9IHN0YXRlLmdldCgnc2VydmVyX3JlbWFpbmluZycpDQogICAgICAgICAgICAgICAgcmVzZXRfYXQgPSBzdGF0ZS5nZXQoJ3NlcnZlcl9yZXNldF9hdCcsIDAuKQ0KICAgICAgICAgICAgICAgIGlmIHJlc2V0X2F0IDw9IGluc3RhbnQ6DQogICAgICAgICAgICAgICAgICAgIHJlbWFpbmluZyA9IE5vbmUNCiAgICAgICAgICAgICAgICAgICAgc3RhdGUucG9wKCdzZXJ2ZXJfcmVtYWluaW5nJywgTm9uZSkNCiAgICAgICAgICAgICAgICBsb2NhbF9kZWxheSA9IG1heCgwLiwgc3RhdGVbJ3RpbWVzJ11bLWxpbWl0XSArIHdpbmRvdyAtIGluc3RhbnQpIGlmIGxlbihzdGF0ZVsndGltZXMnXSkgPj0gbGltaXQgZWxzZSAwLg0KICAgICAgICAgICAgICAgIHNlcnZlcl9kZWxheSA9IG1heCgwLiwgc3RhdGUuZ2V0KCdibG9ja2VkX3VudGlsJywgMC4pIC0gaW5zdGFudCwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVzZXRfYXQgLSBpbnN0YW50IGlmIHJlbWFpbmluZyBpcyBub3QgTm9uZSBhbmQgcmVtYWluaW5nIDw9IDAgZWxzZSAwLikNCiAgICAgICAgICAgICAgICBkZWxheSA9IG1heChsb2NhbF9kZWxheSwgc2VydmVyX2RlbGF5KQ0KICAgICAgICAgICAgICAgIHN0YXRlWydlZmZlY3RpdmVfd2luZG93X3NlY29uZHMnXSwgc3RhdGVbJ2VmZmVjdGl2ZV9saW1pdCddID0gd2luZG93LCBsaW1pdA0KICAgICAgICAgICAgICAgIGlmIGRlbGF5IDw9IDA6DQogICAgICAgICAgICAgICAgICAgIHN0YXRlWyd0aW1lcyddLmFwcGVuZChpbnN0YW50KQ0KICAgICAgICAgICAgICAgICAgICBpZiByZW1haW5pbmcgaXMgbm90IE5vbmU6DQogICAgICAgICAgICAgICAgICAgICAgICBzdGF0ZVsnc2VydmVyX3JlbWFpbmluZyddID0gbWF4KDAsIHJlbWFpbmluZyAtIDEpDQogICAgICAgICAgICAgICAgICAgIGF0b21pYyhzZWxmLnBhdGgsIGRhdGEpDQogICAgICAgICAgICAgICAgICAgIHJldHVybg0KICAgICAgICAgICAgICAgICMgUHJlc2VydmUgdGhlIHNlcnZlciBibG9jaywgaGlzdG9yeSBhbmQgcG9saWN5LiBOZXZlciByZXNldA0KICAgICAgICAgICAgICAgICMgdGhlIGxlZGdlciwgZGVsZXRlIG93bmVyc2hpcCwgb3IgY29udmVydCBhIGxvbmcgd2FpdCB0byBhDQogICAgICAgICAgICAgICAgIyBmYWtlIG5ldHdvcmsgVGltZW91dEVycm9yLg0KICAgICAgICAgICAgICAgIGF0b21pYyhzZWxmLnBhdGgsIGRhdGEpDQogICAgICAgICAgICBpZiBpbnN0YW50ID49IG5leHRfbm90aWNlOg0KICAgICAgICAgICAgICAgIGV2ZW50KHNlbGYucm9vdCAvICdSQVRFX0VWRU5UUy5qc29ubCcsICdxdW90YV93YWl0JywgYnVja2V0PWJ1Y2tldCwgc2Vjb25kcz1kZWxheSwgd2luZG93X3NlY29uZHM9d2luZG93LCBsb2NhbF9saW1pdD1saW1pdCwNCiAgICAgICAgICAgICAgICAgICAgICByZWFzb249J3NlcnZlcl9oZWFkZXInIGlmIHNlcnZlcl9kZWxheSA+PSBsb2NhbF9kZWxheSBlbHNlICdsb2NhbF93aW5kb3cnKQ0KICAgICAgICAgICAgICAgIHByaW50KCdIRicsIGJ1Y2tldCwgJ2J1ZGdldC9iYWNrb2ZmOiB3YWl0aW5nJywgcm91bmQoZGVsYXksIDEpLA0KICAgICAgICAgICAgICAgICAgICAgICdzZWNvbmRzOyBzYXZlZCBwcm9ncmVzcyBpcyByZXRhaW5lZC4nLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgICAgIG5leHRfbm90aWNlID0gaW5zdGFudCArIDMwDQogICAgICAgICAgICB0aW1lLnNsZWVwKG1pbigzMC4sIGRlbGF5KSkNCg0KICAgIGRlZiBvYnNlcnZlKHNlbGYsIGJ1Y2tldCwgcmVzcG9uc2UpOg0KICAgICAgICBoZWFkZXJzLCBpbnN0YW50ID0gcmVzcG9uc2UuaGVhZGVycywgdGltZS50aW1lKCkNCiAgICAgICAgZXhwZWN0ZWQgPSAncmVzb2x2ZXJzJyBpZiBidWNrZXQgPT0gJ3Jlc29sdmVyJyBlbHNlIGJ1Y2tldA0KICAgICAgICBwb2xpY3kgPSBOb25lDQogICAgICAgIGZvciBtYXRjaCBpbiByZS5maW5kaXRlcihyJyIoW15cIl0rKSJccyo7XHMqcT0oXGQrKVxzKjtccyp3PShcZCspJywgaGVhZGVycy5nZXQoJ1JhdGVMaW1pdC1Qb2xpY3knLCcnKSk6DQogICAgICAgICAgICBpZiBleHBlY3RlZCBpbiBtYXRjaFsxXS5zcGxpdCgnfCcpOg0KICAgICAgICAgICAgICAgIHBvbGljeSA9IGRpY3QocXVvdGE9aW50KG1hdGNoWzJdKSwgd2luZG93X3NlY29uZHM9aW50KG1hdGNoWzNdKSkNCiAgICAgICAgICAgICAgICBicmVhaw0KICAgICAgICByZW1haW5pbmcsIHJlc2V0ID0gTm9uZSwgTm9uZQ0KICAgICAgICBmb3IgbWF0Y2ggaW4gcmUuZmluZGl0ZXIociciKFteXCJdKykiXHMqO1xzKnI9KC0/XGQrKVxzKjtccyp0PShcZCspJywgaGVhZGVycy5nZXQoJ1JhdGVMaW1pdCcsJycpKToNCiAgICAgICAgICAgIGlmIGV4cGVjdGVkIGluIG1hdGNoWzFdLnNwbGl0KCd8Jyk6DQogICAgICAgICAgICAgICAgcmVtYWluaW5nLCByZXNldCA9IG1heCgwLCBpbnQobWF0Y2hbMl0pKSwgaW50KG1hdGNoWzNdKQ0KICAgICAgICAgICAgICAgIGJyZWFrDQogICAgICAgIGRlbGF5ID0gMC4NCiAgICAgICAgaWYgcmVzcG9uc2Uuc3RhdHVzX2NvZGUgPT0gNDI5Og0KICAgICAgICAgICAgdmFsdWUgPSBoZWFkZXJzLmdldCgnUmV0cnktQWZ0ZXInLCAnMCcpDQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgZGVsYXkgPSBtYXgoMC4sIGZsb2F0KHZhbHVlKSkNCiAgICAgICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOg0KICAgICAgICAgICAgICAgIGZyb20gZW1haWwudXRpbHMgaW1wb3J0IHBhcnNlZGF0ZV90b19kYXRldGltZQ0KICAgICAgICAgICAgICAgIGRlbGF5ID0gbWF4KDAuLCBwYXJzZWRhdGVfdG9fZGF0ZXRpbWUodmFsdWUpLnRpbWVzdGFtcCgpIC0gaW5zdGFudCkNCiAgICAgICAgICAgIGRlbGF5ID0gbWF4KGRlbGF5LCBmbG9hdChyZXNldCBvciAwKSkNCiAgICAgICAgZWxpZiByZW1haW5pbmcgPT0gMDoNCiAgICAgICAgICAgIGRlbGF5ID0gZmxvYXQocmVzZXQgb3IgMCkNCiAgICAgICAgd2l0aCBzZWxmLmxvY2s6DQogICAgICAgICAgICBkYXRhID0gcmVhZChzZWxmLnBhdGgpIGlmIHNlbGYucGF0aC5leGlzdHMoKSBlbHNlIHt9DQogICAgICAgICAgICBzdGF0ZSA9IGRhdGEuc2V0ZGVmYXVsdChidWNrZXQsIGRpY3QodGltZXM9W10sIGJsb2NrZWRfdW50aWw9MCkpDQogICAgICAgICAgICBpZiBwb2xpY3k6DQogICAgICAgICAgICAgICAgYXNzZXJ0IHBvbGljeVsncXVvdGEnXSA+IDAgYW5kIHBvbGljeVsnd2luZG93X3NlY29uZHMnXSA+IDANCiAgICAgICAgICAgICAgICBzdGF0ZVsnc2VydmVyX3BvbGljeSddID0gcG9saWN5DQogICAgICAgICAgICBpZiByZW1haW5pbmcgaXMgbm90IE5vbmU6DQogICAgICAgICAgICAgICAgc3RhdGVbJ3NlcnZlcl9yZW1haW5pbmcnXSwgc3RhdGVbJ3NlcnZlcl9yZXNldF9hdCddID0gcmVtYWluaW5nLCBpbnN0YW50ICsgcmVzZXQNCiAgICAgICAgICAgIGlmIGRlbGF5Og0KICAgICAgICAgICAgICAgIHN0YXRlWydibG9ja2VkX3VudGlsJ10gPSBtYXgoc3RhdGUuZ2V0KCdibG9ja2VkX3VudGlsJywwLiksIGluc3RhbnQgKyBkZWxheSkNCiAgICAgICAgICAgIGF0b21pYyhzZWxmLnBhdGgsIGRhdGEpDQogICAgICAgIGlmIHBvbGljeSBvciBkZWxheToNCiAgICAgICAgICAgIGV2ZW50KHNlbGYucm9vdCAvICdSQVRFX0VWRU5UUy5qc29ubCcsICdzZXJ2ZXJfcG9saWN5JywgYnVja2V0PWJ1Y2tldCwgcG9saWN5PXBvbGljeSwgcmVtYWluaW5nPXJlbWFpbmluZywgcmVzZXRfc2Vjb25kcz1yZXNldCwNCiAgICAgICAgICAgICAgICAgIGJsb2NrZWRfc2Vjb25kcz1kZWxheSwgc3RhdHVzPXJlc3BvbnNlLnN0YXR1c19jb2RlKQ0KDQoNCg0KY2xhc3MgQnVkZ2V0U2Vzc2lvbihyZXF1ZXN0cy5TZXNzaW9uKToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgYnVkZ2V0LCBsb2cpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgc2VsZi5idWRnZXQsIHNlbGYubG9nID0gYnVkZ2V0LCBQYXRoKGxvZykNCiAgICAgICAgc2VsZi50cnVzdF9lbnYgPSBGYWxzZQ0KDQogICAgZGVmIHNlbmQoc2VsZiwgcmVxdWVzdCwgKiprd2FyZ3MpOg0KICAgICAgICBwYXJzZWQ9dXJscGFyc2UocmVxdWVzdC51cmwpDQogICAgICAgIGlmIHJlcXVlc3QubWV0aG9kLnVwcGVyKCk9PSdQT1NUJyBhbmQgcGFyc2VkLmhvc3RuYW1lPT0naHVnZ2luZ2ZhY2UuY28nIGFuZCAnL2NvbW1pdC8nIGluIHBhcnNlZC5wYXRoOg0KICAgICAgICAgICAgc2VsZi5idWRnZXQucmVzZXJ2ZSgncmVwb19jb21taXQnKQ0KICAgICAgICBjYXRlZ29yeSA9IHNlbGYuYnVkZ2V0LmNhdGVnb3J5KHJlcXVlc3QudXJsKQ0KICAgICAgICBzZWxmLmJ1ZGdldC5yZXNlcnZlKGNhdGVnb3J5KQ0KICAgICAgICBiZWdpbiA9IHRpbWUubW9ub3RvbmljKCkNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgcmVzcG9uc2UgPSBzdXBlcigpLnNlbmQocmVxdWVzdCwgKiprd2FyZ3MpDQogICAgICAgICAgICBzZWxmLmJ1ZGdldC5vYnNlcnZlKGNhdGVnb3J5LCByZXNwb25zZSkNCiAgICAgICAgICAgIGV2ZW50KHNlbGYubG9nLCAnaHR0cCcsIGNhdGVnb3J5PWNhdGVnb3J5LCBtZXRob2Q9cmVxdWVzdC5tZXRob2QsIHN0YXR1cz1yZXNwb25zZS5zdGF0dXNfY29kZSwNCiAgICAgICAgICAgICAgICAgIHNlY29uZHM9dGltZS5tb25vdG9uaWMoKSAtIGJlZ2luLCByYXRlX2xpbWl0PXJlc3BvbnNlLmhlYWRlcnMuZ2V0KCdSYXRlTGltaXQnKSwNCiAgICAgICAgICAgICAgICAgIHJhdGVfcG9saWN5PXJlc3BvbnNlLmhlYWRlcnMuZ2V0KCdSYXRlTGltaXQtUG9saWN5JyksIHJldHJ5X2FmdGVyPXJlc3BvbnNlLmhlYWRlcnMuZ2V0KCdSZXRyeS1BZnRlcicpKQ0KICAgICAgICAgICAgcmV0dXJuIHJlc3BvbnNlDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXJyb3I6DQogICAgICAgICAgICBldmVudChzZWxmLmxvZywgJ2h0dHBfZXJyb3InLCBjYXRlZ29yeT1jYXRlZ29yeSwgbWV0aG9kPXJlcXVlc3QubWV0aG9kLCBlcnJvcl90eXBlPXR5cGUoZXJyb3IpLl9fbmFtZV9fKQ0KICAgICAgICAgICAgcmFpc2UNCg0KDQpkZWYgY29uZmlndXJlKHJvb3QsIGxvZyk6DQogICAgIyB2MC4zNi4yIGlzIHRoZSBwaW5uZWQgdHJhbnNwb3J0IHRlc3RlZCBieSB0aGlzIGRlbGl2ZXJ5LiBEbyBub3QgcGF0Y2gNCiAgICAjIGFyYml0cmFyeSBuZXdlciBTREsgaW50ZXJuYWxzIG9yIGluc3RhbGwvcmVwbGFjZSBUb3JjaCB0byBzYXRpc2Z5IGl0Lg0KICAgIGltcG9ydCBodWdnaW5nZmFjZV9odWINCiAgICBhc3NlcnQgaHVnZ2luZ2ZhY2VfaHViLl9fdmVyc2lvbl9fID09ICcwLjM2LjInLCAnSW5zdGFsbCBvbmx5IGh1Z2dpbmdmYWNlX2h1Yj09MC4zNi4yIGluIHRoZSBLYWdnbGUgc2V0dXAgY2VsbCcNCiAgICBmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgY29uZmlndXJlX2h0dHBfYmFja2VuZA0KICAgIGJ1ZGdldCA9IEJ1ZGdldChyb290KQ0KICAgIGNvbmZpZ3VyZV9odHRwX2JhY2tlbmQoYmFja2VuZF9mYWN0b3J5PWxhbWJkYTogQnVkZ2V0U2Vzc2lvbihidWRnZXQsIGxvZykpDQogICAgcmV0dXJuIGJ1ZGdldA0K', 'sha256': '55a8df1ee9fb498266858514d654c89ddd5276d7f8e4dd7bb96baa57277e5238'}, 'report_geo.py': {'base64': 'IiIiRW5naW5lZXJpbmcgZXZpZGVuY2UgZ2FsbGVyeTsgbGFiZWxsZWQgcHJlZGljdGlvbnMsIG5vdCBtZWFzdXJlZCB0eXJlIGNvbmRpdGlvbi4iIiIKaW1wb3J0IGJhc2U2NAppbXBvcnQgY3N2CmltcG9ydCBodG1sCmltcG9ydCBpbwppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gUElMIGltcG9ydCBJbWFnZSwgSW1hZ2VEcmF3CmZyb20gY29tbW9uIGltcG9ydCBhdG9taWMsIGNzdl93cml0ZSwgcmVhZAoKCmRlZiByZW5kZXIob3V0cHV0KToKICAgIG91dHB1dCA9IFBhdGgob3V0cHV0KQogICAgcHJvb2YgPSByZWFkKG91dHB1dC8nRU5HSU5FRVJJTkdfUFJPT0YuanNvbicpCiAgICByb3dzID0gW10KICAgIGZvciBwYXRoIGluIHNvcnRlZCgob3V0cHV0LydkaXJlY3QvdW5pdHMnKS5nbG9iKCcqL29iamVjdGl2ZXMuY3N2JykpOgogICAgICAgIHdpdGggcGF0aC5vcGVuKGVuY29kaW5nPSd1dGYtOCcsIG5ld2xpbmU9JycpIGFzIHN0cmVhbToKICAgICAgICAgICAgcm93cy5leHRlbmQoY3N2LkRpY3RSZWFkZXIoc3RyZWFtKSkKICAgIGltYWdlcyA9IFtdCiAgICBmb3IgdW5pdCBpbiBzb3J0ZWQoKG91dHB1dC8nZGlyZWN0L3VuaXRzJykuaXRlcmRpcigpKToKICAgICAgICB2YWx1ZXMgPSBucC5sb2FkKHVuaXQvJ2dlb21ldHJ5X2xvZ2l0cy5ucHknLCBhbGxvd19waWNrbGU9RmFsc2UpCiAgICAgICAgd2l0aCAodW5pdC8nc2FtcGxlcy5jc3YnKS5vcGVuKGVuY29kaW5nPSd1dGYtOCcsIG5ld2xpbmU9JycpIGFzIHN0cmVhbToKICAgICAgICAgICAgc2FtcGxlcyA9IGxpc3QoY3N2LkRpY3RSZWFkZXIoc3RyZWFtKSkKICAgICAgICBmb3Igc2xvdCwgbG9naXRzIGluIGVudW1lcmF0ZSh2YWx1ZXMpOgogICAgICAgICAgICB0aWxlID0gbnAuemVyb3MoKDEyOCwgOTYsIDMpLCBkdHlwZT1ucC51aW50OCkKICAgICAgICAgICAgdGlsZVtsb2dpdHNbMF0+PTBdID0gKDcyLCAxMzksIDIzNSkKICAgICAgICAgICAgdGlsZVtsb2dpdHNbMV0+PTBdID0gKDI0OCwgMTkxLCA2MCkKICAgICAgICAgICAgaW1hZ2UgPSBJbWFnZS5mcm9tYXJyYXkodGlsZSkucmVzaXplKCgxOTIsIDI1NiksIEltYWdlLlJlc2FtcGxpbmcuTkVBUkVTVCkKICAgICAgICAgICAgc3RyZWFtID0gaW8uQnl0ZXNJTygpCiAgICAgICAgICAgIGltYWdlLnNhdmUoc3RyZWFtLCBmb3JtYXQ9J1BORycpCiAgICAgICAgICAgIGltYWdlcy5hcHBlbmQoZGljdCh1cGRhdGU9aW50KHVuaXQubmFtZSksIGltYWdlX2lkPXNhbXBsZXNbc2xvdF1bJ2ltYWdlX2lkJ10sIGNvbXBvbmVudD1zYW1wbGVzW3Nsb3RdWydjb21wb25lbnQnXSwKICAgICAgICAgICAgICAgICAgICAgICAgcG5nPWJhc2U2NC5iNjRlbmNvZGUoc3RyZWFtLmdldHZhbHVlKCkpLmRlY29kZSgpKSkKICAgIHBheWxvYWQgPSBqc29uLmR1bXBzKGltYWdlcywgc2VwYXJhdG9ycz0oJywnLCAnOicpKS5yZXBsYWNlKCc8JywgJ1xcdTAwM2MnKQogICAgdGFibGUgPSAnPHRyPjx0aD5VcGRhdGU8L3RoPjx0aD5HZW9tZXRyeSBsb3NzPC90aD48dGg+TWFza2VkIEh1YmVyPC90aD48dGg+U1NMIGltYWdlczwvdGg+PC90cj4nCiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgdGFibGUgKz0gJzx0cj4nKycnLmpvaW4oJzx0ZD4nK2h0bWwuZXNjYXBlKHJvd1trXSkrJzwvdGQ+JyBmb3IgayBpbiAoJ3VwZGF0ZScsICdnZW9tZXRyeV9nZW9tZXRyeScsICd0ZXh0dXJlX21hc2tlZF9odWJlcicsICd0ZXh0dXJlX2VsaWdpYmxlX2ltYWdlcycpKSsnPC90cj4nCiAgICBwYWdlID0gJycnPCFkb2N0eXBlIGh0bWw+PGh0bWwgbGFuZz0iZW4iPjxtZXRhIGNoYXJzZXQ9InV0Zi04Ij48bWV0YSBuYW1lPSJ2aWV3cG9ydCIgY29udGVudD0id2lkdGg9ZGV2aWNlLXdpZHRoIj4KPHRpdGxlPlRSRUFELVRXTyBHRU8gZW5naW5lZXJpbmcgcGFzc3BvcnQ8L3RpdGxlPjxzdHlsZT5ib2R5e2ZvbnQ6MTZweCBzeXN0ZW0tdWk7YmFja2dyb3VuZDojMTAxNzIyO2NvbG9yOiNlZGYyZmE7bWFyZ2luOjJyZW07bWF4LXdpZHRoOjEzMDBweH1oMXtjb2xvcjojZmZjODVhfXRhYmxle2JvcmRlci1jb2xsYXBzZTpjb2xsYXBzZX10ZCx0aHtib3JkZXI6MXB4IHNvbGlkICM1MDYwNzg7cGFkZGluZzouNXJlbX1zZWxlY3QsYnV0dG9ue2ZvbnQ6aW5oZXJpdDtwYWRkaW5nOi41cmVtfSNnYWxsZXJ5e2Rpc3BsYXk6Z3JpZDtncmlkLXRlbXBsYXRlLWNvbHVtbnM6cmVwZWF0KGF1dG8tZmlsbCxtaW5tYXgoMjEwcHgsMWZyKSk7Z2FwOjFyZW07bWFyZ2luLXRvcDoxcmVtfWFydGljbGV7YmFja2dyb3VuZDojMjIzMDQ3O3BhZGRpbmc6MXJlbTtvdmVyZmxvdy13cmFwOmFueXdoZXJlfWltZ3tkaXNwbGF5OmJsb2NrO21hcmdpbjphdXRvO21heC13aWR0aDoxMDAlfS5tdXRlZHtjb2xvcjojYjJjMmQ2fTwvc3R5bGU+CjxoMT5UUkVBRC1UV08gLyBHRU8gwrcgZW5naW5lZXJpbmcgcGFzc3BvcnQ8L2gxPjxwPkRpc3Bvc2FibGUgdHJhaW4tb25seSB1cGRhdGVzLiBCbHVlOiBwcmVkaWN0ZWQgdHlyZTsgZ29sZDogcHJlZGljdGVkIHRyZWFkLiBUaGVzZSBtYXNrcyBhcmUgZW5naW5lZXJpbmcgb2JzZXJ2YXRpb25zLCBub3QgcGh5c2ljYWwgd2VhciwgZGVwdGgsIHNhZmV0eSBvciBtZWFzdXJlZC1maXQgcmVzdWx0cy48L3A+CjxwIGlkPSJzdW1tYXJ5Ij48L3A+PHRhYmxlPlRBQkxFPC90YWJsZT48aDI+SW50ZXJhY3RpdmUgcHJlZGljdGVkLW1hc2sgZ2FsbGVyeTwvaDI+PGxhYmVsPlVwZGF0ZSA8c2VsZWN0IGlkPSJmaWx0ZXIiPjxvcHRpb24gdmFsdWU9ImFsbCI+QWxsPC9vcHRpb24+PG9wdGlvbj4xPC9vcHRpb24+PG9wdGlvbj4yPC9vcHRpb24+PG9wdGlvbj4zPC9vcHRpb24+PG9wdGlvbj40PC9vcHRpb24+PC9zZWxlY3Q+PC9sYWJlbD48cCBpZD0iY291bnQiIGNsYXNzPSJtdXRlZCI+PC9wPjxkaXYgaWQ9ImdhbGxlcnkiPjwvZGl2Pgo8c2NyaXB0PmNvbnN0IGNhc2VzPVBBWUxPQUQ7ZnVuY3Rpb24gZHJhdygpe2NvbnN0IGZpbHRlcj1kb2N1bWVudC5nZXRFbGVtZW50QnlJZCgnZmlsdGVyJykudmFsdWU7Y29uc3QgY2hvc2VuPWNhc2VzLmZpbHRlcihjPT5maWx0ZXI9PT0nYWxsJ3x8U3RyaW5nKGMudXBkYXRlKT09PWZpbHRlcik7ZG9jdW1lbnQuZ2V0RWxlbWVudEJ5SWQoJ2NvdW50JykudGV4dENvbnRlbnQ9Y2hvc2VuLmxlbmd0aCsnIGVuZ2luZWVyaW5nIGNhc2VzJztjb25zdCBnPWRvY3VtZW50LmdldEVsZW1lbnRCeUlkKCdnYWxsZXJ5Jyk7Zy5yZXBsYWNlQ2hpbGRyZW4oKTtmb3IoY29uc3QgYyBvZiBjaG9zZW4pe2NvbnN0IGE9ZG9jdW1lbnQuY3JlYXRlRWxlbWVudCgnYXJ0aWNsZScpLHRpdGxlPWRvY3VtZW50LmNyZWF0ZUVsZW1lbnQoJ3AnKSxpbWc9ZG9jdW1lbnQuY3JlYXRlRWxlbWVudCgnaW1nJyksbm90ZT1kb2N1bWVudC5jcmVhdGVFbGVtZW50KCdwJyk7dGl0bGUudGV4dENvbnRlbnQ9J1VwZGF0ZSAnK2MudXBkYXRlKycgwrcgJytjLmltYWdlX2lkO25vdGUudGV4dENvbnRlbnQ9Yy5jb21wb25lbnQ7aW1nLnNyYz0nZGF0YTppbWFnZS9wbmc7YmFzZTY0LCcrYy5wbmc7aW1nLmFsdD0nUHJlZGljdGVkIHR5cmUgYW5kIHRyZWFkIG1hc2tzJzthLmFwcGVuZCh0aXRsZSxpbWcsbm90ZSk7Zy5hcHBlbmQoYSl9fWRvY3VtZW50LmdldEVsZW1lbnRCeUlkKCdmaWx0ZXInKS5hZGRFdmVudExpc3RlbmVyKCdjaGFuZ2UnLGRyYXcpO2RyYXcoKTs8L3NjcmlwdD48L2h0bWw+JycnLnJlcGxhY2UoJ1RBQkxFJywgdGFibGUpLnJlcGxhY2UoJ1BBWUxPQUQnLCBwYXlsb2FkKQogICAgKG91dHB1dC8nRW5naW5lZXJpbmdfUGFzc3BvcnQuaHRtbCcpLndyaXRlX3RleHQocGFnZSwgZW5jb2Rpbmc9J3V0Zi04JykKICAgIGF0b21pYyhvdXRwdXQvJ0dBTExFUlkuanNvbicsIGRpY3QoY2FzZXM9bGVuKGltYWdlcyksIHNjb3BlPSdlbmdpbmVlcmluZ190cmFpbl9vbmx5JywgaW50ZXJhY3RpdmVfZmlsdGVyPSd1cGRhdGUnKSkKICAgIHJldHVybiBkaWN0KGludGVyYWN0aXZlX2Nhc2VzPWxlbihpbWFnZXMpLCBodG1sPSdFbmdpbmVlcmluZ19QYXNzcG9ydC5odG1sJykK', 'sha256': 'd81cb2ec67e66819df526c7753980685eab27d86955ad9e9a399e409c591351e'}, 'schedule_geo.py': {'base64': 'IiIiRnVsbCAzODYtc291cmNlIGNvdmVyYWdlLCBzaXggbWFya2VkIHBhZGRpbmcgZHJhd3MsIGRldGVybWluaXN0aWMgY29tcG9uZW50IG1peGluZy4iIiIKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBjb21tb24gaW1wb3J0IGRpZ2VzdAoKCmRlZiBzY2hlZHVsZShyb3dzLCBzZWVkPTEsIGVwb2NoPTEpOgogICAgcm93cyA9IHNvcnRlZChyb3dzLCBrZXk9bGFtYmRhIHI6IHJbJ2ltYWdlX2lkJ10pCiAgICBieV9pZCA9IHtyWydpbWFnZV9pZCddOiByIGZvciByIGluIHJvd3N9CiAgICBhc3NlcnQgbGVuKHJvd3MpID09IGxlbihieV9pZCkgPT0gMzg2CiAgICBncm91cHMgPSBDb3VudGVyKHJbJ2NvbXBvbmVudCddIGZvciByIGluIHJvd3MpCiAgICBsYXJnZXN0ID0gc29ydGVkKGdyb3Vwcywga2V5PWxhbWJkYSB4OiAoLWdyb3Vwc1t4XSwgeCkpWzBdCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCAqIDEwMDAwMDMgKyBlcG9jaCkKICAgIGEgPSBsaXN0KHJuZy5wZXJtdXRhdGlvbihbclsnaW1hZ2VfaWQnXSBmb3IgciBpbiByb3dzIGlmIHJbJ2NvbXBvbmVudCddID09IGxhcmdlc3RdKSkKICAgIGMgPSBsaXN0KHJuZy5wZXJtdXRhdGlvbihbclsnaW1hZ2VfaWQnXSBmb3IgciBpbiByb3dzIGlmIHJbJ2NvbXBvbmVudCddICE9IGxhcmdlc3RdKSkKICAgIGFzc2VydCAobGVuKGEpLCBsZW4oYykpID09ICgyNDksIDEzNykKICAgIGJhdGNoZXMgPSBbWyhhW2ldLCBGYWxzZSksIChjW2ldLCBGYWxzZSldIGZvciBpIGluIHJhbmdlKDQ5KV0KICAgIHJlc3QgPSBbKHgsIEZhbHNlKSBmb3IgeCBpbiBhWzQ5Ol0gKyBjWzQ5Ol1dCiAgICByZXN0ICs9IFsoeCwgVHJ1ZSkgZm9yIHggaW4gcm5nLnBlcm11dGF0aW9uKHNvcnRlZChieV9pZCkpWzo2XV0KICAgIHJlc3QgPSBbcmVzdFtpXSBmb3IgaSBpbiBybmcucGVybXV0YXRpb24obGVuKHJlc3QpKV0KICAgIGZvciBiaSwgYmF0Y2ggaW4gZW51bWVyYXRlKGJhdGNoZXMpOgogICAgICAgIHdoaWxlIGxlbihiYXRjaCkgPCA4OgogICAgICAgICAgICBwcmVzZW50ID0ge3ggZm9yIHgsIF8gaW4gYmF0Y2h9CiAgICAgICAgICAgIGZvdW5kID0gbmV4dCgoaiBmb3IgaiwgdiBpbiBlbnVtZXJhdGUocmVzdCkgaWYgdlswXSBub3QgaW4gcHJlc2VudCksIE5vbmUpCiAgICAgICAgICAgIGlmIGZvdW5kIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgYmF0Y2guYXBwZW5kKHJlc3QucG9wKGZvdW5kKSkKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHJlcGFpcmVkID0gRmFsc2UKICAgICAgICAgICAgZm9yIHByaW9yIGluIGJhdGNoZXNbOmJpXToKICAgICAgICAgICAgICAgIGZvciBzbG90IGluIHJhbmdlKDIsIGxlbihwcmlvcikpOgogICAgICAgICAgICAgICAgICAgIGlmIHByaW9yW3Nsb3RdWzBdIGluIHByZXNlbnQ6CiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgcmVwbGFjZW1lbnQgPSBuZXh0KChqIGZvciBqLCB2IGluIGVudW1lcmF0ZShyZXN0KSBpZiB2WzBdIG5vdCBpbiB7eCBmb3IgeCwgXyBpbiBwcmlvcn0pLCBOb25lKQogICAgICAgICAgICAgICAgICAgIGlmIHJlcGxhY2VtZW50IGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgICAgICBjYW5kaWRhdGUgPSBwcmlvcltzbG90XQogICAgICAgICAgICAgICAgICAgICAgICBwcmlvcltzbG90XSA9IHJlc3QucG9wKHJlcGxhY2VtZW50KQogICAgICAgICAgICAgICAgICAgICAgICBiYXRjaC5hcHBlbmQoY2FuZGlkYXRlKQogICAgICAgICAgICAgICAgICAgICAgICByZXBhaXJlZCA9IFRydWUKICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcGFpcmVkOgogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGFzc2VydCByZXBhaXJlZCwgJ0Nhbm5vdCBwYWNrIGEgZHVwbGljYXRlLWZyZWUgYmF0Y2gnCiAgICBhc3NlcnQgbm90IHJlc3QKICAgIGFzc2VydCBDb3VudGVyKHggZm9yIGJhdGNoIGluIGJhdGNoZXMgZm9yIHgsIHBhZCBpbiBiYXRjaCBpZiBub3QgcGFkKSA9PSBDb3VudGVyKGJ5X2lkLmtleXMoKSkKICAgIGFzc2VydCBzdW0ocGFkIGZvciBiYXRjaCBpbiBiYXRjaGVzIGZvciBfLCBwYWQgaW4gYmF0Y2gpID09IDYKICAgIG91dHB1dCA9IFtdCiAgICBmb3IgYmksIGJhdGNoIGluIGVudW1lcmF0ZShiYXRjaGVzKToKICAgICAgICBhc3NlcnQgbGVuKHt4IGZvciB4LCBfIGluIGJhdGNofSkgPT0gOCBhbmQgbGVuKHtieV9pZFt4XVsnY29tcG9uZW50J10gZm9yIHgsIF8gaW4gYmF0Y2h9KSA+PSAyCiAgICAgICAgb3V0cHV0LmFwcGVuZChbeydpbWFnZV9pZCc6IHN0cih4KSwgJ2NvbXBvbmVudCc6IGJ5X2lkW3hdWydjb21wb25lbnQnXSwgJ3BhZGRpbmcnOiBib29sKHBhZCksCiAgICAgICAgICAgICAgICAgICAgICAgICdlcG9jaCc6IGVwb2NoLCAnYmF0Y2gnOiBiaSwgJ3Nsb3QnOiBzbG90LAogICAgICAgICAgICAgICAgICAgICAgICAnc2VlZCc6IGludChzZWVkICogMTAwMDAwMTkgKyBlcG9jaCAqIDEwMDAwMyArIGJpICogOCArIHNsb3QpfQogICAgICAgICAgICAgICAgICAgICAgIGZvciBzbG90LCAoeCwgcGFkKSBpbiBlbnVtZXJhdGUoYmF0Y2gpXSkKICAgIHJldHVybiBvdXRwdXQsIGRpZ2VzdChvdXRwdXQpCg==', 'sha256': 'c46cf30848c416d866b34238dc4ad4a4eb1cec438b15d9045905212c38992772'}, 'SCHEMAS.json': {'base64': '...', 'sha256': '7ddfbd71d3aa892cabf797827bc80aba2bd536848db7409232a6328bd3703a80'}, 'SOURCE_COMPONENTS.csv': {'base64': '...', 'sha256': '25fc0fd2ce5592297c3b1fe0929970db9c97040227b3b4fb71be107a5e3e293f'}, 'store_geo.py': {'base64': 'IiIiRW5naW5lZXJpbmcgY29tcGxldGUtdXBkYXRlIHRyYW5zYWN0aW9ucywgZGlyZWN0IENTViBhbmQgdW5jb21wcmVzc2VkIE5QWSBldmlkZW5jZS4KCkVhY2ggZml4dHVyZSB1bml0IGlzIGltbXV0YWJsZS4gVHdvIHJvbGxpbmcgc3RhdGUgZmlsZXMgcm90YXRlIG9ubHkgYWZ0ZXIgYSBuZXcKaGFzaC12ZXJpZmllZCBwb2ludGVyIGlzIGR1cmFibGUuIE5vIG1lYXN1cmVkLWZpdCBzaGFyZC9jYXBhY2l0eSBjbGFpbSBpcyBtYWRlLgoiIiIKaW1wb3J0IGNzdgppbXBvcnQgaW8KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gY29tbW9uIGltcG9ydCBhdG9taWMsIGNhbm9uaWNhbCwgY3B1LCBkaWdlc3QsIGZpbGVfbWV0YSwgaW5zaWRlLCBuYXRpdmUsIHJlYWQsIHJlc3RvcmVfcm5nLCBybmdfc3RhdGUsIHNoYSwgc3luY19kaXJlY3RvcnkKCgpjbGFzcyBTdG9yZToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCByb290LCBjb250cmFjdCk6CiAgICAgICAgc2VsZi5yb290ID0gUGF0aChyb290KQogICAgICAgIHNlbGYucm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgc2VsZi5jb250cmFjdCA9IGNvbnRyYWN0CgogICAgZGVmIGNvbW1pdChzZWxmLCBtb2RlbCwgb3B0aW1pemVyLCB1cGRhdGUsIGNvdW50ZXJzLCByb3dzLCBhcnJheXMsIGNyYXNoPU5vbmUpOgogICAgICAgIHBvaW50ZXJfcGF0aCA9IHNlbGYucm9vdC8nUE9JTlRFUi5qc29uJwogICAgICAgIHByZXZpb3VzID0gcmVhZChwb2ludGVyX3BhdGgpIGlmIHBvaW50ZXJfcGF0aC5leGlzdHMoKSBlbHNlIE5vbmUKICAgICAgICBhc3NlcnQgcHJldmlvdXMgaXMgTm9uZSBvciB1cGRhdGUgPT0gcHJldmlvdXNbJ3VwZGF0ZSddKzEKICAgICAgICByZWxhdGl2ZSA9IGYndW5pdHMve3VwZGF0ZTowNGR9JwogICAgICAgIGZvbGRlciA9IHNlbGYucm9vdC9yZWxhdGl2ZQogICAgICAgIGZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgaW52ZW50b3J5ID0ge30KICAgICAgICBmb3IgZmFtaWx5LCByZWNvcmRzIGluIHJvd3MuaXRlbXMoKToKICAgICAgICAgICAgYXNzZXJ0IHJlY29yZHMgYW5kIGxlbih7dHVwbGUoci5rZXlzKCkpIGZvciByIGluIHJlY29yZHN9KSA9PSAxCiAgICAgICAgICAgIHBhdGggPSBmb2xkZXIvKGZhbWlseSsnLmNzdicpCiAgICAgICAgICAgICMgQSBzaW5nbGUgZXhwbGljaXQgQ1NWIHNjaGVtYSBwZXIgZmFtaWx5L3VuaXQ7IGZhaWwgb24gZXh0cmEvbWlzc2luZyBmaWVsZHMuCiAgICAgICAgICAgIHdpdGggcGF0aC5vcGVuKCd3JywgZW5jb2Rpbmc9J3V0Zi04JywgbmV3bGluZT0nJykgYXMgc3RyZWFtOgogICAgICAgICAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoc3RyZWFtLCBmaWVsZG5hbWVzPWxpc3QocmVjb3Jkc1swXSksIGxpbmV0ZXJtaW5hdG9yPSdcbicpCiAgICAgICAgICAgICAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgICAgICAgICAgICAgZm9yIHJlY29yZCBpbiByZWNvcmRzOgogICAgICAgICAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdyh7azogc2VsZi5mb3JtYXQodikgZm9yIGssIHYgaW4gcmVjb3JkLml0ZW1zKCl9KQogICAgICAgICAgICAgICAgc3RyZWFtLmZsdXNoKCkKICAgICAgICAgICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKICAgICAgICAgICAgaW52ZW50b3J5W3BhdGgucmVsYXRpdmVfdG8oc2VsZi5yb290KS5hc19wb3NpeCgpXSA9IGZpbGVfbWV0YShwYXRoKQogICAgICAgIGZvciBuYW1lLCBhcnJheSBpbiBhcnJheXMuaXRlbXMoKToKICAgICAgICAgICAgcGF0aCA9IGZvbGRlci8obmFtZSsnLm5weScpCiAgICAgICAgICAgIG5hdGl2ZShwYXRoLCBucC5hc2FycmF5KGFycmF5KSkKICAgICAgICAgICAgaW52ZW50b3J5W3BhdGgucmVsYXRpdmVfdG8oc2VsZi5yb290KS5hc19wb3NpeCgpXSA9IGZpbGVfbWV0YShwYXRoKQogICAgICAgIGlmIGNyYXNoID09ICdhZnRlcl9ldmlkZW5jZSc6CiAgICAgICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ0luamVjdGVkIGludGVycnVwdGlvbiBhZnRlciB1bmNvbW1pdHRlZCBldmlkZW5jZScpCiAgICAgICAgc3RhdGVfcGF0aCA9IHNlbGYucm9vdC9mJ3N0YXRlX3t1cGRhdGUlMn0ucHQnCiAgICAgICAgc3RhdGUgPSBkaWN0KGNvbnRyYWN0PXNlbGYuY29udHJhY3QsIHVwZGF0ZT11cGRhdGUsIGNvdW50ZXJzPWNvdW50ZXJzLCBtb2RlbD1jcHUobW9kZWwuc3RhdGVfZGljdCgpKSwKICAgICAgICAgICAgICAgICAgICAgb3B0aW1pemVyPWNwdShvcHRpbWl6ZXIuc3RhdGVfZGljdCgpKSwgcm5nPXJuZ19zdGF0ZSgpLCBwcmVjaXNpb249J0ZQMzJfbm9fc2NhbGVyJywKICAgICAgICAgICAgICAgICAgICAgc2NoZWR1bGVyPWRpY3QoY29tcGxldGVkX3VwZGF0ZXM9dXBkYXRlLCB0b3RhbF91cGRhdGVzPTI5NDAsIGNvbnZlbnRpb249J2Nvc2luZV9hdF9zdGFydF91c2luZ19wcmV2aW91c19jb21wbGV0ZV9jdXJzb3InKSwKICAgICAgICAgICAgICAgICAgICAgcHJpb3JfdW5pdHM9W10gaWYgcHJldmlvdXMgaXMgTm9uZSBlbHNlIHByZXZpb3VzWyd1bml0cyddKQogICAgICAgIG5hdGl2ZShzdGF0ZV9wYXRoLCBzdGF0ZSkKICAgICAgICBpZiBjcmFzaCA9PSAnYWZ0ZXJfc3RhdGUnOgogICAgICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCdJbmplY3RlZCBpbnRlcnJ1cHRpb24gYmVmb3JlIHBvaW50ZXIgc3dhcCcpCiAgICAgICAgcmVjb3JkID0gZGljdCh1cGRhdGU9dXBkYXRlLCBmaWxlcz1pbnZlbnRvcnksIHByZXZpb3VzPU5vbmUgaWYgcHJldmlvdXMgaXMgTm9uZSBlbHNlIHByZXZpb3VzWydjaGFpbiddKQogICAgICAgIHJlY29yZFsnY2hhaW4nXSA9IGRpZ2VzdChyZWNvcmQpCiAgICAgICAgYXRvbWljKGZvbGRlci8nVU5JVC5qc29uJywgcmVjb3JkKQogICAgICAgIHBvaW50ZXIgPSBkaWN0KGNvbnRyYWN0PXNlbGYuY29udHJhY3QsIHVwZGF0ZT11cGRhdGUsIGNoYWluPXJlY29yZFsnY2hhaW4nXSwgc3RhdGU9c3RhdGVfcGF0aC5uYW1lLAogICAgICAgICAgICAgICAgICAgICAgIHN0YXRlX21ldGE9ZmlsZV9tZXRhKHN0YXRlX3BhdGgpLCB1bml0cz1zdGF0ZVsncHJpb3JfdW5pdHMnXStbcmVsYXRpdmUrJy9VTklULmpzb24nXSkKICAgICAgICBhdG9taWMocG9pbnRlcl9wYXRoLCBwb2ludGVyKQogICAgICAgIGlmIGNyYXNoID09ICdhZnRlcl9wb2ludGVyJzoKICAgICAgICAgICAgcmFpc2UgSW50ZXJydXB0ZWRFcnJvcignSW5qZWN0ZWQgaW50ZXJydXB0aW9uIGFmdGVyIGR1cmFibGUgcG9pbnRlcicpCiAgICAgICAgcmV0dXJuIHBvaW50ZXIKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgZm9ybWF0KHZhbHVlKToKICAgICAgICBpZiB2YWx1ZSBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gJycKICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAoYm9vbCwgbnAuYm9vbF8pKToKICAgICAgICAgICAgcmV0dXJuIGludCh2YWx1ZSkKICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAoZmxvYXQsIG5wLmZsb2F0aW5nKSk6CiAgICAgICAgICAgIGFzc2VydCBucC5pc2Zpbml0ZSh2YWx1ZSkKICAgICAgICAgICAgcmV0dXJuIGZvcm1hdChmbG9hdCh2YWx1ZSksICcuOWcnIGlmIGlzaW5zdGFuY2UodmFsdWUsIG5wLmZsb2F0MzIpIGVsc2UgJy4xN2cnKQogICAgICAgIHJldHVybiB2YWx1ZQoKICAgIGRlZiB2ZXJpZnkoc2VsZik6CiAgICAgICAgcG9pbnRlciA9IHJlYWQoc2VsZi5yb290LydQT0lOVEVSLmpzb24nKQogICAgICAgIGFzc2VydCBwb2ludGVyWydjb250cmFjdCddID09IHNlbGYuY29udHJhY3QKICAgICAgICBhc3NlcnQgZmlsZV9tZXRhKGluc2lkZShzZWxmLnJvb3QsIHBvaW50ZXJbJ3N0YXRlJ10pKSA9PSBwb2ludGVyWydzdGF0ZV9tZXRhJ10KICAgICAgICBwcmlvciA9IE5vbmUKICAgICAgICBmb3IgbmFtZSBpbiBwb2ludGVyWyd1bml0cyddOgogICAgICAgICAgICByZWNvcmQgPSByZWFkKGluc2lkZShzZWxmLnJvb3QsIG5hbWUpKQogICAgICAgICAgICBhc3NlcnQgcmVjb3JkWydwcmV2aW91cyddID09IHByaW9yCiAgICAgICAgICAgIGFzc2VydCBkaWdlc3Qoe2s6diBmb3Igayx2IGluIHJlY29yZC5pdGVtcygpIGlmIGshPSdjaGFpbid9KSA9PSByZWNvcmRbJ2NoYWluJ10KICAgICAgICAgICAgZm9yIHJlbGF0aXZlLCBtZXRhIGluIHJlY29yZFsnZmlsZXMnXS5pdGVtcygpOgogICAgICAgICAgICAgICAgYXNzZXJ0IGZpbGVfbWV0YShpbnNpZGUoc2VsZi5yb290LCByZWxhdGl2ZSkpID09IG1ldGEsIHJlbGF0aXZlCiAgICAgICAgICAgIHByaW9yID0gcmVjb3JkWydjaGFpbiddCiAgICAgICAgYXNzZXJ0IHByaW9yID09IHBvaW50ZXJbJ2NoYWluJ10gYW5kIGxlbihwb2ludGVyWyd1bml0cyddKSA9PSBwb2ludGVyWyd1cGRhdGUnXQogICAgICAgIHJldHVybiBwb2ludGVyCgogICAgZGVmIHJlc3RvcmUoc2VsZiwgbW9kZWwsIG9wdGltaXplcik6CiAgICAgICAgcG9pbnRlciA9IHNlbGYudmVyaWZ5KCkKICAgICAgICBzdGF0ZSA9IHRvcmNoLmxvYWQoc2VsZi5yb290L3BvaW50ZXJbJ3N0YXRlJ10sIG1hcF9sb2NhdGlvbj0nY3B1Jywgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgIGFzc2VydCBzdGF0ZVsnY29udHJhY3QnXSA9PSBzZWxmLmNvbnRyYWN0IGFuZCBzdGF0ZVsndXBkYXRlJ10gPT0gcG9pbnRlclsndXBkYXRlJ10KICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3Qoc3RhdGVbJ21vZGVsJ10sIHN0cmljdD1UcnVlKQogICAgICAgIG9wdGltaXplci5sb2FkX3N0YXRlX2RpY3Qoc3RhdGVbJ29wdGltaXplciddKQogICAgICAgIHJlc3RvcmVfcm5nKHN0YXRlWydybmcnXSkKICAgICAgICByZXR1cm4gc3RhdGUKCiAgICBkZWYgc25hcHNob3Qoc2VsZiwgZGVzdGluYXRpb24pOgogICAgICAgICIiIkZyb3plbiBjb3BpZXMgb2Ygc2VsZWN0ZWQgZmlsZXMgb25seTogbmV2ZXIgaW5jbHVkZSB1bmNvbW1pdHRlZCB0YWlscy4iIiIKICAgICAgICBpbXBvcnQgc2h1dGlsCiAgICAgICAgcG9pbnRlciA9IHNlbGYudmVyaWZ5KCkKICAgICAgICB0YXJnZXQgPSBQYXRoKGRlc3RpbmF0aW9uKQogICAgICAgIHRhcmdldC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgbmFtZXMgPSBbJ1BPSU5URVIuanNvbicsIHBvaW50ZXJbJ3N0YXRlJ11dCiAgICAgICAgZm9yIHVuaXQgaW4gcG9pbnRlclsndW5pdHMnXToKICAgICAgICAgICAgbmFtZXMuYXBwZW5kKHVuaXQpCiAgICAgICAgICAgIG5hbWVzLmV4dGVuZChyZWFkKHNlbGYucm9vdC91bml0KVsnZmlsZXMnXSkKICAgICAgICBmb3IgbmFtZSBpbiBuYW1lczoKICAgICAgICAgICAgcGF0aCA9IHRhcmdldC9uYW1lCiAgICAgICAgICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgc2h1dGlsLmNvcHlmaWxlKHNlbGYucm9vdC9uYW1lLCBwYXRoKQogICAgICAgIGF0b21pYyh0YXJnZXQvJ01BTklGRVNULmpzb24nLCBkaWN0KGNvbnRyYWN0PXNlbGYuY29udHJhY3QsIGZpbGVzPXtuOmZpbGVfbWV0YSh0YXJnZXQvbikgZm9yIG4gaW4gbmFtZXN9KSkKICAgICAgICByZXR1cm4gdGFyZ2V0Cg==', 'sha256': '2e2824d5dad9a9db6654109641a915cb69f9d1fd8083cd14835546245556a184'}, 'VENDORED.json': {'base64': 'ew0KICAiY29tbW9uLnB5Ijogew0KICAgICJzb3VyY2UiOiAicGhhc2UzL3RyZWFkX3R3by9jb250ZXh0X3BpbG90L2NvbW1vbi5weSIsDQogICAgInNoYTI1NiI6ICI5MjJjODJiODI2MzUzMTNjZTYxZWNjMjk5Mjk5MGY1YmMwYjdmM2NiMWQ2YzhmY2EwMDUyN2RlNzJhMzRhMjA2Ig0KICB9LA0KICAiZnJvemVuX2FuY2hvci5weSI6IHsNCiAgICAic291cmNlIjogInBoYXNlMy90cmVhZF90d28vY29udGV4dF9waWxvdC9mcm96ZW5fYW5jaG9yLnB5IiwNCiAgICAic2hhMjU2IjogImQxNmNkNGVlZjc1Nzc5MDQ0ZGQ5NTkwMGMwNTMxZmVlNDc0NzA0Y2Y4OGY1MjMzMjZlZTA2OTExNTM0N2JhMDkiDQogIH0sDQogICJwaGFzZTJfdHJhaW5pbmdfZGF0YS5weSI6IHsNCiAgICAic291cmNlIjogInBoYXNlMy90cmVhZF90d28vY29udGV4dF9waWxvdC9waGFzZTJfdHJhaW5pbmdfZGF0YS5weSIsDQogICAgInNoYTI1NiI6ICI2MGY5NzU0MDc4OTQ5ZWNhYzU3MDRiNzNlZjkzMDdiN2I5NWMyY2M4YWNjMWEyMGRmMmYzNDVkMDhkN2UyNTc5Ig0KICB9LA0KICAibGVnYWN5X2RhdGEucHkiOiB7DQogICAgInNvdXJjZSI6ICJwaGFzZTMvdHJlYWRfdHdvL2NvbnRleHRfcGlsb3QvbGVnYWN5X2RhdGEucHkiLA0KICAgICJzaGEyNTYiOiAiYTE1YmVkMGUyNTc0OTgwYzJhMjMzOWE4YTkwMDYxM2NjOTI5MmM2MzQyZDJlZTVlMTZlMWY1YTA1Njk5NTVhZSINCiAgfSwNCiAgImdlb21ldHJ5LnB5Ijogew0KICAgICJzb3VyY2UiOiAicGhhc2UzL3RyZWFkX3R3by9jb250ZXh0X3BpbG90L2dlb21ldHJ5LnB5IiwNCiAgICAic2hhMjU2IjogIjA0YTgxNTFmNjY2Yzk2YWI3NDgyYzFhYTQ4YzJhMmVhN2UxNWQzNGRiMDk4OTNiZTNhN2FkMzFlNTA3ZDM2NmEiDQogIH0sDQogICJsY19mZWF0dXJlcy5weSI6IHsNCiAgICAic291cmNlIjogInBoYXNlMy90cmVhZF90d28vY29udGV4dF9waWxvdC9sY19mZWF0dXJlcy5weSIsDQogICAgInNoYTI1NiI6ICJkNDE3ZTU1MmI5MTA4MzZkOGRmMjczNGJiNTQyM2Y3ZGNlZTc5NTZmNDg1YjAyMDQ2NzM5YWQxYmVmNTc0MWQ2Ig0KICB9LA0KICAiZXZpZGVuY2VfaHViLnB5Ijogew0KICAgICJzb3VyY2UiOiAicGhhc2UzL3RyZWFkX3R3by9jb250ZXh0X3BpbG90L2V2aWRlbmNlX2h1Yi5weSIsDQogICAgInNoYTI1NiI6ICIwM2RhNzBjNzliMTBjZmFmMzA3YmQyOWM4MWY5MDc0N2Q0NTQ0MGM3ZWI3ZTJhMGQxOWYxZmVlZjk4MGM3YjlmIg0KICB9LA0KICAicHVibGlzaGVyLnB5Ijogew0KICAgICJzb3VyY2UiOiAicGhhc2UzL3RyZWFkX3R3by9jb250ZXh0X3BpbG90L3B1Ymxpc2hlci5weSIsDQogICAgInNoYTI1NiI6ICIwZWE3N2Q1NTg0YTM5MDNiMDFmZGY3NDY2ZGFhOTVlYjdiZTEwMjFiNGU3MGFjNzZhNjcwNjUzMWEzZDU2MzY0Ig0KICB9LA0KICAicmF0ZV9idWRnZXQucHkiOiB7DQogICAgInNvdXJjZSI6ICJwaGFzZTMvdHJlYWRfdHdvL2NvbnRleHRfcGlsb3QvcmF0ZV9idWRnZXQucHkiLA0KICAgICJzaGEyNTYiOiAiNTVhOGRmMWVlOWZiNDk4MjY2ODU4NTE0ZDY1NGM4OWRkZDUyNzZkN2Y4ZTRkZDdiYjk2YmFhNTcyNzdlNTIzOCINCiAgfQ0KfQ0K', 'sha256': '17fcf7abc28bcda71f9dd8e0787324ca71e4a0271a3191bd526e8aa83253d307'}, 'worker_geo.py': {'base64': 'IiIiU2VjcmV0LWZyZWUgc3VicHJvY2VzcyBlbnRyeSBwb2ludDsgZW5naW5lZXJpbmcgZml4dHVyZSBvciBkb3dubG9hZGVkLXN0YXRlIHNlYW0uIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzeXMKCm9zLmVudmlyb24uc2V0ZGVmYXVsdCgnQ1VCTEFTX1dPUktTUEFDRV9DT05GSUcnLCAnOjQwOTY6OCcpCmFzc2VydCBub3Qgb3MuZW52aXJvbi5nZXQoJ0hGX1RPS0VOJyksICdUaGUgbnVtZXJpYyB3b3JrZXIgbXVzdCBub3QgcmVjZWl2ZSBIRl9UT0tFTicKZnJvbSBjb21tb24gaW1wb3J0IGF0b21pYwpmcm9tIGVuZ2luZWVyaW5nX2dlbyBpbXBvcnQgY29udGludWVfcmVtb3RlLCBydW5fZml4dHVyZQoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCdzdGFnZScsIGNob2ljZXM9KCdmaXh0dXJlJywgJ2NvbnRpbnVlJykpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLW91dHB1dCcsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLXdlaWdodHMnKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1kYXRhLXJvb3QnLCBkZWZhdWx0PScnKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1yZXN0b3JlZCcpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgb3V0cHV0ID0gUGF0aChhcmdzLm91dHB1dCkKICAgIHRyeToKICAgICAgICBpZiBhcmdzLnN0YWdlID09ICdmaXh0dXJlJzoKICAgICAgICAgICAgcnVuX2ZpeHR1cmUob3V0cHV0LCBhcmdzLndlaWdodHMsIGFyZ3MuZGF0YV9yb290KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNvbnRpbnVlX3JlbW90ZShvdXRwdXQsIGFyZ3MucmVzdG9yZWQpCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgYXRvbWljKG91dHB1dC8nV09SS0VSX1NUT1AuanNvbicsIGRpY3Qoa2luZD10eXBlKGV4YykuX19uYW1lX18sIG1lc3NhZ2U9c3RyKGV4YyksCiAgICAgICAgICAgICAgICAgICAgbWVhc3VyZWRfZml0X3VwZGF0ZXM9MCwgc2NvcGU9J2VuZ2luZWVyaW5nX2ZpeHR1cmUnKSkKICAgICAgICByYWlzZQoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBtYWluKCkK', 'sha256': '9c6fb033c070c29ec3bef62a9668dbb7aba21e3e20a4904dd276d43b3b242c02'}}
source = Path('/kaggle/working/geo_source_9bbb92ec75db')
source.mkdir(parents=True,exist_ok=True)
for name,item in PAYLOAD.items():
    assert Path(name).name==name
    data=base64.b64decode(item['base64'],validate=True)
    assert hashlib.sha256(data).hexdigest()==item['sha256']
    path=source/name
    if path.exists() and path.read_bytes()!=data:
        raise RuntimeError('Extracted GEO source changed: '+name+'. Preserve it and restart from the unchanged notebook.')
    path.write_bytes(data)
if 'common' in sys.modules and Path(sys.modules['common'].__file__).resolve().parent!=source.resolve():
    raise RuntimeError('Restart the kernel: another experiment runtime is already imported.')
sys.path.insert(0,str(source))
print('Verified',len(PAYLOAD),'embedded source/config files. Private HF SDK',huggingface_hub.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 6.6 MB/s  0:00:00
Verified 27 embedded source/config files. Private HF SDK 0.36.2


## 2 · Run the actual gate

Expect visible phases for checksum verification, each engineering update,
CSV/native saving, HF publication and downloaded-state continuation. A heartbeat
appears at least every30 seconds while the coordinator is waiting. HF quota waits
may legitimately be longer; keep the session and its files.

Catchable **Stop** asks the separate worker to finish its current complete update,
then immediately attempts publication of its last committed state. A forced
kernel/session kill cannot upload unsent bytes. Restart with the same notebook;
compatible local state is verified and reused. In a fresh empty session the
disposable preparation checks can repeat; they are not the measured experiment.


In [2]:
# 2. HF_TOKEN is read only from the authorized Kaggle Secret, never printed.
from kaggle_secrets import UserSecretsClient
from coordinator_geo import run
token = UserSecretsClient().get_secret('HF_TOKEN')
try:
    result = run(token)
finally:
    token = None


NB16 alive: {"stage": "source verification / initialization"} — next complete boundary is protected.
Verifying every prepared data checksum and historical annotation role...
NB16 alive: {"stage": "source verification / initialization"} — next complete boundary is protected.
/kaggle/working/geo_source_9bbb92ec75db/checks_geo.py:36: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  assert np.isclose(float(terms['masked_numerator']), numerator, rtol=1e-5)
Engineering direct: update 1/4, frozen features then backward / CSV / native commit
Engineering direct: update 2/4, frozen features then backward / CSV / native commit
Engineering direct: update 3/4, frozen features then backward / CSV / native commit
Engineering direct: update 4/4, frozen features then backward / CSV / native commit
Engineering resum

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

HF api budget/backoff: waiting 212.4 seconds; saved progress is retained.
HF api budget/backoff: waiting 182.3 seconds; saved progress is retained.
HF api budget/backoff: waiting 152.3 seconds; saved progress is retained.
HF api budget/backoff: waiting 122.3 seconds; saved progress is retained.
HF api budget/backoff: waiting 92.3 seconds; saved progress is retained.
HF api budget/backoff: waiting 62.3 seconds; saved progress is retained.
HF api budget/backoff: waiting 32.3 seconds; saved progress is retained.
HF api budget/backoff: waiting 2.3 seconds; saved progress is retained.
HF api budget/backoff: waiting 0.2 seconds; saved progress is retained.
HF api budget/backoff: waiting 21.9 seconds; saved progress is retained.
HF api budget/backoff: waiting 0.1 seconds; saved progress is retained.
HF api budget/backoff: waiting 0.0 seconds; saved progress is retained.
HF api budget/backoff: waiting 0.5 seconds; saved progress is retained.
HF api budget/backoff: waiting 1.7 seconds; saved pr

## 3 · Read the receipt and interactive engineering gallery

Success means **engineering_gate_passed** with actual dual-T4 and HF-restored
continuation proofs. **measured_fit_ready stays false**: the full 40-family fit
telemetry/shard/staging integration still needs its own acceptance before NB17.
The passport shows predicted geometry for disposable training examples. It is not
a physical wear, depth, safety or improved-accuracy result.


In [3]:
from IPython.display import HTML, FileLink, display
from pathlib import Path
import json
output=Path('/kaggle/working/tread_two_geo_nb16')
receipt=json.loads((output/'RUN_RESULT.json').read_text())
print(json.dumps(receipt,indent=2))
display(HTML((output/'Engineering_Passport.html').read_text()))
for name in ('RUN_RESULT.json','ENGINEERING_PROOF.json','HF_CONTINUATION_PROOF.json',
             'STORAGE.json','PARAMETERS.json','Engineering_Passport.html'):
    display(FileLink(str(output/name)))


{
  "actual_HF_restore_continue": true,
  "actual_dual_T4": true,
  "measured_fit_ready": false,
  "measured_fit_started": false,
  "next": "Independent NB16 receipt review, finish 40-family fit/shard integration and freeze NB17",
  "protocol": "c8d91064beec55898208953811dfe75ec63bfe067ec2a4f19cf96e358b616979",
  "publication": {
    "destination": "phase3/tread-two/geo-r1/engineering/c8d91064beec55898208953811dfe75ec63bfe067ec2a4f19cf96e358b616979",
    "files": 174,
    "owner_released": true,
    "revision": "9f991635f1d333e4e48eb6f932a9c704ca5eeb39",
    "status": "published_and_verified",
    "uploaded_objects": 3,
    "zip_uploads": 0
  },
  "scope": "Preparation only; no 60-epoch fit or scientific improvement claim",
  "status": "engineering_gate_passed"
}


Update,Geometry loss,Masked Huber,SSL images
1,0.15335267782211304,0.075535111129283905,8
2,0.14378939568996429,0.070823289453983307,8
3,0.15734066069126129,0.058244634419679642,8
4,0.15990673005580902,0.053775187581777573,8


/kaggle/working/tread_two_geo_nb16/RUN_RESULT.json

/kaggle/working/tread_two_geo_nb16/ENGINEERING_PROOF.json

/kaggle/working/tread_two_geo_nb16/HF_CONTINUATION_PROOF.json

/kaggle/working/tread_two_geo_nb16/STORAGE.json

/kaggle/working/tread_two_geo_nb16/PARAMETERS.json

/kaggle/working/tread_two_geo_nb16/Engineering_Passport.html

## Stop, recovery and troubleshooting

| Symptom | Action |
|---|---|
| GPU/input/secret check fails | Correct the Kaggle settings above, then rerun unchanged |
| Source or numerical-environment mismatch | Preserve output; do not edit hashes to bypass the check |
| A prior HF owner is active | Stop the other copy cleanly or wait for the displayed lease; never run concurrent writers |
| HTTP429 / budget wait | Leave the session open; server-directed waiting preserves local state |
| Catchable stop/error | Keep output and rerun unchanged; inspect HF_PENDING.json if transfer failed |
| Forced session death | Retain saved Kaggle output if available; unsent bytes cannot be guaranteed |
| No geometry support / deterministic-op failure | Keep the saved transcript; do not disable determinism or change the model mid-check |
| Complete rerun | Shows the existing compatible receipt without new optimizer steps or HF writes |

HF records live under **phase3/tread-two/geo-r1/engineering/&lt;contract&gt;** in
**Shanmuk4622/tyre-wear-study**, using directly browsable CSV, JSON, HTML, NPY and
native PT. No ZIP/NPZ/database envelope. The coordinator alone receives the token.

After completion, save this executed notebook with outputs and return its result.
We will independently verify the HF revision, inspect the measured storage and
device evidence, update the documents, and finish/freeze NB17. Existing photos only.
